In [ ]:
from pathlib import Path
import argparse
import time
import numpy as np
from itertools import product
import pandas as pd
from scipy.optimize import minimize
from scipy.special import ndtr
from scipy.special import beta as beta_fn
from scipy.special import expit, gammaln, logsumexp
from scipy.stats import gennorm
from scipy.stats import norm
from scipy.stats import t as student_t_dist
import os

# ============================================================
# STEP 6.1 - FROZEN INNOVATION MARGINAL DISTRIBUTION LAYER
#
# This section ports the four finalized R marginal
# distributions to Python:
#
#   normal
#   student_t
#   ged
#   skewed_student_t
#
# All distributions follow the frozen R convention:
#
#   E[Z] = 0
#   Var[Z] = 1
#
# This layer is deliberately independent of:
#
#   - the competition simulator,
#   - the joint 16D dependence system,
#   - the arithmetic-return mapping,
#   - the optimizer.
#
# Step 6.1 only defines and validates marginal distributions.
# ============================================================


SUPPORTED_INNOVATION_FAMILIES = (
    "normal",
    "student_t",
    "ged",
    "skewed_student_t",
)


def _validate_distribution_parameters(
    family,
    parameters=None,
):
    """
    Validate one frozen innovation-distribution specification.
    """

    if parameters is None:
        parameters = {}

    if family not in SUPPORTED_INNOVATION_FAMILIES:
        raise ValueError(
            f"Unsupported innovation family: {family}"
        )

    if family == "normal":
        return {}

    if family == "student_t":
        if "df" not in parameters:
            raise ValueError(
                "student_t requires parameter 'df'."
            )

        df = float(parameters["df"])

        if (
            not np.isfinite(df)
            or df <= 2.0
        ):
            raise ValueError(
                "student_t df must be finite and > 2."
            )

        return {
            "df": df,
        }

    if family == "ged":
        if "shape" not in parameters:
            raise ValueError(
                "ged requires parameter 'shape'."
            )

        shape = float(parameters["shape"])

        if (
            not np.isfinite(shape)
            or shape <= 0.0
        ):
            raise ValueError(
                "ged shape must be finite and > 0."
            )

        return {
            "shape": shape,
        }

    if family == "skewed_student_t":
        if "df" not in parameters:
            raise ValueError(
                "skewed_student_t requires parameter 'df'."
            )

        if "skew" not in parameters:
            raise ValueError(
                "skewed_student_t requires parameter 'skew'."
            )

        df = float(parameters["df"])
        skew = float(parameters["skew"])

        if (
            not np.isfinite(df)
            or df <= 2.0
        ):
            raise ValueError(
                "skewed_student_t df must be finite and > 2."
            )

        if (
            not np.isfinite(skew)
            or skew <= 0.0
        ):
            raise ValueError(
                "skewed_student_t skew must be finite and > 0."
            )

        return {
            "df": df,
            "skew": skew,
        }

    raise RuntimeError(
        "Unhandled innovation family."
    )


def _standardized_student_t_scale(df):
    """
    Scale a conventional Student-t random variable so that
    the resulting variable has variance one.

    If T ~ t_df, then:

        Var(T) = df / (df - 2)

    Therefore:

        Z = T * sqrt((df - 2) / df)

    has Var(Z) = 1.
    """

    df = float(df)

    return np.sqrt(
        (df - 2.0) / df
    )


def _standardized_student_t_cdf(
    x,
    df,
):
    """
    CDF of the mean-zero, variance-one Student-t distribution.
    """

    scale = _standardized_student_t_scale(
        df
    )

    return student_t_dist.cdf(
        np.asarray(
            x,
            dtype=float,
        ) / scale,
        df=df,
    )


def _standardized_student_t_ppf(
    p,
    df,
):
    """
    PPF of the mean-zero, variance-one Student-t distribution.
    """

    scale = _standardized_student_t_scale(
        df
    )

    return (
        student_t_dist.ppf(
            np.asarray(
                p,
                dtype=float,
            ),
            df=df,
        )
        * scale
    )


def _standardized_ged_scale(
    shape,
):
    """
    Scale parameter for the standardized GED.

    SciPy's generalized-normal density uses:

        f(x) proportional to
        exp(-(abs(x) / scale)^shape)

    Choosing:

        scale =
        sqrt(
            Gamma(1 / shape)
            /
            Gamma(3 / shape)
        )

    gives variance one and matches the frozen R GED
    parameterization.
    """

    shape = float(shape)

    log_scale = 0.5 * (
        gammaln(
            1.0 / shape
        )
        - gammaln(
            3.0 / shape
        )
    )

    return np.exp(
        log_scale
    )


def _standardized_ged_cdf(
    x,
    shape,
):
    """
    CDF of the mean-zero, variance-one GED.
    """

    scale = _standardized_ged_scale(
        shape
    )

    return gennorm.cdf(
        np.asarray(
            x,
            dtype=float,
        ),
        beta=shape,
        loc=0.0,
        scale=scale,
    )


def _standardized_ged_ppf(
    p,
    shape,
):
    """
    PPF of the mean-zero, variance-one GED.
    """

    scale = _standardized_ged_scale(
        shape
    )

    return gennorm.ppf(
        np.asarray(
            p,
            dtype=float,
        ),
        beta=shape,
        loc=0.0,
        scale=scale,
    )


def _skewed_student_t_standardization(
    df,
    skew,
):
    """
    Frozen Fernandez-Steel skewed Student-t standardization.

    The underlying symmetric Student-t is already standardized
    to variance one.

    The Fernandez-Steel skewing transformation creates an
    intermediate skewed variable Y.

    This function returns:

        raw_mean = E[Y]
        raw_sd   = SD[Y]

    so that:

        Z = (Y - raw_mean) / raw_sd

    has:

        E[Z] = 0
        Var[Z] = 1
    """

    df = float(df)
    skew = float(skew)

    mean_absolute_standardized_t = (
        2.0
        * np.sqrt(
            df - 2.0
        )
        / (
            (df - 1.0)
            * beta_fn(
                0.5,
                df / 2.0,
            )
        )
    )

    raw_mean = (
        mean_absolute_standardized_t
        * (
            skew
            - 1.0 / skew
        )
    )

    raw_variance = (
        (
            1.0
            - mean_absolute_standardized_t ** 2
        )
        * (
            skew ** 2
            + skew ** -2
        )
        + 2.0
        * mean_absolute_standardized_t ** 2
        - 1.0
    )

    if (
        not np.isfinite(raw_variance)
        or raw_variance <= 0.0
    ):
        raise ValueError(
            "Invalid skewed Student-t standardization variance."
        )

    raw_sd = np.sqrt(
        raw_variance
    )

    return (
        raw_mean,
        raw_sd,
    )


def _skewed_student_t_cdf(
    x,
    df,
    skew,
):
    """
    CDF of the frozen standardized Fernandez-Steel skewed
    Student-t distribution.
    """

    x_array = np.asarray(
        x,
        dtype=float,
    )

    raw_mean, raw_sd = (
        _skewed_student_t_standardization(
            df=df,
            skew=skew,
        )
    )

    raw_value = (
        x_array * raw_sd
        + raw_mean
    )

    skew_squared = (
        skew ** 2
    )

    probability_at_zero = (
        1.0
        / (
            1.0
            + skew_squared
        )
    )

    result = np.empty_like(
        raw_value,
        dtype=float,
    )

    negative_mask = (
        raw_value < 0.0
    )

    if np.any(
        negative_mask
    ):
        base_probability = (
            _standardized_student_t_cdf(
                raw_value[
                    negative_mask
                ] * skew,
                df=df,
            )
        )

        result[
            negative_mask
        ] = (
            2.0
            / (
                1.0
                + skew_squared
            )
            * base_probability
        )

    nonnegative_mask = (
        ~negative_mask
    )

    if np.any(
        nonnegative_mask
    ):
        base_probability = (
            _standardized_student_t_cdf(
                raw_value[
                    nonnegative_mask
                ] / skew,
                df=df,
            )
        )

        result[
            nonnegative_mask
        ] = (
            probability_at_zero
            + (
                2.0
                * skew_squared
                / (
                    1.0
                    + skew_squared
                )
            )
            * (
                base_probability
                - 0.5
            )
        )

    return result


def _skewed_student_t_ppf(
    p,
    df,
    skew,
):
    """
    PPF of the frozen standardized Fernandez-Steel skewed
    Student-t distribution.
    """

    p_array = np.asarray(
        p,
        dtype=float,
    )

    raw_mean, raw_sd = (
        _skewed_student_t_standardization(
            df=df,
            skew=skew,
        )
    )

    skew_squared = (
        skew ** 2
    )

    probability_at_zero = (
        1.0
        / (
            1.0
            + skew_squared
        )
    )

    raw_quantile = np.empty_like(
        p_array,
        dtype=float,
    )

    lower_mask = (
        p_array
        < probability_at_zero
    )

    if np.any(
        lower_mask
    ):
        base_probability = (
            p_array[
                lower_mask
            ]
            * (
                1.0
                + skew_squared
            )
            / 2.0
        )

        raw_quantile[
            lower_mask
        ] = (
            _standardized_student_t_ppf(
                base_probability,
                df=df,
            )
            / skew
        )

    upper_mask = (
        ~lower_mask
    )

    if np.any(
        upper_mask
    ):
        base_probability = (
            0.5
            + (
                p_array[
                    upper_mask
                ]
                - probability_at_zero
            )
            * (
                1.0
                + skew_squared
            )
            / (
                2.0
                * skew_squared
            )
        )

        raw_quantile[
            upper_mask
        ] = (
            skew
            * _standardized_student_t_ppf(
                base_probability,
                df=df,
            )
        )

    return (
        raw_quantile
        - raw_mean
    ) / raw_sd


def distribution_cdf(
    x,
    family,
    parameters=None,
):
    """
    Evaluate the frozen standardized innovation CDF.

    Parameters
    ----------
    x:
        Scalar or array-like evaluation point.

    family:
        One of:

            normal
            student_t
            ged
            skewed_student_t

    parameters:
        Dictionary containing the required frozen shape
        parameters.
    """

    parameters = _validate_distribution_parameters(
        family=family,
        parameters=parameters,
    )

    x_array = np.asarray(
        x,
        dtype=float,
    )

    if family == "normal":
        result = norm.cdf(
            x_array
        )

    elif family == "student_t":
        result = _standardized_student_t_cdf(
            x=x_array,
            df=parameters["df"],
        )

    elif family == "ged":
        result = _standardized_ged_cdf(
            x=x_array,
            shape=parameters["shape"],
        )

    elif family == "skewed_student_t":
        result = _skewed_student_t_cdf(
            x=x_array,
            df=parameters["df"],
            skew=parameters["skew"],
        )

    else:
        raise RuntimeError(
            "Unhandled innovation family."
        )

    result = np.asarray(
        result,
        dtype=float,
    )

    if np.ndim(
        x
    ) == 0:
        return float(
            result
        )

    return result


def distribution_ppf(
    p,
    family,
    parameters=None,
):
    """
    Evaluate the frozen standardized innovation quantile
    function.
    """

    parameters = _validate_distribution_parameters(
        family=family,
        parameters=parameters,
    )

    p_array = np.asarray(
        p,
        dtype=float,
    )

    if np.any(
        ~np.isfinite(
            p_array
        )
    ):
        raise ValueError(
            "Probabilities must be finite."
        )

    if np.any(
        (
            p_array < 0.0
        )
        | (
            p_array > 1.0
        )
    ):
        raise ValueError(
            "Probabilities must lie in [0, 1]."
        )

    if family == "normal":
        result = norm.ppf(
            p_array
        )

    elif family == "student_t":
        result = _standardized_student_t_ppf(
            p=p_array,
            df=parameters["df"],
        )

    elif family == "ged":
        result = _standardized_ged_ppf(
            p=p_array,
            shape=parameters["shape"],
        )

    elif family == "skewed_student_t":
        result = _skewed_student_t_ppf(
            p=p_array,
            df=parameters["df"],
            skew=parameters["skew"],
        )

    else:
        raise RuntimeError(
            "Unhandled innovation family."
        )

    result = np.asarray(
        result,
        dtype=float,
    )

    if np.ndim(
        p
    ) == 0:
        return float(
            result
        )

    return result


def distribution_rvs(
    size,
    family,
    parameters=None,
    rng=None,
):
    """
    Draw random values through inverse-transform sampling.

    Random-number generation is deliberately defined through
    the already-validated PPF rather than through separate
    family-specific RNG implementations.
    """

    if rng is None:
        rng = np.random.default_rng()

    uniforms = rng.random(
        size=size
    )

    probability_epsilon = (
        np.finfo(float).eps
    )

    uniforms = np.clip(
        uniforms,
        probability_epsilon,
        1.0 - probability_epsilon,
    )

    return distribution_ppf(
        p=uniforms,
        family=family,
        parameters=parameters,
    )

# ============================================================
# STEP 6.2 - FROZEN 16D JOINT INNOVATION COUPLING
#
# This section ports the frozen 50.15 joint innovation
# coupling architecture to Python.
#
# Canonical 16D order:
#
#   0  CL
#   1  GC
#   2  SR3
#   3  ZN
#   4  KE
#   5  HO
#   6  ZW
#   7  NQ
#   8  Delta_Final_X1
#   9  Delta_Final_D12
#  10  Delta_Final_D23
#  11  Delta_Final_D34
#  12  Delta_Final_D4_15
#  13  Delta_Final_D15_16
#  14  Delta_Daily_H1
#  15  Delta_Daily_D12
#
# Production architecture:
#
#   1. Draw a 16D Gaussian driver from the frozen
#      time-varying latent correlation matrix.
#
#   2. Futures:
#          U_F = Phi(G_F)
#
#   3. FinalState:
#          S = sqrt(ChiSquare(df) / df)
#          T_Final = G_Final / S
#          U_Final = StudentTCDF(T_Final, df)
#
#      One shared S is used across all six FinalState
#      components within each simulation.
#
#   4. DailyState:
#          U_D = Phi(G_D)
#
#   5. Transform every uniform through the already-frozen
#      Step 6.1 marginal inverse CDF.
#
# This does NOT yet modify simulate_competition_paths().
# ============================================================


PYTHON_INNOVATION_PROJECT_ROOT = (
    str(Path.cwd().parent if Path.cwd().name == "optimizer" else Path.cwd())
)

DEFAULT_PYTHON_INNOVATION_PRODUCTION_DIR = os.path.join(
    PYTHON_INNOVATION_PROJECT_ROOT,
    "data",
    "processed",
    "innovations",
    "python_production",
)


CANONICAL_INNOVATION_COMPONENT_ORDER = [
    "CL",
    "GC",
    "SR3",
    "ZN",
    "KE",
    "HO",
    "ZW",
    "NQ",
    "Delta_Final_X1",
    "Delta_Final_D12",
    "Delta_Final_D23",
    "Delta_Final_D34",
    "Delta_Final_D4_15",
    "Delta_Final_D15_16",
    "Delta_Daily_H1",
    "Delta_Daily_D12",
]


FUTURES_INNOVATION_INDICES = np.arange(
    0,
    8,
    dtype=int,
)

FINAL_STATE_INNOVATION_INDICES = np.arange(
    8,
    14,
    dtype=int,
)

DAILY_STATE_INNOVATION_INDICES = np.arange(
    14,
    16,
    dtype=int,
)


def _parameters_from_python_marginal_row(
    row,
):
    """
    Convert one row of innovation_marginals.csv into the
    parameter dictionary expected by the frozen Step 6.1
    distribution functions.
    """

    family = str(
        row["distribution"]
    )

    if family == "normal":
        return {}

    if family == "student_t":
        return {
            "df": float(
                row["df"]
            ),
        }

    if family == "ged":
        return {
            "shape": float(
                row["shape"]
            ),
        }

    if family == "skewed_student_t":
        return {
            "df": float(
                row["df"]
            ),
            "skew": float(
                row["skew"]
            ),
        }

    raise ValueError(
        f"Unsupported innovation family: {family}"
    )


def load_python_innovation_system(
    production_dir=None,
):
    """
    Load the frozen 50.4 Python-facing innovation interface.

    Returns
    -------
    innovation_system:
        Dictionary containing:

        component_order
        marginal_specs
        production_times
        time_to_index
        latent_correlation
        final_state_student_t_df
        metadata
    """

    if production_dir is None:
        production_dir = (
            DEFAULT_PYTHON_INNOVATION_PRODUCTION_DIR
        )

    marginal_path = os.path.join(
        production_dir,
        "innovation_marginals.csv",
    )

    latent_correlation_path = os.path.join(
        production_dir,
        "innovation_latent_correlation.csv",
    )

    metadata_path = os.path.join(
        production_dir,
        "innovation_system_metadata.csv",
    )

    required_paths = [
        marginal_path,
        latent_correlation_path,
        metadata_path,
    ]

    for path in required_paths:
        if not os.path.exists(
            path
        ):
            raise FileNotFoundError(
                path
            )

    # --------------------------------------------------------
    # LOAD METADATA
    # --------------------------------------------------------

    metadata_df = pd.read_csv(
        metadata_path,
        dtype=str,
        keep_default_na=False,
    )

    required_metadata_columns = {
        "key",
        "value",
    }

    if not required_metadata_columns.issubset(
        metadata_df.columns
    ):
        raise ValueError(
            "innovation_system_metadata.csv must contain "
            "columns 'key' and 'value'."
        )

    if metadata_df["key"].duplicated().any():
        raise ValueError(
            "Duplicate metadata keys found."
        )

    metadata = dict(
        zip(
            metadata_df["key"],
            metadata_df["value"],
        )
    )

    required_metadata_keys = {
        "interface_version",
        "marginal_system_version",
        "dependence_system_version",
        "joint_coupling_version",
        "production_time_count",
        "future_count",
        "final_state_count",
        "daily_state_count",
        "component_count",
        "final_state_student_t_df",
        "futures_dependence",
        "final_state_dependence",
        "daily_state_dependence",
        "cross_system_dependence",
        "canonical_component_order",
        "marginal_production_parameters_final",
        "dependence_production_parameters_final",
        "joint_16d_coupling_final",
        "python_monte_carlo_modified",
    }

    missing_metadata_keys = (
        required_metadata_keys
        - set(
            metadata.keys()
        )
    )

    if missing_metadata_keys:
        raise ValueError(
            "innovation_system_metadata.csv is missing keys: "
            f"{sorted(missing_metadata_keys)}"
        )

    if metadata["interface_version"] != "1.0.0":
        raise ValueError(
            "Unexpected Python innovation interface version."
        )

    if metadata["marginal_system_version"] != "50.0":
        raise ValueError(
            "Unexpected marginal production version."
        )

    if metadata["dependence_system_version"] != "50.1":
        raise ValueError(
            "Unexpected dependence production version."
        )

    if metadata["joint_coupling_version"] != "50.15":
        raise ValueError(
            "Unexpected joint coupling version."
        )

    if metadata["marginal_production_parameters_final"] != "TRUE":
        raise ValueError(
            "Marginal production parameters are not final."
        )

    if metadata["dependence_production_parameters_final"] != "TRUE":
        raise ValueError(
            "Dependence production parameters are not final."
        )

    if metadata["joint_16d_coupling_final"] != "TRUE":
        raise ValueError(
            "Joint 16D coupling is not final."
        )

    if metadata["python_monte_carlo_modified"] != "FALSE":
        raise ValueError(
            "50.4 metadata unexpectedly reports that the "
            "Python Monte Carlo was already modified."
        )

    component_order = (
        metadata[
            "canonical_component_order"
        ].split("|")
    )

    if (
        component_order
        != CANONICAL_INNOVATION_COMPONENT_ORDER
    ):
        raise ValueError(
            "Canonical innovation component order mismatch."
        )

    if int(
        metadata["component_count"]
    ) != 16:
        raise ValueError(
            "Production innovation system must contain "
            "16 components."
        )

    if int(
        metadata["future_count"]
    ) != 8:
        raise ValueError(
            "Production innovation system must contain "
            "8 futures."
        )

    if int(
        metadata["final_state_count"]
    ) != 6:
        raise ValueError(
            "Production innovation system must contain "
            "6 final-state components."
        )

    if int(
        metadata["daily_state_count"]
    ) != 2:
        raise ValueError(
            "Production innovation system must contain "
            "2 daily-state components."
        )

    final_state_student_t_df = float(
        metadata[
            "final_state_student_t_df"
        ]
    )

    if (
        not np.isfinite(
            final_state_student_t_df
        )
        or final_state_student_t_df <= 2.0
    ):
        raise ValueError(
            "Invalid FinalState Student-t copula df."
        )

    if (
        metadata["futures_dependence"]
        != "Gaussian_Copula"
    ):
        raise ValueError(
            "Unexpected futures dependence architecture."
        )

    if (
        metadata["final_state_dependence"]
        != "StudentT_Copula"
    ):
        raise ValueError(
            "Unexpected FinalState dependence architecture."
        )

    if (
        metadata["daily_state_dependence"]
        != "Existing_Gaussian_Dependence"
    ):
        raise ValueError(
            "Unexpected DailyState dependence architecture."
        )

    if (
        metadata["cross_system_dependence"]
        !=
        "Existing_Conservative_PostShrinkage_"
        "Latent_Gaussian_Coupling"
    ):
        raise ValueError(
            "Unexpected cross-system dependence architecture."
        )

    # --------------------------------------------------------
    # LOAD MARGINAL SPECIFICATIONS
    # --------------------------------------------------------

    marginal_df = pd.read_csv(
        marginal_path
    )

    required_marginal_columns = {
        "component",
        "component_index",
        "distribution",
        "df",
        "shape",
        "skew",
    }

    if not required_marginal_columns.issubset(
        marginal_df.columns
    ):
        raise ValueError(
            "innovation_marginals.csv is missing required "
            "columns."
        )

    if len(
        marginal_df
    ) != 16:
        raise ValueError(
            "innovation_marginals.csv must contain "
            "exactly 16 rows."
        )

    marginal_df = marginal_df.sort_values(
        "component_index"
    ).reset_index(
        drop=True
    )

    if (
        marginal_df[
            "component_index"
        ].to_numpy(
            dtype=int
        ).tolist()
        != list(
            range(
                1,
                17,
            )
        )
    ):
        raise ValueError(
            "Marginal component indices must be 1 through 16."
        )

    if (
        marginal_df[
            "component"
        ].tolist()
        != component_order
    ):
        raise ValueError(
            "Marginal component order does not match "
            "canonical metadata order."
        )

    marginal_specs = []

    for _, row in marginal_df.iterrows():

        family = str(
            row["distribution"]
        )

        parameters = (
            _parameters_from_python_marginal_row(
                row
            )
        )

        parameters = (
            _validate_distribution_parameters(
                family=family,
                parameters=parameters,
            )
        )

        marginal_specs.append(
            {
                "component":
                    str(
                        row["component"]
                    ),
                "family":
                    family,
                "parameters":
                    parameters,
            }
        )

    # --------------------------------------------------------
    # LOAD 151 X 16 X 16 LATENT CORRELATION ARRAY
    # --------------------------------------------------------

    latent_df = pd.read_csv(
        latent_correlation_path
    )

    required_latent_columns = {
        "t_original",
        "component_1_index",
        "component_2_index",
        "component_1",
        "component_2",
        "correlation",
    }

    if not required_latent_columns.issubset(
        latent_df.columns
    ):
        raise ValueError(
            "innovation_latent_correlation.csv is missing "
            "required columns."
        )

    production_times = np.sort(
        latent_df[
            "t_original"
        ].unique().astype(int)
    )

    expected_time_count = int(
        metadata[
            "production_time_count"
        ]
    )

    if len(
        production_times
    ) != expected_time_count:
        raise ValueError(
            "Latent-correlation production time count "
            "does not match metadata."
        )

    expected_latent_rows = (
        expected_time_count
        * 16
        * 16
    )

    if len(
        latent_df
    ) != expected_latent_rows:
        raise ValueError(
            "innovation_latent_correlation.csv has "
            "unexpected row count."
        )

    latent_correlation = np.empty(
        (
            expected_time_count,
            16,
            16,
        ),
        dtype=float,
    )

    for time_index, t_original in enumerate(
        production_times
    ):

        current_rows = latent_df[
            latent_df[
                "t_original"
            ].astype(int)
            == int(
                t_original
            )
        ].copy()

        if len(
            current_rows
        ) != 256:
            raise ValueError(
                "Each production interval must contain "
                "exactly 256 latent-correlation rows."
            )

        current_rows = current_rows.sort_values(
            [
                "component_1_index",
                "component_2_index",
            ]
        )

        matrix = np.empty(
            (
                16,
                16,
            ),
            dtype=float,
        )

        seen_pairs = set()

        for row in current_rows.itertuples(
            index=False
        ):

            i = int(
                row.component_1_index
            ) - 1

            j = int(
                row.component_2_index
            ) - 1

            if not (
                0 <= i < 16
                and 0 <= j < 16
            ):
                raise ValueError(
                    "Latent component index outside 1 through 16."
                )

            pair = (
                i,
                j,
            )

            if pair in seen_pairs:
                raise ValueError(
                    "Duplicate latent-correlation matrix entry."
                )

            seen_pairs.add(
                pair
            )

            if (
                str(
                    row.component_1
                )
                != component_order[i]
            ):
                raise ValueError(
                    "component_1 name/index mismatch in "
                    "latent-correlation file."
                )

            if (
                str(
                    row.component_2
                )
                != component_order[j]
            ):
                raise ValueError(
                    "component_2 name/index mismatch in "
                    "latent-correlation file."
                )

            matrix[
                i,
                j
            ] = float(
                row.correlation
            )

        if len(
            seen_pairs
        ) != 256:
            raise ValueError(
                "Incomplete latent correlation matrix."
            )

        if not np.all(
            np.isfinite(
                matrix
            )
        ):
            raise ValueError(
                "Latent correlation matrix contains "
                "non-finite values."
            )

        symmetry_error = float(
            np.max(
                np.abs(
                    matrix
                    - matrix.T
                )
            )
        )

        if symmetry_error > 1e-12:
            raise ValueError(
                "Latent correlation matrix is not symmetric."
            )

        diagonal_error = float(
            np.max(
                np.abs(
                    np.diag(
                        matrix
                    )
                    - 1.0
                )
            )
        )

        if diagonal_error > 1e-12:
            raise ValueError(
                "Latent correlation matrix diagonal "
                "is not one."
            )

        # The frozen 50.15 matrices are positive definite.
        # Do not numerically repair or alter them here.
        np.linalg.cholesky(
            matrix
        )

        latent_correlation[
            time_index
        ] = matrix

    time_to_index = {
        int(t_original): time_index
        for time_index, t_original
        in enumerate(
            production_times
        )
    }

    innovation_system = {
        "component_order":
            component_order,
        "marginal_specs":
            marginal_specs,
        "production_times":
            production_times,
        "time_to_index":
            time_to_index,
        "latent_correlation":
            latent_correlation,
        "final_state_student_t_df":
            final_state_student_t_df,
        "metadata":
            metadata,
    }

    return innovation_system

# ============================================================
# STEP 7B - PRACTICE INNOVATION SYSTEM FILE LOADER
#
# Load the heterogeneous practice innovation specification
# serialized by Step 7A.
#
# This loader is intentionally separate from the frozen
# production loader. It does not weaken, bypass, or modify any
# production-version validation.
# ============================================================


def load_practice_innovation_system(
    input_dir,
):
    """
    Load and validate the Step 7A practice innovation system.

    Parameters
    ----------
    input_dir:
        Directory containing:

            innovation_marginals.csv
            innovation_latent_correlation.csv
            innovation_system_metadata.csv

    Returns
    -------
    innovation_system:
        Dictionary compatible with build_joint_innovations().
    """

    if input_dir is None:
        raise ValueError(
            "input_dir must be provided for the practice "
            "innovation loader."
        )

    marginal_path = os.path.join(
        input_dir,
        "innovation_marginals.csv",
    )

    latent_correlation_path = os.path.join(
        input_dir,
        "innovation_latent_correlation.csv",
    )

    metadata_path = os.path.join(
        input_dir,
        "innovation_system_metadata.csv",
    )

    required_paths = [
        marginal_path,
        latent_correlation_path,
        metadata_path,
    ]

    for path in required_paths:

        if not os.path.exists(
            path
        ):
            raise FileNotFoundError(
                path
            )

    # --------------------------------------------------------
    # LOAD AND VALIDATE STEP 7A METADATA
    # --------------------------------------------------------

    metadata_df = pd.read_csv(
        metadata_path,
        dtype=str,
        keep_default_na=False,
    )

    if not {
        "key",
        "value",
    }.issubset(
        metadata_df.columns
    ):
        raise ValueError(
            "Practice innovation_system_metadata.csv must "
            "contain columns 'key' and 'value'."
        )

    if metadata_df["key"].duplicated().any():
        raise ValueError(
            "Duplicate practice metadata keys found."
        )

    metadata = dict(
        zip(
            metadata_df[
                "key"
            ],
            metadata_df[
                "value"
            ],
        )
    )

    required_metadata_keys = {
        "interface_version",
        "system_type",
        "serialization_step",
        "practice_mode",
        "production_time_count",
        "future_count",
        "final_state_count",
        "daily_state_count",
        "component_count",
        "final_state_copula",
        "final_state_student_t_df",
        "futures_dependence",
        "final_state_dependence",
        "daily_state_dependence",
        "cross_system_dependence",
        "canonical_component_order",
        "futures_order",
        "arithmetic_mapping_family",
        "return_floor",
        "return_floor_epsilon",
    }

    missing_metadata_keys = (
        required_metadata_keys
        - set(
            metadata.keys()
        )
    )

    if missing_metadata_keys:
        raise ValueError(
            "Practice innovation metadata is missing keys: "
            f"{sorted(missing_metadata_keys)}"
        )

    if (
        metadata[
            "interface_version"
        ]
        != "practice_7A_1.0.0"
    ):
        raise ValueError(
            "Unexpected practice innovation interface version."
        )

    if (
        metadata[
            "system_type"
        ]
        != "practice_heterogeneous_distributional"
    ):
        raise ValueError(
            "Unexpected practice innovation system type."
        )

    if (
        metadata[
            "serialization_step"
        ]
        != "7A"
    ):
        raise ValueError(
            "Practice innovation files were not serialized "
            "by Step 7A."
        )

    if (
        metadata[
            "practice_mode"
        ]
        != "heterogeneous_distributional"
    ):
        raise ValueError(
            "Unexpected practice innovation mode."
        )

    expected_time_count = int(
        metadata[
            "production_time_count"
        ]
    )

    if expected_time_count <= 0:
        raise ValueError(
            "Practice production time count must be positive."
        )

    expected_counts = {
        "future_count":
            8,
        "final_state_count":
            6,
        "daily_state_count":
            2,
        "component_count":
            16,
    }

    for key, expected_value in expected_counts.items():

        actual_value = int(
            metadata[
                key
            ]
        )

        if actual_value != expected_value:
            raise ValueError(
                f"Unexpected practice metadata value for "
                f"{key}: {actual_value}."
            )

    component_order = (
        metadata[
            "canonical_component_order"
        ].split("|")
    )

    if (
        component_order
        != CANONICAL_INNOVATION_COMPONENT_ORDER
    ):
        raise ValueError(
            "Practice component order does not match the "
            "canonical 16D innovation order."
        )

    metadata_futures_order = (
        metadata[
            "futures_order"
        ].split("|")
    )

    if (
        metadata_futures_order
        != FUTURES_PRODUCTION_ORDER
    ):
        raise ValueError(
            "Practice futures order does not match the "
            "canonical futures order."
        )

    final_state_copula = str(
        metadata[
            "final_state_copula"
        ]
    ).strip().lower()

    if final_state_copula not in (
        "gaussian",
        "student_t",
    ):
        raise ValueError(
            "Practice final-state copula must be either "
            "'gaussian' or 'student_t'."
        )

    final_state_student_t_df = float(
        metadata[
            "final_state_student_t_df"
        ]
    )

    if final_state_copula == "student_t":

        if (
            not np.isfinite(
                final_state_student_t_df
            )
            or final_state_student_t_df <= 2.0
        ):
            raise ValueError(
                "Practice FinalState Student-t df must be "
                "finite and greater than 2."
            )

    if (
        metadata[
            "futures_dependence"
        ]
        != "Gaussian_Copula"
    ):
        raise ValueError(
            "Unexpected practice futures dependence."
        )

    if (
        metadata[
            "final_state_dependence"
        ]
        != "StudentT_Copula"
    ):
        raise ValueError(
            "Unexpected practice FinalState dependence."
        )

    if (
        metadata[
            "daily_state_dependence"
        ]
        != "Gaussian_Copula"
    ):
        raise ValueError(
            "Unexpected practice DailyState dependence."
        )

    if (
        metadata[
            "cross_system_dependence"
        ]
        != "Latent_Gaussian_Coupling"
    ):
        raise ValueError(
            "Unexpected practice cross-system dependence."
        )

    # --------------------------------------------------------
    # LOAD PRACTICE MARGINAL SPECIFICATIONS
    # --------------------------------------------------------

    marginal_df = pd.read_csv(
        marginal_path
    )

    required_marginal_columns = {
        "component",
        "component_index",
        "distribution",
        "df",
        "shape",
        "skew",
    }

    if not required_marginal_columns.issubset(
        marginal_df.columns
    ):
        raise ValueError(
            "Practice innovation_marginals.csv is missing "
            "required columns."
        )

    if len(
        marginal_df
    ) != 16:
        raise ValueError(
            "Practice innovation_marginals.csv must contain "
            "exactly 16 rows."
        )

    marginal_df = marginal_df.sort_values(
        "component_index"
    ).reset_index(
        drop=True
    )

    if (
        marginal_df[
            "component_index"
        ].to_numpy(
            dtype=int
        ).tolist()
        != list(
            range(
                1,
                17,
            )
        )
    ):
        raise ValueError(
            "Practice marginal component indices must be "
            "1 through 16."
        )

    if (
        marginal_df[
            "component"
        ].tolist()
        != component_order
    ):
        raise ValueError(
            "Practice marginal component order does not "
            "match metadata."
        )

    marginal_specs = []

    for _, row in marginal_df.iterrows():

        family = str(
            row[
                "distribution"
            ]
        )

        parameters = (
            _parameters_from_python_marginal_row(
                row
            )
        )

        parameters = (
            _validate_distribution_parameters(
                family=family,
                parameters=parameters,
            )
        )

        marginal_specs.append(
            {
                "component":
                    str(
                        row[
                            "component"
                        ]
                    ),
                "family":
                    family,
                "parameters":
                    parameters,
            }
        )

    # --------------------------------------------------------
    # LOAD PRACTICE TIME-VARYING 16D LATENT CORRELATION
    # --------------------------------------------------------

    latent_df = pd.read_csv(
        latent_correlation_path
    )

    required_latent_columns = {
        "t_original",
        "component_1_index",
        "component_2_index",
        "component_1",
        "component_2",
        "correlation",
    }

    if not required_latent_columns.issubset(
        latent_df.columns
    ):
        raise ValueError(
            "Practice innovation_latent_correlation.csv is "
            "missing required columns."
        )

    production_times = np.sort(
        latent_df[
            "t_original"
        ].unique().astype(
            int
        )
    )

    if len(
        production_times
    ) != expected_time_count:
        raise ValueError(
            "Practice latent-correlation time count does not "
            "match metadata."
        )

    expected_production_times = np.arange(
        expected_time_count,
        dtype=int,
    )

    if not np.array_equal(
        production_times,
        expected_production_times,
    ):
        raise ValueError(
            "Practice production times must be consecutive "
            "integers from 0 through T - 1."
        )

    expected_latent_rows = (
        expected_time_count
        * 16
        * 16
    )

    if len(
        latent_df
    ) != expected_latent_rows:
        raise ValueError(
            "Practice innovation_latent_correlation.csv has "
            "an unexpected row count."
        )

    latent_correlation = np.empty(
        (
            expected_time_count,
            16,
            16,
        ),
        dtype=float,
    )

    minimum_eigenvalue = np.inf

    for time_index, t_original in enumerate(
        production_times
    ):

        current_rows = latent_df[
            latent_df[
                "t_original"
            ].astype(
                int
            )
            == int(
                t_original
            )
        ].copy()

        if len(
            current_rows
        ) != 256:
            raise ValueError(
                "Each practice interval must contain exactly "
                "256 latent-correlation rows."
            )

        current_rows = current_rows.sort_values(
            [
                "component_1_index",
                "component_2_index",
            ]
        )

        matrix = np.empty(
            (
                16,
                16,
            ),
            dtype=float,
        )

        seen_pairs = set()

        for row in current_rows.itertuples(
            index=False
        ):

            i = int(
                row.component_1_index
            ) - 1

            j = int(
                row.component_2_index
            ) - 1

            if not (
                0 <= i < 16
                and 0 <= j < 16
            ):
                raise ValueError(
                    "Practice latent component index is "
                    "outside 1 through 16."
                )

            pair = (
                i,
                j,
            )

            if pair in seen_pairs:
                raise ValueError(
                    "Duplicate practice latent-correlation "
                    "matrix entry."
                )

            seen_pairs.add(
                pair
            )

            if (
                str(
                    row.component_1
                )
                != component_order[
                    i
                ]
            ):
                raise ValueError(
                    "Practice component_1 name/index mismatch."
                )

            if (
                str(
                    row.component_2
                )
                != component_order[
                    j
                ]
            ):
                raise ValueError(
                    "Practice component_2 name/index mismatch."
                )

            matrix[
                i,
                j
            ] = float(
                row.correlation
            )

        if len(
            seen_pairs
        ) != 256:
            raise ValueError(
                "Incomplete practice latent correlation matrix."
            )

        if not np.all(
            np.isfinite(
                matrix
            )
        ):
            raise ValueError(
                "Practice latent correlation contains "
                "non-finite values."
            )

        symmetry_error = float(
            np.max(
                np.abs(
                    matrix
                    - matrix.T
                )
            )
        )

        if symmetry_error > 1e-12:
            raise ValueError(
                "Practice latent correlation matrix is not "
                "symmetric."
            )

        diagonal_error = float(
            np.max(
                np.abs(
                    np.diag(
                        matrix
                    )
                    - 1.0
                )
            )
        )

        if diagonal_error > 1e-12:
            raise ValueError(
                "Practice latent correlation matrix does not "
                "have unit diagonal."
            )

        try:

            np.linalg.cholesky(
                matrix
            )

        except np.linalg.LinAlgError as exc:

            raise ValueError(
                "Practice latent correlation matrix is not "
                f"positive definite at t={t_original}."
            ) from exc

        current_minimum_eigenvalue = float(
            np.min(
                np.linalg.eigvalsh(
                    matrix
                )
            )
        )

        minimum_eigenvalue = min(
            minimum_eigenvalue,
            current_minimum_eigenvalue,
        )

        latent_correlation[
            time_index
        ] = matrix

    time_to_index = {
        int(
            t_original
        ):
            int(
                time_index
            )
        for time_index, t_original
        in enumerate(
            production_times
        )
    }

    innovation_system = {
        "component_order":
            component_order,
        "marginal_specs":
            marginal_specs,
        "production_times":
            production_times,
        "time_to_index":
            time_to_index,
        "latent_correlation":
            latent_correlation,
        "final_state_copula":
            final_state_copula,
        "final_state_student_t_df":
            final_state_student_t_df,
        "practice_mode":
            metadata[
                "practice_mode"
            ],
        "metadata":
            metadata,
        "minimum_latent_eigenvalue":
            minimum_eigenvalue,
    }

    return innovation_system

def build_joint_innovations(
    t_original,
    n_sims,
    rng,
    innovation_system,
    clip_probability=1e-10,
    return_details=False,
):
    """
    Generate one interval of the frozen 16D production
    innovation system.

    Parameters
    ----------
    t_original:
        Original production time index.

    n_sims:
        Number of Monte Carlo simulations.

    rng:
        numpy.random.Generator.

    innovation_system:
        Output of load_python_innovation_system().

    clip_probability:
        Probability clipping rule used by the frozen R
        simulator. Default is 1e-10.

    return_details:
        If False, return only Z_sims with shape (N, 16).

        If True, return a dictionary containing the final
        innovations plus the intermediate Gaussian driver,
        uniforms, FinalState t driver, and shared scale.

    Returns
    -------
    Z_sims:
        Shape (N, 16), in canonical component order.
    """

    if not isinstance(
        n_sims,
        (int, np.integer),
    ):
        raise TypeError(
            "n_sims must be an integer."
        )

    if n_sims < 1:
        raise ValueError(
            "n_sims must be at least 1."
        )

    if (
        not np.isfinite(
            clip_probability
        )
        or clip_probability <= 0.0
        or clip_probability >= 0.5
    ):
        raise ValueError(
            "clip_probability must lie strictly "
            "between 0 and 0.5."
        )

    time_to_index = innovation_system[
        "time_to_index"
    ]

    t_original = int(
        t_original
    )

    if t_original not in time_to_index:
        raise ValueError(
            f"Unknown production t_original: {t_original}"
        )

    time_index = time_to_index[
        t_original
    ]

    latent_correlation = innovation_system[
        "latent_correlation"
    ][
        time_index
    ]

    component_order = innovation_system[
        "component_order"
    ]

    marginal_specs = innovation_system[
        "marginal_specs"
    ]

    final_state_copula = str(
        innovation_system.get(
            "final_state_copula",
            "student_t",
        )
    ).strip().lower()

    if final_state_copula not in (
        "gaussian",
        "student_t",
    ):
        raise ValueError(
            "final_state_copula must be either "
            "'gaussian' or 'student_t'."
        )

    if final_state_copula == "student_t":

        final_state_student_t_df = float(
            innovation_system[
                "final_state_student_t_df"
            ]
        )

        if (
            not np.isfinite(
                final_state_student_t_df
            )
            or final_state_student_t_df <= 2.0
        ):
            raise ValueError(
                "Student-t FinalState copula requires "
                "finite df > 2."
            )

    else:

        final_state_student_t_df = np.nan

    if (
        component_order
        != CANONICAL_INNOVATION_COMPONENT_ORDER
    ):
        raise ValueError(
            "Innovation-system component order changed."
        )

    if len(
        marginal_specs
    ) != 16:
        raise ValueError(
            "Innovation system must contain "
            "16 marginal specifications."
        )

    if latent_correlation.shape != (
        16,
        16,
    ):
        raise ValueError(
            "Latent correlation matrix must have "
            "shape (16, 16)."
        )

    # --------------------------------------------------------
    # 1. 16D GAUSSIAN DRIVER
    #
    # NumPy cholesky returns L such that:
    #
    #     R = L @ L.T
    #
    # Therefore independent row vectors are multiplied by L.T.
    # --------------------------------------------------------

    cholesky_factor = np.linalg.cholesky(
        latent_correlation
    )

    independent_normals = rng.standard_normal(
        size=(
            n_sims,
            16,
        )
    )

    gaussian_driver = (
        independent_normals
        @ cholesky_factor.T
    )

    if gaussian_driver.shape != (
        n_sims,
        16,
    ):
        raise AssertionError(
            "Gaussian driver has incorrect shape."
        )

    # --------------------------------------------------------
    # 2. BUILD COPULA UNIFORMS
    # --------------------------------------------------------

    uniform_matrix = np.empty(
        (
            n_sims,
            16,
        ),
        dtype=float,
    )

    # Futures Gaussian copula.
    uniform_matrix[
        :,
        FUTURES_INNOVATION_INDICES
    ] = norm.cdf(
        gaussian_driver[
            :,
            FUTURES_INNOVATION_INDICES
        ]
    )

    # --------------------------------------------------------
    # 3. FINAL-STATE COPULA
    #
    # Production:
    #     Student-t copula with one shared scale.
    #
    # Gaussian compatibility practice mode:
    #     Gaussian copula.
    #
    # The default remains Student-t, so the frozen production
    # architecture is unchanged.
    # --------------------------------------------------------

    if final_state_copula == "student_t":

        final_scale = np.sqrt(
            rng.chisquare(
                df=final_state_student_t_df,
                size=n_sims,
            )
            / final_state_student_t_df
        )

        final_t_driver = (
            gaussian_driver[
                :,
                FINAL_STATE_INNOVATION_INDICES
            ]
            / final_scale[
                :,
                None
            ]
        )

        uniform_matrix[
            :,
            FINAL_STATE_INNOVATION_INDICES
        ] = student_t_dist.cdf(
            final_t_driver,
            df=final_state_student_t_df,
        )

    else:

        final_scale = np.ones(
            n_sims,
            dtype=float,
        )

        final_t_driver = (
            gaussian_driver[
                :,
                FINAL_STATE_INNOVATION_INDICES
            ].copy()
        )

        uniform_matrix[
            :,
            FINAL_STATE_INNOVATION_INDICES
        ] = norm.cdf(
            final_t_driver
        )

    # DailyState retained Gaussian dependence.
    uniform_matrix[
        :,
        DAILY_STATE_INNOVATION_INDICES
    ] = norm.cdf(
        gaussian_driver[
            :,
            DAILY_STATE_INNOVATION_INDICES
        ]
    )

    # Match the frozen R simulator probability clipping rule.
    uniform_matrix = np.clip(
        uniform_matrix,
        clip_probability,
        1.0 - clip_probability,
    )

    if not np.all(
        np.isfinite(
            uniform_matrix
        )
    ):
        raise ValueError(
            "Joint innovation uniforms contain "
            "non-finite values."
        )

    if not np.all(
        (
            uniform_matrix > 0.0
        )
        & (
            uniform_matrix < 1.0
        )
    ):
        raise ValueError(
            "Joint innovation uniforms must lie "
            "strictly inside (0, 1)."
        )

    # --------------------------------------------------------
    # 4. APPLY FROZEN STEP 6.1 MARGINAL INVERSE CDFS
    # --------------------------------------------------------

    Z_sims = np.empty(
        (
            n_sims,
            16,
        ),
        dtype=float,
    )

    for component_index, specification in enumerate(
        marginal_specs
    ):

        if (
            specification[
                "component"
            ]
            != component_order[
                component_index
            ]
        ):
            raise ValueError(
                "Marginal specification order mismatch."
            )

        Z_sims[
            :,
            component_index
        ] = distribution_ppf(
            p=uniform_matrix[
                :,
                component_index
            ],
            family=specification[
                "family"
            ],
            parameters=specification[
                "parameters"
            ],
        )

    if Z_sims.shape != (
        n_sims,
        16,
    ):
        raise AssertionError(
            "Joint innovation matrix has incorrect shape."
        )

    if not np.all(
        np.isfinite(
            Z_sims
        )
    ):
        raise ValueError(
            "Joint innovation matrix contains "
            "non-finite values."
        )

    if return_details:

        return {
            "Z_sims":
                Z_sims,
            "uniforms":
                uniform_matrix,
            "gaussian_driver":
                gaussian_driver,
            "final_t_driver":
                final_t_driver,
            "final_scale":
                final_scale,
            "t_original":
                t_original,
            "time_index":
                time_index,
            "component_order":
                component_order.copy(),
        }

    return Z_sims

# ============================================================
# STEP 6.3 - FROZEN 50.25 FUTURES ARITHMETIC-RETURN MAPPING
#
# This section ports the finalized 50.25 arithmetic-return
# mapping to Python.
#
# IMPORTANT ARCHITECTURE:
#
#   Cross-future dependence:
#       determined by the frozen 50.15 joint coupling.
#
#   Arithmetic marginal mean:
#       mu[t, k]
#
#   Arithmetic marginal variance:
#       Sigma[t, k, k]
#
#   Old Sigma off-diagonal values:
#       benchmark / diagnostic only in the NEW simulator.
#
# The new mapping functions deliberately accept only a vector
# of marginal variances, not a full covariance matrix. This
# prevents accidental double-imposition of dependence.
#
# This section does NOT yet modify simulate_competition_paths().
# ============================================================


FUTURES_PRODUCTION_ORDER = [
    "CL",
    "GC",
    "SR3",
    "ZN",
    "KE",
    "HO",
    "ZW",
    "NQ",
]


EXPECTED_ARITHMETIC_MAPPING_FAMILY = (
    "Monotone_Standardized_Soft_Lower_Floor"
)


def load_python_arithmetic_mapping(
    production_dir=None,
):
    """
    Load the frozen 50.25 Python-facing arithmetic-return
    mapping exported by Notebook 50.4.

    Returns
    -------
    mapping_system:
        Dictionary containing:

        futures_order
        floor_parameter
        raw_mean
        raw_sd
        return_floor
        return_floor_epsilon
        mapping_family
        parameter_table
    """

    if production_dir is None:
        production_dir = (
            DEFAULT_PYTHON_INNOVATION_PRODUCTION_DIR
        )

    mapping_path = os.path.join(
        production_dir,
        "innovation_arithmetic_mapping.csv",
    )

    metadata_path = os.path.join(
        production_dir,
        "innovation_system_metadata.csv",
    )

    if not os.path.exists(
        mapping_path
    ):
        raise FileNotFoundError(
            mapping_path
        )

    if not os.path.exists(
        metadata_path
    ):
        raise FileNotFoundError(
            metadata_path
        )

    # --------------------------------------------------------
    # VALIDATE FROZEN 50.25 METADATA
    # --------------------------------------------------------

    metadata_df = pd.read_csv(
        metadata_path,
        dtype=str,
        keep_default_na=False,
    )

    if not {
        "key",
        "value",
    }.issubset(
        metadata_df.columns
    ):
        raise ValueError(
            "innovation_system_metadata.csv must contain "
            "columns 'key' and 'value'."
        )

    if metadata_df["key"].duplicated().any():
        raise ValueError(
            "Duplicate metadata keys found."
        )

    metadata = dict(
        zip(
            metadata_df["key"],
            metadata_df["value"],
        )
    )

    required_metadata_keys = {
        "arithmetic_mapping_version",
        "arithmetic_mapping_family",
        "futures_order",
        "arithmetic_return_mapping_final",
        "old_off_diagonal_arithmetic_covariance_policy",
    }

    missing_metadata_keys = (
        required_metadata_keys
        - set(
            metadata.keys()
        )
    )

    if missing_metadata_keys:
        raise ValueError(
            "innovation_system_metadata.csv is missing "
            "arithmetic-mapping keys: "
            f"{sorted(missing_metadata_keys)}"
        )

    if (
        metadata["arithmetic_mapping_version"]
        != "50.25"
    ):
        raise ValueError(
            "Unexpected arithmetic mapping version."
        )

    if (
        metadata["arithmetic_mapping_family"]
        != EXPECTED_ARITHMETIC_MAPPING_FAMILY
    ):
        raise ValueError(
            "Unexpected arithmetic mapping family."
        )

    if (
        metadata["arithmetic_return_mapping_final"]
        != "TRUE"
    ):
        raise ValueError(
            "Arithmetic return mapping is not final."
        )

    if (
        metadata[
            "old_off_diagonal_arithmetic_covariance_policy"
        ]
        != "Benchmark_Only"
    ):
        raise ValueError(
            "Old arithmetic covariance policy must be "
            "Benchmark_Only."
        )

    metadata_futures_order = (
        metadata[
            "futures_order"
        ].split("|")
    )

    if (
        metadata_futures_order
        != FUTURES_PRODUCTION_ORDER
    ):
        raise ValueError(
            "Futures order in metadata does not match "
            "the canonical production order."
        )

    # --------------------------------------------------------
    # LOAD FROZEN MAPPING PARAMETERS
    # --------------------------------------------------------

    mapping_df = pd.read_csv(
        mapping_path
    )

    required_mapping_columns = {
        "future",
        "future_index",
        "mapping_family",
        "floor_parameter",
        "raw_mean",
        "raw_sd",
        "return_floor",
        "return_floor_epsilon",
    }

    if not required_mapping_columns.issubset(
        mapping_df.columns
    ):
        raise ValueError(
            "innovation_arithmetic_mapping.csv is missing "
            "required columns."
        )

    if len(
        mapping_df
    ) != 8:
        raise ValueError(
            "innovation_arithmetic_mapping.csv must contain "
            "exactly eight rows."
        )

    mapping_df = mapping_df.sort_values(
        "future_index"
    ).reset_index(
        drop=True
    )

    if (
        mapping_df[
            "future_index"
        ].to_numpy(
            dtype=int
        ).tolist()
        != list(
            range(
                1,
                9,
            )
        )
    ):
        raise ValueError(
            "Arithmetic mapping future indices must be "
            "1 through 8."
        )

    if (
        mapping_df[
            "future"
        ].tolist()
        != FUTURES_PRODUCTION_ORDER
    ):
        raise ValueError(
            "Arithmetic mapping futures are not in "
            "canonical production order."
        )

    if not np.all(
        mapping_df[
            "mapping_family"
        ].to_numpy(
            dtype=str
        )
        == EXPECTED_ARITHMETIC_MAPPING_FAMILY
    ):
        raise ValueError(
            "Arithmetic mapping family mismatch."
        )

    floor_parameter = mapping_df[
        "floor_parameter"
    ].to_numpy(
        dtype=float
    )

    raw_mean = mapping_df[
        "raw_mean"
    ].to_numpy(
        dtype=float
    )

    raw_sd = mapping_df[
        "raw_sd"
    ].to_numpy(
        dtype=float
    )

    return_floor_values = mapping_df[
        "return_floor"
    ].to_numpy(
        dtype=float
    )

    return_floor_epsilon_values = mapping_df[
        "return_floor_epsilon"
    ].to_numpy(
        dtype=float
    )

    if not np.all(
        np.isfinite(
            floor_parameter
        )
    ):
        raise ValueError(
            "Floor parameters must be finite."
        )

    if not np.all(
        np.isfinite(
            raw_mean
        )
    ):
        raise ValueError(
            "Raw transformed means must be finite."
        )

    if not np.all(
        np.isfinite(
            raw_sd
        )
    ):
        raise ValueError(
            "Raw transformed SDs must be finite."
        )

    if np.any(
        raw_sd <= 0.0
    ):
        raise ValueError(
            "Raw transformed SDs must be positive."
        )

    if not np.all(
        np.isfinite(
            return_floor_values
        )
    ):
        raise ValueError(
            "Return-floor values must be finite."
        )

    if not np.all(
        return_floor_values == -1.0
    ):
        raise ValueError(
            "Frozen arithmetic return floor must be -1."
        )

    if not np.all(
        np.isfinite(
            return_floor_epsilon_values
        )
    ):
        raise ValueError(
            "Return-floor epsilon values must be finite."
        )

    if np.any(
        return_floor_epsilon_values <= 0.0
    ):
        raise ValueError(
            "Return-floor epsilon values must be positive."
        )

    if not np.allclose(
        return_floor_values,
        return_floor_values[0],
        rtol=0.0,
        atol=0.0,
    ):
        raise ValueError(
            "Return floor must be identical across futures."
        )

    if not np.allclose(
        return_floor_epsilon_values,
        return_floor_epsilon_values[0],
        rtol=0.0,
        atol=0.0,
    ):
        raise ValueError(
            "Return-floor epsilon must be identical "
            "across futures."
        )

    return {
        "futures_order":
            FUTURES_PRODUCTION_ORDER.copy(),
        "floor_parameter":
            floor_parameter,
        "raw_mean":
            raw_mean,
        "raw_sd":
            raw_sd,
        "return_floor":
            float(
                return_floor_values[0]
            ),
        "return_floor_epsilon":
            float(
                return_floor_epsilon_values[0]
            ),
        "mapping_family":
            EXPECTED_ARITHMETIC_MAPPING_FAMILY,
        "parameter_table":
            mapping_df.copy(),
        "metadata":
            metadata,
    }

# ============================================================
# STEP 7B - PRACTICE ARITHMETIC MAPPING FILE LOADER
#
# Load the heterogeneous practice futures mapping serialized
# by Step 7A.
#
# This loader remains separate from the frozen production
# 50.25 loader.
# ============================================================


def load_practice_arithmetic_mapping(
    input_dir,
):
    """
    Load and validate the Step 7A practice arithmetic-return
    mapping.

    Parameters
    ----------
    input_dir:
        Directory containing:

            innovation_arithmetic_mapping.csv
            innovation_system_metadata.csv

    Returns
    -------
    mapping_system:
        Dictionary compatible with
        map_standardized_futures_innovations().
    """

    if input_dir is None:
        raise ValueError(
            "input_dir must be provided for the practice "
            "arithmetic mapping loader."
        )

    mapping_path = os.path.join(
        input_dir,
        "innovation_arithmetic_mapping.csv",
    )

    metadata_path = os.path.join(
        input_dir,
        "innovation_system_metadata.csv",
    )

    if not os.path.exists(
        mapping_path
    ):
        raise FileNotFoundError(
            mapping_path
        )

    if not os.path.exists(
        metadata_path
    ):
        raise FileNotFoundError(
            metadata_path
        )

    # --------------------------------------------------------
    # LOAD PRACTICE METADATA
    # --------------------------------------------------------

    metadata_df = pd.read_csv(
        metadata_path,
        dtype=str,
        keep_default_na=False,
    )

    if not {
        "key",
        "value",
    }.issubset(
        metadata_df.columns
    ):
        raise ValueError(
            "Practice innovation_system_metadata.csv must "
            "contain columns 'key' and 'value'."
        )

    if metadata_df["key"].duplicated().any():
        raise ValueError(
            "Duplicate practice metadata keys found."
        )

    metadata = dict(
        zip(
            metadata_df[
                "key"
            ],
            metadata_df[
                "value"
            ],
        )
    )

    required_metadata_keys = {
        "interface_version",
        "system_type",
        "serialization_step",
        "practice_mode",
        "futures_order",
        "arithmetic_mapping_family",
        "return_floor",
        "return_floor_epsilon",
    }

    missing_metadata_keys = (
        required_metadata_keys
        - set(
            metadata.keys()
        )
    )

    if missing_metadata_keys:
        raise ValueError(
            "Practice arithmetic-mapping metadata is "
            "missing keys: "
            f"{sorted(missing_metadata_keys)}"
        )

    if (
        metadata[
            "interface_version"
        ]
        != "practice_7A_1.0.0"
    ):
        raise ValueError(
            "Unexpected practice mapping interface version."
        )

    if (
        metadata[
            "system_type"
        ]
        != "practice_heterogeneous_distributional"
    ):
        raise ValueError(
            "Unexpected practice mapping system type."
        )

    if (
        metadata[
            "serialization_step"
        ]
        != "7A"
    ):
        raise ValueError(
            "Practice mapping was not serialized by Step 7A."
        )

    if (
        metadata[
            "practice_mode"
        ]
        != "heterogeneous_distributional"
    ):
        raise ValueError(
            "Unexpected practice mapping mode."
        )

    metadata_futures_order = (
        metadata[
            "futures_order"
        ].split("|")
    )

    if (
        metadata_futures_order
        != FUTURES_PRODUCTION_ORDER
    ):
        raise ValueError(
            "Practice mapping futures order does not match "
            "the canonical futures order."
        )

    expected_mapping_family = (
        "Practice_Standardized_Soft_Lower_Floor"
    )

    if (
        metadata[
            "arithmetic_mapping_family"
        ]
        != expected_mapping_family
    ):
        raise ValueError(
            "Unexpected practice arithmetic mapping family."
        )

    metadata_return_floor = float(
        metadata[
            "return_floor"
        ]
    )

    metadata_return_floor_epsilon = float(
        metadata[
            "return_floor_epsilon"
        ]
    )

    if (
        not np.isfinite(
            metadata_return_floor
        )
        or metadata_return_floor != -1.0
    ):
        raise ValueError(
            "Practice return floor must equal -1."
        )

    if (
        not np.isfinite(
            metadata_return_floor_epsilon
        )
        or metadata_return_floor_epsilon <= 0.0
    ):
        raise ValueError(
            "Practice return-floor epsilon must be "
            "finite and positive."
        )

    # --------------------------------------------------------
    # LOAD PRACTICE MAPPING TABLE
    # --------------------------------------------------------

    mapping_df = pd.read_csv(
        mapping_path
    )

    required_mapping_columns = {
        "future",
        "future_index",
        "mapping_family",
        "floor_parameter",
        "raw_mean",
        "raw_sd",
        "return_floor",
        "return_floor_epsilon",
    }

    if not required_mapping_columns.issubset(
        mapping_df.columns
    ):
        raise ValueError(
            "Practice innovation_arithmetic_mapping.csv is "
            "missing required columns."
        )

    if len(
        mapping_df
    ) != 8:
        raise ValueError(
            "Practice innovation_arithmetic_mapping.csv must "
            "contain exactly eight rows."
        )

    mapping_df = mapping_df.sort_values(
        "future_index"
    ).reset_index(
        drop=True
    )

    if (
        mapping_df[
            "future_index"
        ].to_numpy(
            dtype=int
        ).tolist()
        != list(
            range(
                1,
                9,
            )
        )
    ):
        raise ValueError(
            "Practice mapping future indices must be "
            "1 through 8."
        )

    if (
        mapping_df[
            "future"
        ].tolist()
        != FUTURES_PRODUCTION_ORDER
    ):
        raise ValueError(
            "Practice mapping futures are not in canonical "
            "futures order."
        )

    if not np.all(
        mapping_df[
            "mapping_family"
        ].to_numpy(
            dtype=str
        )
        == expected_mapping_family
    ):
        raise ValueError(
            "Practice mapping-family column mismatch."
        )

    floor_parameter = mapping_df[
        "floor_parameter"
    ].to_numpy(
        dtype=float
    )

    raw_mean = mapping_df[
        "raw_mean"
    ].to_numpy(
        dtype=float
    )

    raw_sd = mapping_df[
        "raw_sd"
    ].to_numpy(
        dtype=float
    )

    return_floor_values = mapping_df[
        "return_floor"
    ].to_numpy(
        dtype=float
    )

    return_floor_epsilon_values = mapping_df[
        "return_floor_epsilon"
    ].to_numpy(
        dtype=float
    )

    for values, field_name in [
        (
            floor_parameter,
            "floor_parameter",
        ),
        (
            raw_mean,
            "raw_mean",
        ),
        (
            raw_sd,
            "raw_sd",
        ),
        (
            return_floor_values,
            "return_floor",
        ),
        (
            return_floor_epsilon_values,
            "return_floor_epsilon",
        ),
    ]:

        if not np.all(
            np.isfinite(
                values
            )
        ):
            raise ValueError(
                f"Practice mapping field {field_name} "
                "contains non-finite values."
            )

    if np.any(
        raw_sd <= 0.0
    ):
        raise ValueError(
            "Practice raw transformed SDs must be positive."
        )

    if not np.all(
        return_floor_values == -1.0
    ):
        raise ValueError(
            "Practice arithmetic return floor must be -1."
        )

    if np.any(
        return_floor_epsilon_values <= 0.0
    ):
        raise ValueError(
            "Practice return-floor epsilon values must be "
            "positive."
        )

    if not np.allclose(
        return_floor_values,
        return_floor_values[
            0
        ],
        rtol=0.0,
        atol=0.0,
    ):
        raise ValueError(
            "Practice return floor must be identical across "
            "all futures."
        )

    if not np.allclose(
        return_floor_epsilon_values,
        return_floor_epsilon_values[
            0
        ],
        rtol=0.0,
        atol=0.0,
    ):
        raise ValueError(
            "Practice return-floor epsilon must be identical "
            "across all futures."
        )

    if not np.isclose(
        float(
            return_floor_values[
                0
            ]
        ),
        metadata_return_floor,
        rtol=0.0,
        atol=0.0,
    ):
        raise ValueError(
            "Practice return floor does not match metadata."
        )

    if not np.isclose(
        float(
            return_floor_epsilon_values[
                0
            ]
        ),
        metadata_return_floor_epsilon,
        rtol=0.0,
        atol=0.0,
    ):
        raise ValueError(
            "Practice return-floor epsilon does not match "
            "metadata."
        )

    return {
        "futures_order":
            FUTURES_PRODUCTION_ORDER.copy(),
        "floor_parameter":
            floor_parameter,
        "raw_mean":
            raw_mean,
        "raw_sd":
            raw_sd,
        "return_floor":
            float(
                return_floor_values[
                    0
                ]
            ),
        "return_floor_epsilon":
            float(
                return_floor_epsilon_values[
                    0
                ]
            ),
        "mapping_family":
            expected_mapping_family,
        "practice_mode":
            metadata[
                "practice_mode"
            ],
        "parameter_table":
            mapping_df.copy(),
        "metadata":
            metadata,
    }

def _stable_softplus(
    x,
):
    """
    Numerically stable implementation of:

        log(1 + exp(x))

    matching the frozen 50.25 R implementation.
    """

    x_array = np.asarray(
        x,
        dtype=float,
    )

    output = np.empty_like(
        x_array,
        dtype=float,
    )

    high_mask = (
        x_array > 35.0
    )

    low_mask = (
        x_array < -35.0
    )

    middle_mask = (
        ~high_mask
        & ~low_mask
    )

    output[
        high_mask
    ] = x_array[
        high_mask
    ]

    output[
        low_mask
    ] = np.exp(
        x_array[
            low_mask
        ]
    )

    output[
        middle_mask
    ] = np.log1p(
        np.exp(
            x_array[
                middle_mask
            ]
        )
    )

    if np.ndim(
        x
    ) == 0:
        return float(
            output
        )

    return output


def soft_floor_raw(
    z,
    floor_parameter,
):
    """
    Frozen 50.25 raw soft-floor transformation:

        c + softplus(z - c)

    floor_parameter may be a scalar or a broadcast-compatible
    NumPy array.
    """

    z_array = np.asarray(
        z,
        dtype=float,
    )

    floor_parameter_array = np.asarray(
        floor_parameter,
        dtype=float,
    )

    if not np.all(
        np.isfinite(
            floor_parameter_array
        )
    ):
        raise ValueError(
            "floor_parameter must contain only finite values."
        )

    result = (
        floor_parameter_array
        + _stable_softplus(
            z_array
            - floor_parameter_array
        )
    )

    if (
        np.ndim(
            z
        ) == 0
        and np.ndim(
            floor_parameter
        ) == 0
    ):
        return float(
            result
        )

    return result


def map_standardized_futures_innovations(
    Z_futures,
    mapping_system,
):
    """
    Apply the frozen 50.25 componentwise standardized
    soft-floor mapping.

    Parameters
    ----------
    Z_futures:
        Shape (N, 8), in canonical futures order.

    mapping_system:
        Output of load_python_arithmetic_mapping().

    Returns
    -------
    mapped_Z:
        Shape (N, 8).

        Each column is re-standardized using the frozen
        50.25 raw transformed mean and SD.
    """

    Z_futures = np.asarray(
        Z_futures,
        dtype=float,
    )

    if Z_futures.ndim != 2:
        raise ValueError(
            "Z_futures must be a two-dimensional array."
        )

    if Z_futures.shape[1] != 8:
        raise ValueError(
            "Z_futures must have shape (N, 8)."
        )

    if not np.all(
        np.isfinite(
            Z_futures
        )
    ):
        raise ValueError(
            "Z_futures must contain only finite values."
        )

    if (
        mapping_system[
            "futures_order"
        ]
        != FUTURES_PRODUCTION_ORDER
    ):
        raise ValueError(
            "Arithmetic mapping futures order changed."
        )

    floor_parameter = np.asarray(
        mapping_system[
            "floor_parameter"
        ],
        dtype=float,
    )

    raw_mean = np.asarray(
        mapping_system[
            "raw_mean"
        ],
        dtype=float,
    )

    raw_sd = np.asarray(
        mapping_system[
            "raw_sd"
        ],
        dtype=float,
    )

    if floor_parameter.shape != (
        8,
    ):
        raise ValueError(
            "floor_parameter must have shape (8,)."
        )

    if raw_mean.shape != (
        8,
    ):
        raise ValueError(
            "raw_mean must have shape (8,)."
        )

    if raw_sd.shape != (
        8,
    ):
        raise ValueError(
            "raw_sd must have shape (8,)."
        )

    if np.any(
        raw_sd <= 0.0
    ):
        raise ValueError(
            "raw_sd must be strictly positive."
        )

    transformed = soft_floor_raw(
        Z_futures,
        floor_parameter[
            None,
            :
        ],
    )

    mapped_Z = (
        transformed
        - raw_mean[
            None,
            :
        ]
    ) / raw_sd[
        None,
        :
    ]

    if mapped_Z.shape != Z_futures.shape:
        raise AssertionError(
            "Mapped innovation matrix changed shape."
        )

    if not np.all(
        np.isfinite(
            mapped_Z
        )
    ):
        raise ValueError(
            "Mapped innovations contain non-finite values."
        )

    return mapped_Z


def map_futures_innovations_to_arithmetic_returns(
    Z_futures,
    mu_t,
    variance_t,
    mapping_system,
):
    """
    Convert selected futures innovations into arithmetic
    futures returns using the frozen 50.25 mapping.

    IMPORTANT:
        variance_t is deliberately a VECTOR with shape (8,).

        This function does NOT accept the full 8x8 Sigma
        matrix. The new cross-future dependence already comes
        from Step 6.2 / the frozen 50.15 coupling.

        Only the diagonal marginal variances remain
        authoritative here.

    Formula
    -------
        mapped_z_k
            =
        (
            soft_floor_raw(z_k, c_k)
            - raw_mean_k
        )
        / raw_sd_k

        R_tk
            =
        mu_tk
        + sqrt(variance_tk) * mapped_z_k
    """

    Z_futures = np.asarray(
        Z_futures,
        dtype=float,
    )

    mu_t = np.asarray(
        mu_t,
        dtype=float,
    )

    variance_t = np.asarray(
        variance_t,
        dtype=float,
    )

    if Z_futures.ndim != 2:
        raise ValueError(
            "Z_futures must be a two-dimensional array."
        )

    if Z_futures.shape[1] != 8:
        raise ValueError(
            "Z_futures must have shape (N, 8)."
        )

    if mu_t.shape != (
        8,
    ):
        raise ValueError(
            "mu_t must have shape (8,)."
        )

    if variance_t.shape != (
        8,
    ):
        raise ValueError(
            "variance_t must have shape (8,). "
            "Do not pass the full Sigma matrix."
        )

    if not np.all(
        np.isfinite(
            mu_t
        )
    ):
        raise ValueError(
            "mu_t must contain only finite values."
        )

    if not np.all(
        np.isfinite(
            variance_t
        )
    ):
        raise ValueError(
            "variance_t must contain only finite values."
        )

    if np.any(
        variance_t <= 0.0
    ):
        raise ValueError(
            "variance_t must be strictly positive."
        )

    mapped_Z = (
        map_standardized_futures_innovations(
            Z_futures=Z_futures,
            mapping_system=mapping_system,
        )
    )

    target_sd = np.sqrt(
        variance_t
    )

    arithmetic_returns = (
        mu_t[
            None,
            :
        ]
        + target_sd[
            None,
            :
        ]
        * mapped_Z
    )

    if arithmetic_returns.shape != Z_futures.shape:
        raise AssertionError(
            "Arithmetic-return matrix changed shape."
        )

    if not np.all(
        np.isfinite(
            arithmetic_returns
        )
    ):
        raise ValueError(
            "Arithmetic returns contain non-finite values."
        )

    return arithmetic_returns

def make_synthetic_problem(
    T,
    K=8,
    C=8,
    seed=20260813,
):
    """
    Load futures forecasts plus the JOINT eight-dimensional
    competition-state forecasts.

    State order:

        0  Delta_Final_X1
        1  Delta_Final_D12
        2  Delta_Final_D23
        3  Delta_Final_D34
        4  Delta_Final_D4_15
        5  Delta_Final_D15_16
        6  Delta_Daily_H1
        7  Delta_Daily_D12

    Returns
    -------
    mu:
        shape (T, K)

    Sigma:
        shape (T, K, K)

    state_mean_competition:
        shape (T, 8)

    state_cov_competition:
        shape (T, 8, 8)

    state_futures_cov_competition:
        shape (8, T, K)
    """

    DATA_DIR = ACTIVE_FORECAST_DIR

    future_names = [
        f"Future_{k + 1}"
        for k in range(K)
    ]

    state_names = [
        "Delta_Final_X1",
        "Delta_Final_D12",
        "Delta_Final_D23",
        "Delta_Final_D34",
        "Delta_Final_D4_15",
        "Delta_Final_D15_16",
        "Delta_Daily_H1",
        "Delta_Daily_D12",
    ]

    C_state = len(state_names)

    if C != C_state:
        raise ValueError(
            "Joint competition state must contain "
            "exactly eight components."
        )


    # ========================================================
    # HELPER: ORIGINAL TIME INDEX
    # ========================================================

    def get_original_times(df):

        if "t_original" in df.columns:

            times = df[
                "t_original"
            ].to_numpy(
                dtype=int
            )

        else:

            times = df[
                "t"
            ].to_numpy(
                dtype=int
            )

        if (
            np.any(
                times < 0
            )
            or np.any(
                times >= T
            )
        ):

            raise ValueError(
                "CSV contains time indices "
                f"outside [0, {T - 1}]."
            )

        if (
            len(
                np.unique(
                    times
                )
            )
            != len(
                times
            )
        ):

            raise ValueError(
                "Duplicate original time indices found."
            )

        return times


    # ========================================================
    # 1. mu
    # ========================================================

    mu_df = pd.read_csv(
        os.path.join(
            DATA_DIR,
            "mu.csv",
        )
    )

    missing_futures = [
        name
        for name in future_names
        if name not in mu_df.columns
    ]

    if missing_futures:

        raise ValueError(
            "mu.csv is missing futures: "
            f"{missing_futures}"
        )

    mu_times = get_original_times(
        mu_df
    )

    mu = np.zeros(
        (
            T,
            K,
        ),
        dtype=float,
    )

    mu[
        mu_times,
        :
    ] = (
        mu_df[
            future_names
        ]
        .to_numpy(
            dtype=float
        )
    )


    # ========================================================
    # 2. Sigma
    # ========================================================

    sigma_df = pd.read_csv(
        os.path.join(
            DATA_DIR,
            "Sigma.csv",
        )
    )

    sigma_time_column = (
        "t_original"
        if "t_original" in sigma_df.columns
        else "t"
    )

    Sigma = np.zeros(
        (
            T,
            K,
            K,
        ),
        dtype=float,
    )

    available_sigma_times = np.sort(
        sigma_df[
            sigma_time_column
        ]
        .unique()
        .astype(int)
    )

    for t_original in available_sigma_times:

        sigma_t = sigma_df[
            sigma_df[
                sigma_time_column
            ]
            == t_original
        ]

        for k1, future_1 in enumerate(
            future_names
        ):

            for k2, future_2 in enumerate(
                future_names
            ):

                match = sigma_t[
                    (
                        sigma_t[
                            "future_1"
                        ]
                        == future_1
                    )
                    &
                    (
                        sigma_t[
                            "future_2"
                        ]
                        == future_2
                    )
                ]

                if len(match) != 1:

                    raise ValueError(
                        "Sigma.csv must contain exactly "
                        "one covariance for "
                        f"t={t_original}, "
                        f"{future_1}, {future_2}."
                    )

                Sigma[
                    t_original,
                    k1,
                    k2,
                ] = float(
                    match[
                        "covariance"
                    ].iloc[0]
                )


    # ========================================================
    # 3. JOINT STATE MEANS
    # ========================================================

    state_mean_df = pd.read_csv(
        os.path.join(
            DATA_DIR,
            "state_mean_competition.csv",
        )
    )

    state_mean_times = get_original_times(
        state_mean_df
    )

    missing_states = [
        state
        for state in state_names
        if state not in state_mean_df.columns
    ]

    if missing_states:

        raise ValueError(
            "state_mean_competition.csv is missing "
            f"states: {missing_states}"
        )

    state_mean_competition = np.zeros(
        (
            T,
            C_state,
        ),
        dtype=float,
    )

    state_mean_competition[
        state_mean_times,
        :
    ] = (
        state_mean_df[
            state_names
        ]
        .to_numpy(
            dtype=float
        )
    )


    # ========================================================
    # 4. JOINT STATE COVARIANCE
    # ========================================================

    state_cov_df = pd.read_csv(
        os.path.join(
            DATA_DIR,
            "state_cov_competition.csv",
        )
    )

    state_cov_time_column = (
        "t_original"
        if "t_original" in state_cov_df.columns
        else "t"
    )

    state_cov_competition = np.zeros(
        (
            T,
            C_state,
            C_state,
        ),
        dtype=float,
    )

    available_state_cov_times = np.sort(
        state_cov_df[
            state_cov_time_column
        ]
        .unique()
        .astype(int)
    )

    for t_original in available_state_cov_times:

        cov_t = state_cov_df[
            state_cov_df[
                state_cov_time_column
            ]
            == t_original
        ]

        for i, state_1 in enumerate(
            state_names
        ):

            for j, state_2 in enumerate(
                state_names
            ):

                match = cov_t[
                    (
                        cov_t[
                            "state_1"
                        ]
                        == state_1
                    )
                    &
                    (
                        cov_t[
                            "state_2"
                        ]
                        == state_2
                    )
                ]

                if len(match) != 1:

                    raise ValueError(
                        "state_cov_competition.csv must "
                        "contain exactly one covariance for "
                        f"t={t_original}, "
                        f"{state_1}, {state_2}."
                    )

                state_cov_competition[
                    t_original,
                    i,
                    j,
                ] = float(
                    match[
                        "covariance"
                    ].iloc[0]
                )


    # ========================================================
    # 5. JOINT STATE/FUTURES COVARIANCE
    # ========================================================

    state_futures_cov_df = pd.read_csv(
        os.path.join(
            DATA_DIR,
            "state_futures_cov_competition.csv",
        )
    )

    state_futures_time_column = (
        "t_original"
        if "t_original"
        in state_futures_cov_df.columns
        else "t"
    )

    state_futures_cov_competition = np.zeros(
        (
            C_state,
            T,
            K,
        ),
        dtype=float,
    )

    available_times = np.sort(
        state_futures_cov_df[
            state_futures_time_column
        ]
        .unique()
        .astype(int)
    )

    for t_original in available_times:

        cov_t = state_futures_cov_df[
            state_futures_cov_df[
                state_futures_time_column
            ]
            == t_original
        ]

        for i, state_name in enumerate(
            state_names
        ):

            state_t = cov_t[
                cov_t[
                    "state"
                ]
                == state_name
            ]

            for k, future in enumerate(
                future_names
            ):

                match = state_t[
                    state_t[
                        "future"
                    ]
                    == future
                ]

                if len(match) != 1:

                    raise ValueError(
                        "state_futures_cov_competition.csv "
                        "must contain exactly one covariance "
                        f"for t={t_original}, "
                        f"state={state_name}, "
                        f"future={future}."
                    )

                state_futures_cov_competition[
                    i,
                    t_original,
                    k,
                ] = float(
                    match[
                        "covariance"
                    ].iloc[0]
                )


    # ========================================================
    # SHAPE CHECKS
    # ========================================================

    assert mu.shape == (
        T,
        K,
    )

    assert Sigma.shape == (
        T,
        K,
        K,
    )

    assert state_mean_competition.shape == (
        T,
        8,
    )

    assert state_cov_competition.shape == (
        T,
        8,
        8,
    )

    assert state_futures_cov_competition.shape == (
        8,
        T,
        K,
    )


    return (
        mu,
        Sigma,
        state_mean_competition,
        state_cov_competition,
        state_futures_cov_competition,
    )


def make_synthetic_split_problem(
    T,
    K=8,
    C_final=6,
    C_daily=2,
    seed=20260813,
):
    """
    Compatibility wrapper for the optimizer.

    The final and daily states are no longer loaded
    independently. They form one joint 8-D state.
    """

    if C_final != 6:
        raise ValueError(
            "C_final must equal 6."
        )

    if C_daily != 2:
        raise ValueError(
            "C_daily must equal 2."
        )

    return make_synthetic_problem(
        T=T,
        K=K,
        C=C_final + C_daily,
        seed=seed,
    )

def simulate_competition_paths_gaussian(
    mu,
    Sigma,
    state_mean_competition,
    state_cov_competition,
    state_futures_cov_competition,
    current_final_place_balances,
    current_daily_place_pnl,
    day_labels,
    n_sims=10000,
    seed=12345,
):
    """
    Simulate ONE coherent competition world.

    Each simulation contains jointly:

        1. Raw futures returns

        2. Six ordered final-leaderboard states

        3. Two ordered daily-leaderboard states


    JOINT STATE
    -----------

    S =
    [
        Final_X1,
        Final_D12,
        Final_D23,
        Final_D34,
        Final_D4_15,
        Final_D15_16,
        Daily_H1,
        Daily_D12,
    ]


    Final:
        X1 = log Rank1 balance

        D12    = X1 - X2
        D23    = X2 - X3
        D34    = X3 - X4
        D4_15  = X4 - X15
        D15_16 = X15 - X16


    Daily:
        H1 = daily 1st-place P&L

        Daily_D12
        =
        daily 1st-place P&L
        -
        daily 2nd-place P&L


    Both gap systems are constrained to be nonnegative.

    The daily state resets to zero whenever the
    competition day changes.


    Returns
    -------
    R_sims:
        shape (N, T, K)

    final_place_balance_sims:
        shape (N, 6, T)

    B_sims:
        shape (N, 2, T)

        Future INCREMENTS in daily 1st/2nd P&L.

        This preserves compatibility with
        calculate_daily_accounting().
    """

    rng = np.random.default_rng(
        seed
    )

    T, K = mu.shape

    C_state = 8

    day_labels = np.asarray(
        day_labels
    )

    if len(day_labels) != T:
        raise ValueError(
            "day_labels must have length T."
        )


    # ========================================================
    # VALIDATE FORECAST SHAPES
    # ========================================================

    if state_mean_competition.shape != (
        T,
        C_state,
    ):
        raise ValueError(
            "state_mean_competition must have "
            f"shape ({T}, 8)."
        )

    if state_cov_competition.shape != (
        T,
        C_state,
        C_state,
    ):
        raise ValueError(
            "state_cov_competition must have "
            f"shape ({T}, 8, 8)."
        )

    if state_futures_cov_competition.shape != (
        C_state,
        T,
        K,
    ):
        raise ValueError(
            "state_futures_cov_competition must have "
            f"shape (8, {T}, {K})."
        )


    # ========================================================
    # CURRENT FINAL STATE
    # ========================================================

    current_final_place_balances = np.asarray(
        current_final_place_balances,
        dtype=float,
    )

    if current_final_place_balances.shape != (6,):

        raise ValueError(
            "current_final_place_balances "
            "must have shape (6,)."
        )

    if (
        not np.all(
            np.isfinite(
                current_final_place_balances
            )
        )
        or np.any(
            current_final_place_balances <= 0.0
        )
    ):

        raise ValueError(
            "current_final_place_balances must "
            "contain positive finite values."
        )

    if not np.all(
        current_final_place_balances[:-1]
        >= current_final_place_balances[1:]
    ):

        raise ValueError(
            "Current final-place balances are "
            "not correctly ordered."
        )


    current_log_levels = np.log(
        current_final_place_balances
    )


    # ========================================================
    # CURRENT DAILY STATE
    # ========================================================

    current_daily_place_pnl = np.asarray(
        current_daily_place_pnl,
        dtype=float,
    )

    if current_daily_place_pnl.shape != (2,):

        raise ValueError(
            "current_daily_place_pnl "
            "must have shape (2,)."
        )

    if not np.all(
        np.isfinite(
            current_daily_place_pnl
        )
    ):

        raise ValueError(
            "current_daily_place_pnl must "
            "contain finite values."
        )

    if (
        current_daily_place_pnl[0]
        < current_daily_place_pnl[1]
    ):

        raise ValueError(
            "Current daily leaderboard must satisfy "
            "daily 1st P&L >= daily 2nd P&L."
        )


    # ========================================================
    # CONSTRUCT CURRENT 8-D STATE
    # ========================================================

    current_state = np.array(
        [
            # Final X1
            current_log_levels[0],

            # Final D12
            current_log_levels[0]
            - current_log_levels[1],

            # Final D23
            current_log_levels[1]
            - current_log_levels[2],

            # Final D34
            current_log_levels[2]
            - current_log_levels[3],

            # Final D4_15
            current_log_levels[3]
            - current_log_levels[4],

            # Final D15_16
            current_log_levels[4]
            - current_log_levels[5],

            # Daily H1
            current_daily_place_pnl[0],

            # Daily D12
            current_daily_place_pnl[0]
            - current_daily_place_pnl[1],
        ],
        dtype=float,
    )


    state_sims = np.repeat(
        current_state[
            None,
            :
        ],
        repeats=n_sims,
        axis=0,
    )


    # ========================================================
    # OUTPUT ARRAYS
    # ========================================================

    R_sims = np.empty(
        (
            n_sims,
            T,
            K,
        ),
        dtype=float,
    )

    final_place_balance_sims = np.empty(
        (
            n_sims,
            6,
            T,
        ),
        dtype=float,
    )

    B_sims = np.empty(
        (
            n_sims,
            2,
            T,
        ),
        dtype=float,
    )


    # ========================================================
    # WALK FORWARD
    # ========================================================

    for t in range(T):

        # ----------------------------------------------------
        # RESET DAILY STATE WHEN A NEW DAY STARTS
        #
        # At t=0 we preserve the observed current-day state.
        # For every later day transition:
        #
        #   Daily_H1  = 0
        #   Daily_D12 = 0
        # ----------------------------------------------------

        if (
            t > 0
            and day_labels[t]
            != day_labels[t - 1]
        ):

            state_sims[
                :,
                6:
            ] = 0.0


        # ----------------------------------------------------
        # SAVE PRE-INCREMENT DAILY LEVELS
        #
        # Needed because B_sims must remain DAILY P&L
        # INCREMENTS for compatibility with existing
        # daily accounting code.
        # ----------------------------------------------------

        daily_first_before = (
            state_sims[
                :,
                6
            ].copy()
        )

        daily_second_before = (
            state_sims[
                :,
                6
            ]
            - state_sims[
                :,
                7
            ]
        )


        # ----------------------------------------------------
        # ONE FUTURES DRAW FOR THE ENTIRE COMPETITION WORLD
        # ----------------------------------------------------

        R_t_sims = rng.multivariate_normal(
            mean=mu[t],
            cov=Sigma[t],
            size=n_sims,
        )

        R_sims[
            :,
            t,
            :
        ] = R_t_sims

        futures_shocks = (
            R_t_sims
            - mu[t]
        )


        # ----------------------------------------------------
        # JOINT 8-D CONDITIONAL STATE DISTRIBUTION
        # ----------------------------------------------------

        C_t = (
            state_futures_cov_competition[
                :,
                t,
                :
            ]
        )

        beta_state = np.linalg.solve(
            Sigma[t],
            C_t.T,
        )


        explained_state_cov = (
            C_t
            @ beta_state
        )


        residual_cov = (
            state_cov_competition[t]
            - explained_state_cov
        )

        residual_cov = (
            0.5
            * (
                residual_cov
                + residual_cov.T
            )
        )


        # ----------------------------------------------------
        # NUMERICAL PSD REPAIR
        # ----------------------------------------------------

        eigenvalues, eigenvectors = np.linalg.eigh(
            residual_cov
        )

        eigenvalues = np.maximum(
            eigenvalues,
            0.0,
        )

        residual_cov = (
            eigenvectors
            @ np.diag(
                eigenvalues
            )
            @ eigenvectors.T
        )


        # ----------------------------------------------------
        # ONE JOINT EIGHT-DIMENSIONAL RESIDUAL DRAW
        #
        # This is where:
        #
        # final <-> final
        # daily <-> daily
        # final <-> daily
        #
        # residual dependence is preserved.
        # ----------------------------------------------------

        epsilon = rng.multivariate_normal(
            mean=np.zeros(
                C_state
            ),
            cov=residual_cov,
            size=n_sims,
        )


        delta_state = (
            state_mean_competition[
                t
            ][
                None,
                :
            ]
            + futures_shocks
            @ beta_state
            + epsilon
        )


        # ====================================================
        # UPDATE FINAL STATE
        # ====================================================

        state_sims[
            :,
            0
        ] += (
            delta_state[
                :,
                0
            ]
        )


        # Five final gaps
        state_sims[
            :,
            1:6
        ] = np.maximum(
            0.0,
            state_sims[
                :,
                1:6
            ]
            + delta_state[
                :,
                1:6
            ],
        )


        # ====================================================
        # UPDATE DAILY STATE
        # ====================================================

        state_sims[
            :,
            6
        ] += (
            delta_state[
                :,
                6
            ]
        )


        # Daily 1st-minus-2nd gap
        state_sims[
            :,
            7
        ] = np.maximum(
            0.0,
            state_sims[
                :,
                7
            ]
            + delta_state[
                :,
                7
            ],
        )


        # ====================================================
        # RECONSTRUCT FINAL BALANCES
        # ====================================================

        X1 = state_sims[:, 0]

        D12 = state_sims[:, 1]
        D23 = state_sims[:, 2]
        D34 = state_sims[:, 3]
        D4_15 = state_sims[:, 4]
        D15_16 = state_sims[:, 5]


        X2 = (
            X1
            - D12
        )

        X3 = (
            X2
            - D23
        )

        X4 = (
            X3
            - D34
        )

        X15 = (
            X4
            - D4_15
        )

        X16 = (
            X15
            - D15_16
        )


        log_levels_t = np.column_stack(
            (
                X1,
                X2,
                X3,
                X4,
                X15,
                X16,
            )
        )

        balances_t = np.exp(
            log_levels_t
        )

        if not np.all(
            np.isfinite(
                balances_t
            )
        ):

            raise ValueError(
                "Final-place simulation produced "
                "non-finite balances."
            )

        final_place_balance_sims[
            :,
            :,
            t
        ] = balances_t


        # ====================================================
        # RECONSTRUCT DAILY 1ST / 2ND LEVELS
        # ====================================================

        daily_first_after = (
            state_sims[
                :,
                6
            ]
        )

        daily_second_after = (
            state_sims[
                :,
                6
            ]
            - state_sims[
                :,
                7
            ]
        )


        if not np.all(
            daily_first_after
            >= daily_second_after
        ):

            raise AssertionError(
                "Simulated daily 1st P&L "
                "< daily 2nd P&L."
            )


        # Return INCREMENTS to existing daily-accounting code.
        B_sims[
            :,
            0,
            t
        ] = (
            daily_first_after
            - daily_first_before
        )

        B_sims[
            :,
            1,
            t
        ] = (
            daily_second_after
            - daily_second_before
        )


    # ========================================================
    # NON-NEGOTIABLE FINAL ORDER VALIDATION
    # ========================================================

    if not np.all(
        final_place_balance_sims[
            :,
            0,
            :
        ]
        >= final_place_balance_sims[
            :,
            1,
            :
        ]
    ):

        raise AssertionError(
            "Simulated Rank1 < Rank2."
        )


    if not np.all(
        final_place_balance_sims[
            :,
            1,
            :
        ]
        >= final_place_balance_sims[
            :,
            2,
            :
        ]
    ):

        raise AssertionError(
            "Simulated Rank2 < Rank3."
        )


    if not np.all(
        final_place_balance_sims[
            :,
            2,
            :
        ]
        >= final_place_balance_sims[
            :,
            3,
            :
        ]
    ):

        raise AssertionError(
            "Simulated Rank3 < Rank4."
        )


    if not np.all(
        final_place_balance_sims[
            :,
            3,
            :
        ]
        >= final_place_balance_sims[
            :,
            4,
            :
        ]
    ):

        raise AssertionError(
            "Simulated Rank4 < Rank15."
        )


    if not np.all(
        final_place_balance_sims[
            :,
            4,
            :
        ]
        >= final_place_balance_sims[
            :,
            5,
            :
        ]
    ):

        raise AssertionError(
            "Simulated Rank15 < Rank16."
        )


    return (
        R_sims,
        final_place_balance_sims,
        B_sims,
    )

# ============================================================
# STEP 6.4 - DISTRIBUTIONAL COMPETITION SIMULATOR
#
# The legacy Gaussian simulator remains available as:
#
#     simulate_competition_paths_gaussian(...)
#
# The new production simulator uses:
#
#     Step 6.2:
#         frozen 16D joint innovation coupling
#
#     Step 6.3:
#         frozen futures arithmetic-return mapping
#
# NEW DEPENDENCE ARCHITECTURE
# ---------------------------
#
# Futures dependence:
#     frozen 50.15 coupling
#
# Final-state dependence:
#     frozen 50.15 coupling
#
# Daily-state dependence:
#     frozen 50.15 coupling
#
# Cross-system dependence:
#     frozen 50.15 coupling
#
# Therefore the NEW simulator does NOT use:
#
#     Sigma off-diagonals
#     state_cov_competition off-diagonals
#     state_futures_cov_competition values
#     beta_state
#     conditional Gaussian residual covariance
#
# The following marginal forecast information remains
# authoritative:
#
#     mu[t, k]
#     Sigma[t, k, k]
#     state_mean_competition[t, c]
#     state_cov_competition[t, c, c]
#
# The output contract is unchanged:
#
#     R_sims                   (N, T, K)
#     final_place_balance_sims (N, 6, T)
#     B_sims                   (N, 2, T)
# ============================================================


def simulate_competition_paths_distributional(
    mu,
    Sigma,
    state_mean_competition,
    state_cov_competition,
    state_futures_cov_competition,
    current_final_place_balances,
    current_daily_place_pnl,
    day_labels,
    t_original_values,
    n_sims=10000,
    seed=12345,
    innovation_system=None,
    mapping_system=None,
):
    """
    Simulate one coherent competition world using the frozen
    distributional innovation architecture.

    Returns
    -------
    R_sims:
        shape (N, T, K)

    final_place_balance_sims:
        shape (N, 6, T)

    B_sims:
        shape (N, 2, T)

        Future increments in daily 1st/2nd P&L.
    """

    rng = np.random.default_rng(
        seed
    )

    mu = np.asarray(
        mu,
        dtype=float,
    )

    Sigma = np.asarray(
        Sigma,
        dtype=float,
    )

    state_mean_competition = np.asarray(
        state_mean_competition,
        dtype=float,
    )

    state_cov_competition = np.asarray(
        state_cov_competition,
        dtype=float,
    )

    state_futures_cov_competition = np.asarray(
        state_futures_cov_competition,
        dtype=float,
    )

    T, K = mu.shape

    C_state = 8

    if K != 8:
        raise ValueError(
            "The frozen distributional production system "
            "requires exactly 8 futures."
        )

    day_labels = np.asarray(
        day_labels
    )

    t_original_values = np.asarray(
        t_original_values,
        dtype=int,
    )


    # ========================================================
    # VALIDATE FORECAST SHAPES
    # ========================================================

    if Sigma.shape != (
        T,
        K,
        K,
    ):
        raise ValueError(
            "Sigma must have shape "
            f"({T}, {K}, {K})."
        )

    if state_mean_competition.shape != (
        T,
        C_state,
    ):
        raise ValueError(
            "state_mean_competition must have "
            f"shape ({T}, 8)."
        )

    if state_cov_competition.shape != (
        T,
        C_state,
        C_state,
    ):
        raise ValueError(
            "state_cov_competition must have "
            f"shape ({T}, 8, 8)."
        )

    if state_futures_cov_competition.shape != (
        C_state,
        T,
        K,
    ):
        raise ValueError(
            "state_futures_cov_competition must have "
            f"shape (8, {T}, {K})."
        )

    if len(
        day_labels
    ) != T:
        raise ValueError(
            "day_labels must have length T."
        )

    if t_original_values.shape != (
        T,
    ):
        raise ValueError(
            "t_original_values must have shape "
            f"({T},)."
        )

    if len(
        np.unique(
            t_original_values
        )
    ) != T:
        raise ValueError(
            "t_original_values must not contain duplicates."
        )


    # ========================================================
    # VALIDATE FINITE FORECAST INPUTS
    # ========================================================

    for array_name, array_value in [
        ("mu", mu),
        ("Sigma", Sigma),
        (
            "state_mean_competition",
            state_mean_competition,
        ),
        (
            "state_cov_competition",
            state_cov_competition,
        ),
        (
            "state_futures_cov_competition",
            state_futures_cov_competition,
        ),
    ]:

        if not np.all(
            np.isfinite(
                array_value
            )
        ):
            raise ValueError(
                f"{array_name} contains non-finite values."
            )


    # ========================================================
    # EXTRACT ONLY AUTHORITATIVE MARGINAL VARIANCES
    #
    # IMPORTANT:
    #
    # The full covariance matrices are NOT used to generate
    # dependence in the distributional simulator.
    # ========================================================

    futures_variance = np.diagonal(
        Sigma,
        axis1=1,
        axis2=2,
    ).copy()

    state_variance = np.diagonal(
        state_cov_competition,
        axis1=1,
        axis2=2,
    ).copy()


    if futures_variance.shape != (
        T,
        K,
    ):
        raise AssertionError(
            "Futures variance extraction failed."
        )

    if state_variance.shape != (
        T,
        C_state,
    ):
        raise AssertionError(
            "State variance extraction failed."
        )


    if np.any(
        futures_variance <= 0.0
    ):
        raise ValueError(
            "All futures marginal variances must "
            "be strictly positive."
        )


    # State variance may legitimately be zero during
    # inactive daily-state intervals.
    state_variance_tolerance = 1e-14

    if np.any(
        state_variance
        < -state_variance_tolerance
    ):
        raise ValueError(
            "State marginal variances must be nonnegative."
        )

    state_variance = np.maximum(
        state_variance,
        0.0,
    )

    state_sd = np.sqrt(
        state_variance
    )


    # ========================================================
    # LOAD FROZEN DISTRIBUTIONAL SYSTEMS ONCE
    # ========================================================

    if innovation_system is None:

        innovation_system = (
            load_python_innovation_system()
        )

    if mapping_system is None:

        mapping_system = (
            load_python_arithmetic_mapping()
        )


    if (
        innovation_system[
            "component_order"
        ][
            :8
        ]
        != FUTURES_PRODUCTION_ORDER
    ):
        raise ValueError(
            "Innovation futures order does not match "
            "the arithmetic mapping order."
        )


    innovation_time_lookup = (
        innovation_system[
            "time_to_index"
        ]
    )

    missing_times = [
        int(t_original)
        for t_original
        in t_original_values
        if int(t_original)
        not in innovation_time_lookup
    ]

    if missing_times:
        raise ValueError(
            "Distributional innovation system does not "
            "contain required t_original values: "
            f"{missing_times}"
        )


    # ========================================================
    # CURRENT FINAL STATE
    # ========================================================

    current_final_place_balances = np.asarray(
        current_final_place_balances,
        dtype=float,
    )

    if current_final_place_balances.shape != (
        6,
    ):
        raise ValueError(
            "current_final_place_balances "
            "must have shape (6,)."
        )

    if (
        not np.all(
            np.isfinite(
                current_final_place_balances
            )
        )
        or np.any(
            current_final_place_balances <= 0.0
        )
    ):
        raise ValueError(
            "current_final_place_balances must "
            "contain positive finite values."
        )

    if not np.all(
        current_final_place_balances[
            :-1
        ]
        >= current_final_place_balances[
            1:
        ]
    ):
        raise ValueError(
            "Current final-place balances are "
            "not correctly ordered."
        )


    current_log_levels = np.log(
        current_final_place_balances
    )


    # ========================================================
    # CURRENT DAILY STATE
    # ========================================================

    current_daily_place_pnl = np.asarray(
        current_daily_place_pnl,
        dtype=float,
    )

    if current_daily_place_pnl.shape != (
        2,
    ):
        raise ValueError(
            "current_daily_place_pnl "
            "must have shape (2,)."
        )

    if not np.all(
        np.isfinite(
            current_daily_place_pnl
        )
    ):
        raise ValueError(
            "current_daily_place_pnl must "
            "contain finite values."
        )

    if (
        current_daily_place_pnl[0]
        < current_daily_place_pnl[1]
    ):
        raise ValueError(
            "Current daily leaderboard must satisfy "
            "daily 1st P&L >= daily 2nd P&L."
        )


    # ========================================================
    # CONSTRUCT CURRENT 8-D STATE
    # ========================================================

    current_state = np.array(
        [
            current_log_levels[0],

            current_log_levels[0]
            - current_log_levels[1],

            current_log_levels[1]
            - current_log_levels[2],

            current_log_levels[2]
            - current_log_levels[3],

            current_log_levels[3]
            - current_log_levels[4],

            current_log_levels[4]
            - current_log_levels[5],

            current_daily_place_pnl[0],

            current_daily_place_pnl[0]
            - current_daily_place_pnl[1],
        ],
        dtype=float,
    )


    state_sims = np.repeat(
        current_state[
            None,
            :
        ],
        repeats=n_sims,
        axis=0,
    )


    # ========================================================
    # OUTPUT ARRAYS
    # ========================================================

    R_sims = np.empty(
        (
            n_sims,
            T,
            K,
        ),
        dtype=float,
    )

    final_place_balance_sims = np.empty(
        (
            n_sims,
            6,
            T,
        ),
        dtype=float,
    )

    B_sims = np.empty(
        (
            n_sims,
            2,
            T,
        ),
        dtype=float,
    )


    # ========================================================
    # WALK FORWARD
    # ========================================================

    for t in range(
        T
    ):

        # ----------------------------------------------------
        # RESET DAILY STATE WHEN A NEW DAY STARTS
        # ----------------------------------------------------

        if (
            t > 0
            and day_labels[t]
            != day_labels[
                t - 1
            ]
        ):
            state_sims[
                :,
                6:
            ] = 0.0


        # ----------------------------------------------------
        # SAVE PRE-INCREMENT DAILY LEVELS
        # ----------------------------------------------------

        daily_first_before = (
            state_sims[
                :,
                6
            ].copy()
        )

        daily_second_before = (
            state_sims[
                :,
                6
            ]
            - state_sims[
                :,
                7
            ]
        )


        # ----------------------------------------------------
        # DRAW ONE FROZEN 16-D JOINT INNOVATION WORLD
        # ----------------------------------------------------

        joint_innovation_details = build_joint_innovations(
            t_original=int(
                t_original_values[
                    t
                ]
            ),
            n_sims=n_sims,
            rng=rng,
            innovation_system=innovation_system,
            return_details=True,
        )

        Z_joint = joint_innovation_details[
            "Z_sims"
        ]


        if Z_joint.shape != (
            n_sims,
            16,
        ):
            raise AssertionError(
                "Joint innovation draw has "
                "incorrect shape."
            )


        # ----------------------------------------------------
        # FUTURES RETURNS
        #
        # Dependence is ALREADY inside Z_joint.
        #
        # Only the marginal mean and diagonal variance from
        # mu / Sigma are applied here.
        # ----------------------------------------------------

        Z_futures = Z_joint[
            :,
            :8
        ]

        R_t_sims = (
            map_futures_innovations_to_arithmetic_returns(
                Z_futures=Z_futures,
                mu_t=mu[
                    t
                ],
                variance_t=futures_variance[
                    t
                ],
                mapping_system=mapping_system,
            )
        )

        R_sims[
            :,
            t,
            :
        ] = R_t_sims


        # ----------------------------------------------------
        # COMPETITOR STATE INCREMENTS
        #
        # The selected state innovations are already jointly
        # coupled through 50.15.
        #
        # Therefore:
        #
        #     delta_state
        #         =
        #     state_mean
        #     + state_sd * selected_state_innovation
        #
        # No beta_state or conditional Gaussian residual draw.
        # ----------------------------------------------------

        Z_state = Z_joint[
            :,
            8:16
        ]

        delta_state = (
            state_mean_competition[
                t
            ][
                None,
                :
            ]
            + state_sd[
                t
            ][
                None,
                :
            ]
            * Z_state
        )


        if delta_state.shape != (
            n_sims,
            8,
        ):
            raise AssertionError(
                "State increment matrix has "
                "incorrect shape."
            )

        if not np.all(
            np.isfinite(
                delta_state
            )
        ):
            raise ValueError(
                "State increments contain "
                "non-finite values."
            )


        # ====================================================
        # UPDATE FINAL STATE
        # ====================================================

        state_sims[
            :,
            0
        ] += delta_state[
            :,
            0
        ]


        state_sims[
            :,
            1:6
        ] = np.maximum(
            0.0,
            state_sims[
                :,
                1:6
            ]
            + delta_state[
                :,
                1:6
            ],
        )


        # ====================================================
        # UPDATE DAILY STATE
        # ====================================================

        state_sims[
            :,
            6
        ] += delta_state[
            :,
            6
        ]


        state_sims[
            :,
            7
        ] = np.maximum(
            0.0,
            state_sims[
                :,
                7
            ]
            + delta_state[
                :,
                7
            ],
        )


        # ====================================================
        # RECONSTRUCT FINAL BALANCES
        # ====================================================

        X1 = state_sims[
            :,
            0
        ]

        D12 = state_sims[
            :,
            1
        ]

        D23 = state_sims[
            :,
            2
        ]

        D34 = state_sims[
            :,
            3
        ]

        D4_15 = state_sims[
            :,
            4
        ]

        D15_16 = state_sims[
            :,
            5
        ]


        X2 = (
            X1
            - D12
        )

        X3 = (
            X2
            - D23
        )

        X4 = (
            X3
            - D34
        )

        X15 = (
            X4
            - D4_15
        )

        X16 = (
            X15
            - D15_16
        )


        log_levels_t = np.column_stack(
            (
                X1,
                X2,
                X3,
                X4,
                X15,
                X16,
            )
        )


        balances_t = np.exp(
            log_levels_t
        )


        if not np.all(
            np.isfinite(
                balances_t
            )
        ):
            raise ValueError(
                "Final-place simulation produced "
                "non-finite balances."
            )


        final_place_balance_sims[
            :,
            :,
            t
        ] = balances_t


        # ====================================================
        # RECONSTRUCT DAILY 1ST / 2ND LEVELS
        # ====================================================

        daily_first_after = (
            state_sims[
                :,
                6
            ]
        )

        daily_second_after = (
            state_sims[
                :,
                6
            ]
            - state_sims[
                :,
                7
            ]
        )


        if not np.all(
            daily_first_after
            >= daily_second_after
        ):
            raise AssertionError(
                "Simulated daily 1st P&L "
                "< daily 2nd P&L."
            )


        B_sims[
            :,
            0,
            t
        ] = (
            daily_first_after
            - daily_first_before
        )

        B_sims[
            :,
            1,
            t
        ] = (
            daily_second_after
            - daily_second_before
        )


    # ========================================================
    # NON-NEGOTIABLE FINAL ORDER VALIDATION
    # ========================================================

    if not np.all(
        final_place_balance_sims[
            :,
            0,
            :
        ]
        >= final_place_balance_sims[
            :,
            1,
            :
        ]
    ):
        raise AssertionError(
            "Simulated Rank1 < Rank2."
        )


    if not np.all(
        final_place_balance_sims[
            :,
            1,
            :
        ]
        >= final_place_balance_sims[
            :,
            2,
            :
        ]
    ):
        raise AssertionError(
            "Simulated Rank2 < Rank3."
        )


    if not np.all(
        final_place_balance_sims[
            :,
            2,
            :
        ]
        >= final_place_balance_sims[
            :,
            3,
            :
        ]
    ):
        raise AssertionError(
            "Simulated Rank3 < Rank4."
        )


    if not np.all(
        final_place_balance_sims[
            :,
            3,
            :
        ]
        >= final_place_balance_sims[
            :,
            4,
            :
        ]
    ):
        raise AssertionError(
            "Simulated Rank4 < Rank15."
        )


    if not np.all(
        final_place_balance_sims[
            :,
            4,
            :
        ]
        >= final_place_balance_sims[
            :,
            5,
            :
        ]
    ):
        raise AssertionError(
            "Simulated Rank15 < Rank16."
        )


    if not np.all(
        np.isfinite(
            R_sims
        )
    ):
        raise ValueError(
            "Distributional futures simulation "
            "contains non-finite returns."
        )


    return (
        R_sims,
        final_place_balance_sims,
        B_sims,
    )


def simulate_competition_paths(
    mu,
    Sigma,
    state_mean_competition,
    state_cov_competition,
    state_futures_cov_competition,
    current_final_place_balances,
    current_daily_place_pnl,
    day_labels,
    t_original_values=None,
    n_sims=10000,
    seed=12345,
    simulation_model="distributional",
    innovation_system=None,
    mapping_system=None,
):
    """
    Dispatch between the frozen legacy Gaussian simulator and
    the new distributional simulator.

    simulation_model:
        "gaussian"
            Legacy Gaussian benchmark.

        "distributional"
            Frozen 50.0 / 50.15 / 50.25 production
            distributional architecture.
    """

    simulation_model = str(
        simulation_model
    ).strip().lower()


    if simulation_model == "gaussian":

        return simulate_competition_paths_gaussian(
            mu=mu,
            Sigma=Sigma,
            state_mean_competition=
                state_mean_competition,
            state_cov_competition=
                state_cov_competition,
            state_futures_cov_competition=
                state_futures_cov_competition,
            current_final_place_balances=
                current_final_place_balances,
            current_daily_place_pnl=
                current_daily_place_pnl,
            day_labels=day_labels,
            n_sims=n_sims,
            seed=seed,
        )


    if simulation_model == "distributional":

        if t_original_values is None:
            raise ValueError(
                "t_original_values must be supplied for "
                "the distributional simulator."
            )

        return simulate_competition_paths_distributional(
            mu=mu,
            Sigma=Sigma,
            state_mean_competition=
                state_mean_competition,
            state_cov_competition=
                state_cov_competition,
            state_futures_cov_competition=
                state_futures_cov_competition,
            current_final_place_balances=
                current_final_place_balances,
            current_daily_place_pnl=
                current_daily_place_pnl,
            day_labels=day_labels,
            t_original_values=
                t_original_values,
            n_sims=n_sims,
            seed=seed,
            innovation_system=
                innovation_system,
            mapping_system=
                mapping_system,
        )


    raise ValueError(
        "simulation_model must be either "
        "'gaussian' or 'distributional'."
    )

def calculate_daily_accounting(
    R_sims,
    B_sims,
    W,
    day_labels,
    starting_balance=25000.0,
):
    """
    Convert future simulations into daily P&L.

    IMPORTANT UNIT DISTINCTION
    --------------------------

    R_sims:
        Futures RETURNS.

    B_sims:
        Changes in the daily leaderboard-place P&L increments.

        B_sims[s,i,t]
        =
        Delta_H_i,t

        Units: dollars.

    Therefore:

        Our daily P&L:
            comes from compounding our returns.

        Threshold daily P&L increment:
            comes from SUMMING B_sims.

    Parameters
    ----------
    R_sims:
        Shape (N, T, K)

        Simulated futures returns.

    B_sims:
        Shape (N, C, T)

        Simulated dollar changes in the
        daily-prize P&L thresholds.

    W:
        Shape (T, K)

        Our portfolio weights.

    day_labels:
        Length T.

    starting_balance:
        Our actual current account balance.
    """

    N, T, K = R_sims.shape

    N_B, C, T_B = (
        B_sims.shape
    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    assert N_B == N
    assert T_B == T

    assert W.shape == (
        T,
        K
    )

    day_labels = np.asarray(
        day_labels
    )

    assert len(
        day_labels
    ) == T


    # --------------------------------------------------------
    # Preserve day ordering
    # --------------------------------------------------------

    days = list(
        dict.fromkeys(
            day_labels.tolist()
        )
    )

    D = len(days)


    # --------------------------------------------------------
    # Our interval RETURNS
    #
    # r_A,s,t = w_t' R_s,t
    # --------------------------------------------------------

    our_interval_returns = np.einsum(
        "ntk,tk->nt",
        R_sims,
        W,
    )


    # --------------------------------------------------------
    # Daily arrays
    # --------------------------------------------------------

    our_daily_returns = np.empty(
        (N, D)
    )

    # Despite the compatibility name below,
    # this contains DOLLAR P&L increments.
    daily_place_pnl = np.empty(
        (N, C, D)
    )


    # --------------------------------------------------------
    # Aggregate within each day
    # --------------------------------------------------------

    for d, day in enumerate(days):

        mask = (
            day_labels == day
        )


        # ----------------------------------------------------
        # OUR side:
        #
        # compound RETURNS
        # ----------------------------------------------------

        our_daily_returns[:, d] = (
            np.prod(
                1.0
                + our_interval_returns[
                    :,
                    mask
                ],
                axis=1,
            )
            - 1.0
        )


        # ----------------------------------------------------
        # DAILY THRESHOLD side:
        #
        # B_sims already contains dollar P&L changes.
        #
        # Therefore SUM them.
        # ----------------------------------------------------

        daily_place_pnl[
            :,
            :,
            d
        ] = np.sum(
            B_sims[
                :,
                :,
                mask
            ],
            axis=2,
        )


    # --------------------------------------------------------
    # Our balance / P&L arrays
    # --------------------------------------------------------

    our_beginning_balances = np.empty(
        (N, D)
    )

    our_daily_pnl = np.empty(
        (N, D)
    )

    our_ending_balances = np.empty(
        (N, D)
    )


    our_current_balance = np.full(
        N,
        starting_balance,
        dtype=float,
    )


    # --------------------------------------------------------
    # Walk our account forward
    # --------------------------------------------------------

    for d in range(D):

        our_beginning_balances[:, d] = (
            our_current_balance
        )

        our_daily_pnl[:, d] = (
            our_current_balance
            * our_daily_returns[:, d]
        )

        our_current_balance = (
            our_current_balance
            + our_daily_pnl[:, d]
        )

        our_ending_balances[:, d] = (
            our_current_balance
        )


    return {
        "days":
            days,

        "our_interval_returns":
            our_interval_returns,

        "our_daily_returns":
            our_daily_returns,

        "our_beginning_balances":
            our_beginning_balances,

        "our_daily_pnl":
            our_daily_pnl,

        "our_ending_balances":
            our_ending_balances,

        # Compatibility key:
        #
        # This now means FUTURE CHANGE in
        # the daily P&L hurdle, in dollars.
        "daily_place_pnl":
            daily_place_pnl,
    }


def calculate_hard_daily_prize(
    accounting,
    daily_prize=500.0,
    eligible_days=None,
    daily_threshold_days=None,

    # STEP 5
    current_day=None,
    our_realized_current_day_pnl=0.0,
    current_daily_rank=None,
    current_daily_place_pnl=None,
):
    """Hard Monte Carlo value for one day-specific winning hurdle per prize day."""
    our_daily_pnl = accounting["our_daily_pnl"]
    daily_place_pnl = accounting["daily_place_pnl"]

    if daily_threshold_days is None:
        if eligible_days is None:
            threshold_days = list(accounting["days"])
        else:
            threshold_days = [
                day
                for day in accounting["days"]
                if day in eligible_days
            ]
    else:
        threshold_days = [
            day
            for day in daily_threshold_days
            if day in accounting["days"]
            and (
                eligible_days is None
                or day in eligible_days
            )
        ]

    day_to_index = {
        day: i
        for i, day in enumerate(accounting["days"])
    }

    day_indices = np.array(
        [
            day_to_index[day]
            for day in threshold_days
        ],
        dtype=int,
    )

    # Row j is the winning hurdle for threshold_days[j].
    our_selected_pnl = (
        our_daily_pnl[:, day_indices].copy()
    )

    # Add already-realized P&L only to the currently active
    # daily-prize day.
    if current_day in threshold_days:

        j = threshold_days.index(
            current_day
        )

        our_selected_pnl[:, j] += (
            our_realized_current_day_pnl
        )
    # Two daily leaderboard-place processes:
    #
    # index 0 = daily 1st place
    # index 1 = daily 2nd place

    daily_first_pnl = (
        daily_place_pnl[
            :, 0, day_indices
        ].copy()
    )

    daily_second_pnl = (
        daily_place_pnl[
            :, 1, day_indices
        ].copy()
    )

    # Add already-realized 1st/2nd place P&L
    # only to the currently active day.
    if current_day in threshold_days:

        if current_daily_place_pnl is None:
            raise ValueError(
                "current_daily_place_pnl must be supplied "
                "for the active current day."
            )

        current_daily_place_pnl = np.asarray(
            current_daily_place_pnl,
            dtype=float,
        )

        if current_daily_place_pnl.shape != (2,):
            raise ValueError(
                "current_daily_place_pnl must have shape (2,)."
            )

        j = threshold_days.index(
            current_day
        )

        daily_first_pnl[:, j] += (
            current_daily_place_pnl[0]
        )

        daily_second_pnl[:, j] += (
            current_daily_place_pnl[1]
        )

    # Hard state:
    #
    # If we are not leading, 1st place is the hurdle.
    # If we are leading, 2nd place is the hurdle.
    # Default behavior for future prize days, where the
    # current daily rank is not yet observed.
    we_are_leading = (
        our_selected_pnl
        > daily_first_pnl
    )

    selected_threshold_pnl = np.where(
        we_are_leading,
        daily_second_pnl,
        daily_first_pnl,
    )

    # For the currently active day, use our ACTUAL observed
    # daily rank so that our own leaderboard slot is never
    # treated as a competitor.
    if (
        current_day in threshold_days
        and current_daily_rank is not None
    ):

        j = threshold_days.index(
            current_day
        )

        if current_daily_rank == 1:

            selected_threshold_pnl[
                :,
                j
            ] = (
                daily_second_pnl[
                    :,
                    j
                ]
            )

        else:

            selected_threshold_pnl[
                :,
                j
            ] = (
                daily_first_pnl[
                    :,
                    j
                ]
            )

    daily_win_indicators = (
        our_selected_pnl > selected_threshold_pnl
    ).astype(int)

    any_daily_win = np.max(daily_win_indicators, axis=1)
    daily_win_probabilities = np.mean(daily_win_indicators, axis=0)
    probability_win_at_least_once = np.mean(any_daily_win)
    expected_daily_prize = daily_prize * probability_win_at_least_once

    return {
        "days": threshold_days,
        "daily_threshold_pnl": selected_threshold_pnl,
        "daily_win_indicators": daily_win_indicators,
        "any_daily_win": any_daily_win,
        "daily_win_probabilities": daily_win_probabilities,
        "probability_win_at_least_once": probability_win_at_least_once,
        "expected_daily_prize": expected_daily_prize,
    }

def calculate_smooth_daily_prize(
    accounting,
    daily_prize=500.0,
    competitor_temperature=1.0,
    win_temperature=1.0,
    eligible_days=None,
    daily_threshold_days=None,
    current_day=None,
    our_realized_current_day_pnl=0.0,
    current_daily_rank=None,
    current_daily_place_pnl=None,
):
    """Smooth value for one day-specific winning hurdle per prize day."""
    our_daily_pnl = accounting["our_daily_pnl"]
    daily_place_pnl = accounting["daily_place_pnl"]

    if daily_threshold_days is None:
        if eligible_days is None:
            threshold_days = list(
                accounting["days"]
            )
        else:
            threshold_days = [
                day
                for day in accounting["days"]
                if day in eligible_days
            ]
    else:
        threshold_days = [
            day
            for day in daily_threshold_days
            if day in accounting["days"]
            and (
                eligible_days is None
                or day in eligible_days
            )
        ]

    day_to_index = {
        day: i
        for i, day in enumerate(accounting["days"])
    }

    day_indices = np.array(
        [
            day_to_index[day]
            for day in threshold_days
        ],
        dtype=int,
    )

    our_selected_pnl = (
        our_daily_pnl[:, day_indices].copy()
    )

    if current_day in threshold_days:

        j = threshold_days.index(
            current_day
        )

        our_selected_pnl[:, j] += (
            our_realized_current_day_pnl
        )
    # Two daily leaderboard-place processes:
    #
    # index 0 = daily 1st place
    # index 1 = daily 2nd place

    daily_first_pnl = (
        daily_place_pnl[
            :, 0, day_indices
        ].copy()
    )

    daily_second_pnl = (
        daily_place_pnl[
            :, 1, day_indices
        ].copy()
    )

    # Add already-realized 1st/2nd place P&L
    # only to the currently active day.
    if current_day in threshold_days:

        if current_daily_place_pnl is None:
            raise ValueError(
                "current_daily_place_pnl must be supplied "
                "for the active current day."
            )

        current_daily_place_pnl = np.asarray(
            current_daily_place_pnl,
            dtype=float,
        )

        if current_daily_place_pnl.shape != (2,):
            raise ValueError(
                "current_daily_place_pnl must have shape (2,)."
            )

        j = threshold_days.index(
            current_day
        )

        daily_first_pnl[:, j] += (
            current_daily_place_pnl[0]
        )

        daily_second_pnl[:, j] += (
            current_daily_place_pnl[1]
        )

    # Smooth state:
    #
    # If we are below 1st, the hurdle approaches 1st place.
    # If we are above 1st, the hurdle approaches 2nd place.

    rank_gate = expit(
        (our_selected_pnl - daily_first_pnl)
        / competitor_temperature
    )

    selected_threshold_pnl = (
        daily_first_pnl
        + rank_gate
        * (
            daily_second_pnl
            - daily_first_pnl
        )
    )

    margins = our_selected_pnl - selected_threshold_pnl

    # There is now exactly ONE smooth state-dependent hurdle per day,
    # so no smooth-min across separate hurdle processes is needed.
    tau_w = win_temperature
    smooth_daily_win = expit(margins / tau_w)

    smooth_daily_probabilities = np.mean(smooth_daily_win, axis=0)
    smooth_any_win = 1.0 - np.prod(1.0 - smooth_daily_win, axis=1)
    probability_win_at_least_once = np.mean(smooth_any_win)
    expected_daily_prize = daily_prize * probability_win_at_least_once

    return {
        "days": threshold_days,
        "margins": margins,
        "smooth_min_margin": margins,  # compatibility key
        "smooth_daily_win": smooth_daily_win,
        "smooth_daily_probabilities": smooth_daily_probabilities,
        "smooth_any_win": smooth_any_win,
        "probability_win_at_least_once": probability_win_at_least_once,
        "expected_daily_prize": expected_daily_prize,
    }

def smooth_daily_prize_and_gradient(
    W,
    R_sims,
    B_sims,
    day_labels,
    starting_balance=25000.0,
    daily_prize=500.0,
    competitor_temperature=1.0,
    win_temperature=1.0,
    eligible_days=None,
    daily_threshold_days=None,
    current_day=None,
    our_realized_current_day_pnl=0.0,
    current_daily_rank=None,
    current_daily_place_pnl=None,
):
    """
    Smooth daily-prize value and analytic gradient.

    Daily leaderboard processes:
        index 0 = daily 1st-place P&L
        index 1 = daily 2nd-place P&L

    The smooth hurdle transitions from 1st-place P&L
    toward 2nd-place P&L as our P&L moves above 1st.
    """

    accounting = calculate_daily_accounting(
        R_sims=R_sims,
        B_sims=B_sims,
        W=W,
        day_labels=day_labels,
        starting_balance=starting_balance,
    )

    our_daily_pnl = accounting[
        "our_daily_pnl"
    ]

    daily_place_pnl = accounting[
        "daily_place_pnl"
    ]

    our_ending_balances = accounting[
        "our_ending_balances"
    ]

    our_interval_returns = accounting[
        "our_interval_returns"
    ]

    days = accounting["days"]

    N, D = our_daily_pnl.shape
    T, K = W.shape

    # --------------------------------------------------------
    # Prize days that are still relevant
    # --------------------------------------------------------

    if daily_threshold_days is None:
        if eligible_days is None:
            threshold_days = list(
                days
            )
        else:
            threshold_days = [
                day
                for day in days
                if day in eligible_days
            ]
    else:
        threshold_days = [
            day
            for day in daily_threshold_days
            if day in days
            and (
                eligible_days is None
                or day in eligible_days
            )
        ]

    day_to_index = {
        day: i
        for i, day in enumerate(days)
    }

    day_indices = np.array(
        [
            day_to_index[day]
            for day in threshold_days
        ],
        dtype=int,
    )

    # --------------------------------------------------------
    # Our P&L on selected prize days
    # --------------------------------------------------------

    our_selected_pnl = (
        our_daily_pnl[
            :, day_indices
        ].copy()
    )

    if current_day in threshold_days:

        j = threshold_days.index(
            current_day
        )

        our_selected_pnl[:, j] += (
            our_realized_current_day_pnl
        )

    # --------------------------------------------------------
    # Simulated daily 1st- and 2nd-place P&L
    # --------------------------------------------------------

    daily_first_pnl = (
        daily_place_pnl[
            :, 0, day_indices
        ].copy()
    )

    daily_second_pnl = (
        daily_place_pnl[
            :, 1, day_indices
        ].copy()
    )

    if current_day in threshold_days:

        if current_daily_place_pnl is None:
            raise ValueError(
                "current_daily_place_pnl must be supplied "
                "for the active current day."
            )

        current_daily_place_pnl = np.asarray(
            current_daily_place_pnl,
            dtype=float,
        )

        if current_daily_place_pnl.shape != (2,):
            raise ValueError(
                "current_daily_place_pnl must have shape (2,)."
            )

        j = threshold_days.index(
            current_day
        )

        daily_first_pnl[:, j] += (
            current_daily_place_pnl[0]
        )

        daily_second_pnl[:, j] += (
            current_daily_place_pnl[1]
        )

    # --------------------------------------------------------
    # DAILY PRIZE HURDLE
    #
    # For the currently active day we KNOW our observed
    # daily rank.
    #
    # If we are currently daily rank 1:
    #     compare against daily 2nd.
    #
    # Otherwise:
    #     compare against daily 1st.
    #
    # For future prize days there is no observed daily rank
    # yet, so retain the smooth endogenous 1st/2nd switch.
    # --------------------------------------------------------

    tau_c = competitor_temperature

    rank_gate = expit(
        (
            our_selected_pnl
            - daily_first_pnl
        )
        / tau_c
    )

    place_gap = (
        daily_second_pnl
        - daily_first_pnl
    )

    selected_threshold_pnl = (
        daily_first_pnl
        + rank_gate * place_gap
    )

    # Marks the currently active observed day, if it is
    # among the remaining prize days.
    current_day_column = None

    if current_day in threshold_days:

        current_day_column = (
            threshold_days.index(
                current_day
            )
        )

        if current_daily_rank is not None:

            if current_daily_rank == 1:

                selected_threshold_pnl[
                    :,
                    current_day_column
                ] = (
                    daily_second_pnl[
                        :,
                        current_day_column
                    ]
                )

            else:

                selected_threshold_pnl[
                    :,
                    current_day_column
                ] = (
                    daily_first_pnl[
                        :,
                        current_day_column
                    ]
                )

    margins = (
        our_selected_pnl
        - selected_threshold_pnl
    )

    # --------------------------------------------------------
    # Smooth probability of winning each day
    # --------------------------------------------------------

    tau_w = win_temperature

    q = expit(
        margins / tau_w
    )

    q_any = (
        1.0
        - np.prod(
            1.0 - q,
            axis=1,
        )
    )

    objective = (
        daily_prize
        * np.mean(q_any)
    )

    # ========================================================
    # DERIVATIVE WITH RESPECT TO OUR DAILY P&L
    # ========================================================
    #
    # H = P1 + g(P2-P1)
    #
    # dg/dA =
    #     g(1-g) / tau_c
    #
    # dH/dA =
    #     (P2-P1) * dg/dA
    #
    # margin = A - H
    #
    # dmargin/dA =
    #     1 - dH/dA
    # ========================================================

    dgate_dA = (
        rank_gate
        * (1.0 - rank_gate)
        / tau_c
    )

    dthreshold_dA = (
        place_gap
        * dgate_dA
    )

    dmargin_dA = (
        1.0
        - dthreshold_dA
    )

    # For the currently observed prize day, once our actual
    # daily rank identifies which OTHER player is the hurdle,
    # that threshold is independent of our portfolio weights.
    #
    # Therefore d threshold / d our P&L = 0 and
    # d margin / d our P&L = 1.

    if (
        current_day_column is not None
        and current_daily_rank is not None
    ):

        dthreshold_dA[
            :,
            current_day_column
        ] = 0.0

        dmargin_dA[
            :,
            current_day_column
        ] = 1.0

    dq_dA = (
        q
        * (1.0 - q)
        / tau_w
        * dmargin_dA
    )

    # --------------------------------------------------------
    # Derivative of:
    #
    # 1 - product_j(1-q_j)
    #
    # with respect to each day's P&L.
    # --------------------------------------------------------

    dF_dPnL = np.zeros(
        (N, D)
    )

    for j, d in enumerate(
        day_indices
    ):

        other_day_product = np.ones(
            N
        )

        for h in range(
            len(day_indices)
        ):

            if h != j:

                other_day_product *= (
                    1.0 - q[:, h]
                )

        dF_dPnL[:, d] = (
            daily_prize
            / N
            * other_day_product
            * dq_dA[:, j]
        )

    # ========================================================
    # PROPAGATE DAILY-P&L SENSITIVITY THROUGH ACCOUNT BALANCE
    # ========================================================

    effect_by_day = np.zeros(
        (N, D)
    )

    future_balance_effect = np.zeros(
        N
    )

    for d in range(
        D - 1,
        -1,
        -1,
    ):

        effect_by_day[:, d] = (
            dF_dPnL[:, d]
            * our_ending_balances[:, d]
            + future_balance_effect
        )

        future_balance_effect += (
            dF_dPnL[:, d]
            * our_daily_pnl[:, d]
        )

    day_to_index = {
        day: d
        for d, day in enumerate(days)
    }

    interval_day_index = np.array(
        [
            day_to_index[day]
            for day in day_labels
        ]
    )

    interval_effect = (
        effect_by_day[
            :, interval_day_index
        ]
    )

    growth = (
        1.0
        + our_interval_returns
    )

    if np.any(
        np.abs(growth) < 1e-12
    ):

        raise ValueError(
            "An interval produced an account "
            "growth factor too close to zero."
        )

    return_sensitivity = (
        R_sims
        / growth[:, :, None]
    )

    gradient = np.einsum(
        "nt,ntk->tk",
        interval_effect,
        return_sensitivity,
    )

    diagnostics = {
        "accounting":
            accounting,

        "threshold_days":
            threshold_days,

        "daily_first_pnl":
            daily_first_pnl,

        "daily_second_pnl":
            daily_second_pnl,

        "rank_gate":
            rank_gate,

        "daily_threshold_pnl":
            selected_threshold_pnl,

        "smooth_min_margin":
            margins,

        "smooth_daily_win":
            q,

        "smooth_any_win":
            q_any,

        "dF_dPnL":
            dF_dPnL,

        "probability_win_at_least_once":
            np.mean(q_any),

        "expected_daily_prize":
            objective,
    }

    return (
        objective,
        gradient.ravel(),
        diagnostics,
    )

def total_expected_prize_and_gradient(
    x,
    mu,
    Sigma,
    prizes,

    R_final_sims,
    final_place_balance_sims,

    R_sims,
    B_sims,

    day_labels,

    starting_balance=25000.0,
    daily_prize=500.0,
    final_rank_temperature=250.0,
    final_win_temperature=0.02,
    daily_competitor_temperature=1.0,
    daily_win_temperature=1.0,
    eligible_days=None,
    daily_threshold_days=None,

    current_final_place_balances=None,
    current_rank=None,
    
    current_day=None,
    our_realized_current_day_pnl=0.0,
    current_daily_rank=None,
    current_daily_place_pnl=None,
    daily_prize_already_won=False,
):
    """Combined final-rank prize + one-time daily prize."""
    T, K = mu.shape
    W = x.reshape(T, K)

    final_objective, final_gradient = expected_prize_and_gradient(
        x=x,

        R_final_sims=
            R_final_sims,

        final_place_balance_sims=
            final_place_balance_sims,

        prizes=
            prizes,

        starting_balance=
            starting_balance,

        current_final_place_balances=
            current_final_place_balances,

        current_rank=
            current_rank,

        rank_temperature=
            final_rank_temperature,

        win_temperature=
            final_win_temperature,
    )

    # ============================================================
    # STEP 7:
    # DAILY COMPONENT MAY ALREADY BE FINISHED
    # ============================================================

    no_daily_days_remaining = (
        daily_threshold_days is not None
        and len(daily_threshold_days) == 0
    )

    if (
        daily_prize_already_won
        or no_daily_days_remaining
    ):

        daily_objective = 0.0

        daily_gradient = np.zeros_like(
            final_gradient
        )

        daily_diagnostics = {
            "accounting": None,
            "threshold_days": [],
            "daily_threshold_pnl": None,
            "smooth_min_margin": None,
            "smooth_daily_win": None,
            "smooth_any_win": None,
            "dF_dPnL": None,

            "probability_win_at_least_once":
                0.0,

            "expected_daily_prize":
                0.0,
        }

    else:

        daily_objective, daily_gradient, daily_diagnostics = (
            smooth_daily_prize_and_gradient(
                W=W,
                R_sims=R_sims,
                B_sims=B_sims,
                day_labels=day_labels,
                starting_balance=starting_balance,
                daily_prize=daily_prize,
                competitor_temperature=
                    daily_competitor_temperature,
                win_temperature=
                    daily_win_temperature,
                eligible_days=eligible_days,
                daily_threshold_days=
                    daily_threshold_days,
                current_day=current_day,
                our_realized_current_day_pnl=
                    our_realized_current_day_pnl,
                current_daily_rank=
                    current_daily_rank,
                current_daily_place_pnl=
                    current_daily_place_pnl,
            )
        )

    total_objective = final_objective + daily_objective
    total_gradient = final_gradient + daily_gradient

    diagnostics = {
        "final_expected_prize": final_objective,
        "daily_expected_prize": daily_objective,
        "total_expected_prize": total_objective,
        "daily": daily_diagnostics,
    }

    return total_objective, total_gradient, diagnostics

def expected_prize_and_gradient(
    x,
    R_final_sims,
    final_place_balance_sims,
    prizes,
    starting_balance,
    current_final_place_balances,
    current_rank=None,
    rank_temperature=250.0,
    win_temperature=0.02,
):
    """
    Final-prize objective with endogenous rank-dependent thresholds.

    final_place_balance_sims contains SIX simulated leaderboard-place
    arithmetic-return processes in this order:

        1st, 2nd, 3rd, 4th, 15th, 16th

    For each prize objective, the hurdle smoothly switches between
    the place required to OBTAIN the prize and the next place required
    to MAINTAIN it.

    Example for 1st:
        not first -> Rank1 is hurdle
        first     -> Rank2 is hurdle
    """

    N, T, K = R_final_sims.shape

    N_B, C_places, T_B = final_place_balance_sims.shape

    if N_B != N or T_B != T:
        raise ValueError(
            "Final place simulations have incompatible shape."
        )

    if C_places != 6:
        raise ValueError(
            "final_place_balance_sims must contain exactly six place processes: "
            "1st, 2nd, 3rd, 4th, 15th, 16th."
        )

    prizes = np.asarray(prizes, dtype=float)

    if prizes.shape != (4,):
        raise ValueError(
            "prizes must contain four incremental final-prize values."
        )

    current_final_place_balances = np.asarray(
        current_final_place_balances,
        dtype=float,
    )

    if current_final_place_balances.shape != (6,):
        raise ValueError(
            "current_final_place_balances must have shape (6,)."
        )

    W = x.reshape(T, K)

    # ============================================================
    # OUR FUTURE BALANCE PATH
    # ============================================================

    portfolio_returns = np.einsum(
        "ntk,tk->nt",
        R_final_sims,
        W,
    )

    portfolio_growth = 1.0 + portfolio_returns

    if np.any(portfolio_growth <= 0.0):
    
        bad_sim, bad_t = np.argwhere(
            portfolio_growth <= 0.0
        )[0]
    
        contributions = (
            R_final_sims[bad_sim, bad_t, :]
            * W[bad_t, :]
        )
    
        print()
        print("NON-POSITIVE PORTFOLIO GROWTH DIAGNOSTIC")
        print(f"simulation: {bad_sim}")
        print(f"remaining-horizon t: {bad_t}")
        print(f"portfolio return: {portfolio_returns[bad_sim, bad_t]: .8f}")
        print(f"portfolio growth: {portfolio_growth[bad_sim, bad_t]: .8f}")
        print()
    
        future_names = [
            "CL",
            "GC",
            "SR3",
            "ZN",
            "KE",
            "HO",
            "ZW",
            "NQ",
        ]
    
        for k, name in enumerate(future_names):
            print(
                f"{name}: "
                f"weight={W[bad_t, k]: .8f}, "
                f"account_return={R_final_sims[bad_sim, bad_t, k]: .8f}, "
                f"contribution={contributions[k]: .8f}"
            )
    
        print()
        print(
            f"gross weight at bad interval: "
            f"{np.sum(np.abs(W[bad_t, :])): .8f}"
        )
        print(
            f"sum of contributions: "
            f"{np.sum(contributions): .8f}"
        )
        print()
    
        raise ValueError(
            "A simulated portfolio interval has non-positive growth."
        )

    our_growth_path = np.cumprod(
        portfolio_growth,
        axis=1,
    )

    our_balance_path = (
        starting_balance
        * our_growth_path
    )

    our_terminal_balance = (
        our_balance_path[:, -1]
    )

    # ============================================================
    # SIX LEADERBOARD-PLACE BALANCE PATHS
    #
    # These are already generated as structurally valid,
    # jointly simulated leaderboard balances.
    # ============================================================
    
    if final_place_balance_sims.shape != (
        N,
        6,
        T,
    ):
    
        raise ValueError(
            "final_place_balance_sims must have shape "
            f"({N}, 6, {T})."
        )
    
    if (
        not np.all(
            np.isfinite(
                final_place_balance_sims
            )
        )
        or np.any(
            final_place_balance_sims <= 0.0
        )
    ):
    
        raise ValueError(
            "final_place_balance_sims must contain "
            "positive finite balances."
        )
    
    place_terminal_balances = (
        final_place_balance_sims[
            :,
            :,
            -1
        ]
    )

    # ============================================================
    # FINAL-PRIZE COMPETITOR THRESHOLDS
    #
    # For each prize:
    #
    # If we currently DO NOT occupy the prize region,
    # compare against the place required to obtain it.
    #
    # If we currently DO occupy the prize region,
    # compare against the next place below the cutoff.
    #
    # This guarantees that our own current leaderboard slot
    # is never modeled as an independent competitor.
    #
    # Example:
    #
    # If current_rank == 1:
    #     1st prize -> compare against current 2nd
    #
    # If current_rank == 2:
    #     1st prize -> compare against current 1st
    #     top-2     -> compare against current 3rd
    # ============================================================

    place_pairs = [
        (0, 1),   # 1st / 2nd
        (1, 2),   # 2nd / 3rd
        (2, 3),   # 3rd / 4th
        (4, 5),   # 15th / 16th
    ]

    prize_rank_cutoffs = [
        1,
        2,
        3,
        15,
    ]

    objective = 0.0
    gradient = np.zeros((T, K))

    # ============================================================
    # d A_terminal / d w_tk
    #
    # A_T * R_tk / (1 + w_t'R_t)
    # ============================================================

    dA_dw = (
        our_terminal_balance[
            :,
            None,
            None
        ]
        * R_final_sims
        / portfolio_growth[
            :,
            :,
            None
        ]
    )
    # shape (N,T,K)

    # ============================================================
    # FOUR PRIZE OBJECTIVES
    # ============================================================

    for i, (
        obtain_idx,
        maintain_idx,
    ) in enumerate(
        place_pairs
    ):

        prize_cutoff = (
            prize_rank_cutoffs[i]
        )

        # --------------------------------------------------------
        # Select the relevant OTHER-player threshold.
        #
        # t=0:
        #     No realized rank exists yet, so use the ordinary
        #     obtain threshold.
        #
        # Live state:
        #
        #     current_rank <= cutoff
        #         We already qualify.
        #         The relevant rival is one place below
        #         the cutoff.
        #
        #     current_rank > cutoff
        #         We do not qualify.
        #         The relevant rival occupies the cutoff.
        # --------------------------------------------------------

        if current_rank is None:

            threshold_idx = (
                obtain_idx
            )

        elif current_rank <= prize_cutoff:

            threshold_idx = (
                maintain_idx
            )

        else:

            threshold_idx = (
                obtain_idx
            )

        threshold_balance = (
            place_terminal_balances[
                :,
                threshold_idx
            ]
        )

        # --------------------------------------------------------
        # Terminal relative log wealth
        # --------------------------------------------------------

        log_margin = (
            np.log(
                our_terminal_balance
            )
            - np.log(
                threshold_balance
            )
        )

        win_probability_path = expit(
            log_margin
            / win_temperature
        )

        objective += (
            prizes[i]
            * np.mean(
                win_probability_path
            )
        )

        # --------------------------------------------------------
        # Threshold path is an OTHER-player process.
        #
        # It does not depend on our portfolio weights.
        #
        # Therefore:
        #
        # d/dA [
        #     log(A)
        #     - log(threshold)
        # ]
        #
        # = 1/A
        # --------------------------------------------------------

        dmargin_dA = (
            1.0
            / our_terminal_balance
        )

        dwin_dA = (
            win_probability_path
            * (
                1.0
                - win_probability_path
            )
            / win_temperature
            * dmargin_dA
        )

        gradient += (
            prizes[i]
            / N
            * np.einsum(
                "n,ntk->tk",
                dwin_dA,
                dA_dw,
            )
        )

    return (
        objective,
        gradient.ravel(),
    )

def project_weight_vector(
    v,
    gross_limit=1.0,
    weight_bound=1.0,
    tol=1e-12,
):
    """
    Euclidean projection of one portfolio-weight vector onto:

        sum_k |w_k| <= gross_limit

    and

        |w_k| <= weight_bound[k]

    weight_bound may be either:

        - one positive scalar, or
        - one positive bound per future.
    """

    v = np.asarray(
        v,
        dtype=float,
    )

    if gross_limit < 0:
        raise ValueError(
            "gross_limit must be nonnegative"
        )

    weight_bound_array = np.asarray(
        weight_bound,
        dtype=float,
    )

    if weight_bound_array.ndim == 0:
        weight_bound_array = np.full(
            v.shape,
            float(weight_bound_array),
            dtype=float,
        )

    if weight_bound_array.shape != v.shape:
        raise ValueError(
            "weight_bound must be a scalar or have "
            "the same shape as v."
        )

    if not np.all(
        np.isfinite(
            weight_bound_array
        )
    ):
        raise ValueError(
            "weight_bound must contain only finite values."
        )

    if np.any(
        weight_bound_array <= 0.0
    ):
        raise ValueError(
            "All weight bounds must be positive."
        )

    abs_v = np.abs(
        v
    )

    clipped = np.minimum(
        abs_v,
        weight_bound_array,
    )

    if clipped.sum() <= gross_limit + tol:

        return (
            np.sign(v)
            * clipped
        )

    low = 0.0
    high = float(
        np.max(
            abs_v
        )
    )

    for _ in range(100):

        lam = 0.5 * (
            low + high
        )

        projected_abs = np.minimum(
            weight_bound_array,
            np.maximum(
                abs_v - lam,
                0.0,
            ),
        )

        total = float(
            projected_abs.sum()
        )

        if total > gross_limit:
            low = lam
        else:
            high = lam

    lam = high

    projected_abs = np.minimum(
        weight_bound_array,
        np.maximum(
            abs_v - lam,
            0.0,
        ),
    )

    return (
        np.sign(v)
        * projected_abs
    )

def project_weight_matrix(
    W,
    gross_limit=1.0,
    weight_bound=1.0,
):
    """
    Project every row W[t] independently.
    """

    projected = np.empty_like(W)

    for t in range(W.shape[0]):

        projected[t] = project_weight_vector(
            W[t],
            gross_limit=gross_limit,
            weight_bound=weight_bound,
        )

    return projected

def project_weight_matrix_for_execution(
    W,
    current_weights,
    gross_limit=1.0,
    weight_bound=1.0,
    weight_bound_matrix=None,
    tol=1e-12,
):
    """
    Sequential projection of optimizer target weights under the
    grandfathered gross-exposure constraint.

    TRANSITION RULE
    ---------------
    Let previous_weights be the position entering a transition
    and proposed_weights be the requested next position.

    The next position may be in either of two feasible regions:

    1. GRANDFATHERED REDUCTION REGION

       If gross exposure remains above gross_limit, every
       component's absolute exposure must be no larger than its
       immediately preceding absolute exposure.

       Existing exposure may therefore be maintained or reduced,
       but excess grandfathered exposure cannot be increased or
       reallocated to another future.

    2. NORMAL REGION

       The portfolio may be reallocated normally provided:

           sum(abs(weights)) <= gross_limit

       and:

           abs(weights[k]) <= weight_bound

    The constraint is sequential:

        current_weights -> W[0] -> W[1] -> W[2] -> ...

    Therefore, once grandfathered exposure is reduced, the
    surrendered exposure cannot reappear in a later row unless
    the portfolio first moves into the normal gross <= 100%
    region.

    For each transition, this function computes the Euclidean
    projection onto both feasible regions and chooses whichever
    feasible projection is closer to the optimizer's proposed
    row.
    """

    W = np.asarray(
        W,
        dtype=float,
    )

    current_weights = np.asarray(
        current_weights,
        dtype=float,
    )

    if W.ndim != 2:
        raise ValueError(
            "W must be a two-dimensional weight matrix."
        )

    if current_weights.shape != (
        W.shape[1],
    ):
        raise ValueError(
            "current_weights must match the number "
            "of futures in W."
        )

    if not np.all(
        np.isfinite(
            W
        )
    ):
        raise ValueError(
            "W must contain only finite values."
        )

    if not np.all(
        np.isfinite(
            current_weights
        )
    ):
        raise ValueError(
            "current_weights must contain only finite values."
        )

    if gross_limit <= 0.0:
        raise ValueError(
            "gross_limit must be positive."
        )

    if weight_bound <= 0.0:
        raise ValueError(
            "weight_bound must be positive."
        )
    
    if weight_bound_matrix is None:
    
        weight_bound_matrix = np.full(
            W.shape,
            float(weight_bound),
            dtype=float,
        )
    
    else:
    
        weight_bound_matrix = np.asarray(
            weight_bound_matrix,
            dtype=float,
        )
    
        if weight_bound_matrix.shape != W.shape:
            raise ValueError(
                "weight_bound_matrix must have the same "
                "shape as W."
            )
    
        if not np.all(
            np.isfinite(
                weight_bound_matrix
            )
        ):
            raise ValueError(
                "weight_bound_matrix must contain only "
                "finite values."
            )
    
        if np.any(
            weight_bound_matrix <= 0.0
        ):
            raise ValueError(
                "All entries of weight_bound_matrix "
                "must be positive."
            )
    
        weight_bound_matrix = np.minimum(
            weight_bound_matrix,
            float(weight_bound),
        )
    
    projected = np.empty_like(
        W
    )

    previous_weights = current_weights.copy()

    for t in range(
        W.shape[0]
    ):

        proposed = W[
            t,
            :
        ]

        row_weight_bound = (
            weight_bound_matrix[
                t,
                :
            ]
        )

        previous_gross = float(
            np.sum(
                np.abs(
                    previous_weights
                )
            )
        )

        # ----------------------------------------------------
        # NORMAL REGION
        #
        # Any reallocation is allowed provided the resulting
        # portfolio has ordinary gross exposure <= 100%.
        # ----------------------------------------------------

        normal_candidate = (
            project_weight_vector(
                proposed,
                gross_limit=gross_limit,
                weight_bound=row_weight_bound,
            )
        )

        # ----------------------------------------------------
        # GRANDFATHERED REDUCTION REGION
        #
        # This region is relevant only when the immediately
        # preceding portfolio itself exceeds the ordinary gross
        # limit.
        #
        # Each future may maintain or reduce its existing
        # same-sign absolute exposure. It may not increase,
        # reverse, or create exposure in a previously-flat
        # future while remaining above 100% gross.
        # ----------------------------------------------------

        grandfathered_candidate = None

        if previous_gross > gross_limit + tol:

            grandfathered_candidate = np.zeros(
                W.shape[1],
                dtype=float,
            )

            for k in range(
                W.shape[1]
            ):

                previous_k = float(
                    previous_weights[k]
                )

                proposed_k = float(
                    proposed[k]
                )

                previous_abs = abs(
                    previous_k
                )

                if previous_abs <= tol:

                    grandfathered_candidate[k] = 0.0
                    continue

                previous_sign = np.sign(
                    previous_k
                )

                proposed_same_sign_amount = (
                    previous_sign
                    * proposed_k
                )

                grandfathered_abs = np.clip(
                    proposed_same_sign_amount,
                    0.0,
                    min(
                        previous_abs,
                        float(
                            row_weight_bound[k]
                        ),
                    ),
                )

                grandfathered_candidate[k] = (
                    previous_sign
                    * grandfathered_abs
                )

            grandfathered_gross = float(
                np.sum(
                    np.abs(
                        grandfathered_candidate
                    )
                )
            )

            # If the grandfathered projection has already
            # reduced gross exposure to the ordinary limit or
            # below, the normal region contains feasible
            # reallocations and is sufficient. Keeping this
            # candidate is harmless, but both candidates remain
            # explicitly available for nearest-point selection.

            if (
                grandfathered_gross
                > previous_gross + tol
            ):
                raise RuntimeError(
                    "Grandfathered projection increased gross "
                    "exposure unexpectedly."
                )

        # ----------------------------------------------------
        # CHOOSE THE NEAREST FEASIBLE REGION
        # ----------------------------------------------------

        if grandfathered_candidate is None:

            chosen = normal_candidate

        else:

            normal_distance = float(
                np.sum(
                    (
                        normal_candidate
                        - proposed
                    ) ** 2
                )
            )

            grandfathered_distance = float(
                np.sum(
                    (
                        grandfathered_candidate
                        - proposed
                    ) ** 2
                )
            )

            if (
                grandfathered_distance
                < normal_distance
            ):
                chosen = grandfathered_candidate
            else:
                chosen = normal_candidate

        # ----------------------------------------------------
        # DEFENSIVE VALIDATION OF THE CHOSEN TRANSITION
        # ----------------------------------------------------

        chosen_gross = float(
            np.sum(
                np.abs(
                    chosen
                )
            )
        )

        if chosen_gross > gross_limit + tol:

            if previous_gross <= gross_limit + tol:
                raise RuntimeError(
                    "Projected row exceeds the gross limit "
                    "after the portfolio had already returned "
                    "to the normal regime."
                )

            if np.any(
                np.abs(
                    chosen
                )
                >
                np.abs(
                    previous_weights
                )
                + tol
            ):
                raise RuntimeError(
                    "Projected row illegally increases "
                    "absolute exposure while gross exposure "
                    "remains above the ordinary limit."
                )

            same_sign_or_zero = (
                (
                    np.abs(
                        chosen
                    )
                    <= tol
                )
                |
                (
                    np.sign(
                        chosen
                    )
                    ==
                    np.sign(
                        previous_weights
                    )
                )
            )

            if not np.all(
                same_sign_or_zero
            ):
                raise RuntimeError(
                    "Projected grandfathered row illegally "
                    "reverses a position while gross exposure "
                    "remains above the ordinary limit."
                )

        if np.any(
            np.abs(
                chosen
            )
            >
            row_weight_bound
            + tol
        ):
            raise RuntimeError(
                "Projected row violates its individual "
                "exposure bound."
            )

        projected[
            t,
            :
        ] = chosen

        previous_weights = chosen.copy()

    return projected

def calculate_realized_history(
    past_weights,
    realized_returns,
    day_labels_past,
    initial_balance=25000.0,
):
    """
    Calculate our locked realized history for intervals that
    have already occurred.

    Parameters
    ----------
    past_weights:
        Shape (current_t, K)

        Actual weights that were used in completed intervals.

    realized_returns:
        Shape (current_t, K)

        Actual realized futures returns in completed intervals.

    day_labels_past:
        Length current_t.

        Day corresponding to each completed interval.

    initial_balance:
        Account balance at the very beginning of the competition.

    Returns
    -------
    Dictionary containing:
        realized_interval_portfolio_returns
        realized_growth_factors
        realized_balance_path
        current_balance
        total_realized_return
        day_labels_past
    """

    past_weights = np.asarray(
        past_weights,
        dtype=float,
    )

    realized_returns = np.asarray(
        realized_returns,
        dtype=float,
    )

    day_labels_past = np.asarray(
        day_labels_past
    )


    # ========================================================
    # BASIC VALIDATION
    # ========================================================

    if past_weights.ndim != 2:
        raise ValueError(
            "past_weights must be a 2D array."
        )

    if realized_returns.shape != past_weights.shape:
        raise ValueError(
            "realized_returns must have the same shape "
            "as past_weights."
        )

    current_t, K = past_weights.shape

    if len(day_labels_past) != current_t:
        raise ValueError(
            "day_labels_past must have length current_t."
        )

    if not np.all(
        np.isfinite(
            realized_returns
        )
    ):
        raise ValueError(
            "realized_returns contains non-finite values."
        )


    # ========================================================
    # REALIZED PORTFOLIO RETURN IN EACH COMPLETED INTERVAL
    #
    # r_A,t = w_t' R_t
    # ========================================================

    realized_interval_portfolio_returns = np.einsum(
        "tk,tk->t",
        past_weights,
        realized_returns,
    )


    # ========================================================
    # LOCKED GROWTH FACTORS
    # ========================================================

    realized_growth_factors = (
        1.0
        + realized_interval_portfolio_returns
    )

    if np.any(
        realized_growth_factors <= 0.0
    ):
        raise ValueError(
            "A realized interval produced a non-positive "
            "account growth factor."
        )


    # ========================================================
    # REALIZED ACCOUNT BALANCE PATH
    # ========================================================

    realized_balance_path = np.empty(
        current_t + 1
    )

    realized_balance_path[0] = (
        initial_balance
    )

    for t in range(current_t):

        realized_balance_path[
            t + 1
        ] = (
            realized_balance_path[t]
            * realized_growth_factors[t]
        )


    current_balance = (
        realized_balance_path[-1]
    )

    total_realized_return = (
        current_balance
        / initial_balance
        - 1.0
    )


    return {
        "realized_interval_portfolio_returns":
            realized_interval_portfolio_returns,

        "realized_growth_factors":
            realized_growth_factors,

        "realized_balance_path":
            realized_balance_path,

        "current_balance":
            current_balance,

        "total_realized_return":
            total_realized_return,

        "day_labels_past":
            day_labels_past.copy(),
    }

def load_margin_forecasts(
    T,
    K=8,
):
    """
    Load time-varying futures margin forecasts.

    margin[t, k] means:

        forecast dollar margin required
        per contract

    for future k at original competition interval t.

    The file must be:

        margin.csv

    with columns:

        t
        Future_1
        ...
        Future_K

    or, for rolling forecasts:

        t_original
        Future_1
        ...
        Future_K

    Returns
    -------
    margin:
        Shape (T, K)

    Important
    ---------
    Margin is a primitive input to the account-return
    conversion.

    Effective leverage will later be derived inside the
    optimizer from simulated contract notional divided by
    forecast margin.
    """

    DATA_DIR = ACTIVE_FORECAST_DIR

    future_names = [
        f"Future_{k+1}"
        for k in range(K)
    ]

    margin_df = pd.read_csv(
        os.path.join(
            DATA_DIR,
            "margin.csv",
        )
    )

    # --------------------------------------------------------
    # Validate futures columns
    # --------------------------------------------------------

    missing_futures = [
        name
        for name in future_names
        if name not in margin_df.columns
    ]

    if missing_futures:
        raise ValueError(
            "margin.csv is missing futures: "
            f"{missing_futures}"
        )

    # --------------------------------------------------------
    # Determine original time indices
    # --------------------------------------------------------

    if "t_original" in margin_df.columns:

        margin_times = (
            margin_df["t_original"]
            .to_numpy(dtype=int)
        )

    elif "t" in margin_df.columns:

        margin_times = (
            margin_df["t"]
            .to_numpy(dtype=int)
        )

    else:

        raise ValueError(
            "margin.csv must contain either "
            "'t_original' or 't'."
        )

    if np.any(
        margin_times < 0
    ) or np.any(
        margin_times >= T
    ):

        raise ValueError(
            "margin.csv contains time indices "
            f"outside [0, {T - 1}]."
        )

    if len(
        np.unique(margin_times)
    ) != len(
        margin_times
    ):

        raise ValueError(
            "margin.csv contains duplicate "
            "time indices."
        )

    # --------------------------------------------------------
    # Full-timeline margin matrix
    #
    # NaN is deliberate for unsupplied periods. We do not
    # silently invent a margin value.
    # --------------------------------------------------------

    margin = np.full(
        (T, K),
        np.nan,
        dtype=float,
    )

    margin[
        margin_times,
        :
    ] = (
        margin_df[
            future_names
        ]
        .to_numpy(dtype=float)
    )

    # --------------------------------------------------------
    # Validation of supplied values
    # --------------------------------------------------------

    supplied_margin = margin[
        margin_times,
        :
    ]

    if not np.all(
        np.isfinite(
            supplied_margin
        )
    ):

        raise ValueError(
            "margin.csv contains non-finite values."
        )

    if np.any(
        supplied_margin <= 0.0
    ):

        raise ValueError(
            "All supplied margin values must be positive."
        )

    assert margin.shape == (
        T,
        K,
    )

    return margin

def build_pathwise_leverage_and_account_returns(
    R_sims,
    starting_notional,
    margin,
):
    """
    Convert raw simulated futures returns into leveraged
    account returns using pathwise contract notionals.

    Parameters
    ----------
    R_sims:
        Shape (N, T, K)

        Raw simulated arithmetic futures price returns.

    starting_notional:
        Shape (K,)

        Dollar notional of one contract at the beginning
        of the first simulated interval.

    margin:
        Shape (T, K)

        Forecast dollar margin required per contract.

        Margin is deterministic across Monte Carlo paths
        in the current architecture.

    Returns
    -------
    R_account_sims:
        Shape (N, T, K)

        Leveraged return per unit of margin weight.

    pathwise_leverage:
        Shape (N, T, K)

        Effective contract leverage at the START of each
        interval:

            notional / margin

    pathwise_notional:
        Shape (N, T, K)

        Dollar notional at the START of each interval.

    Timing
    ------
    For interval t:

        leverage_t = notional_t / margin_t

        account_return_t
            = leverage_t * futures_return_t

    Then:

        notional_{t+1}
            = notional_t * (1 + futures_return_t)

    Therefore the return realized during interval t changes
    leverage beginning in interval t+1, not during the same
    interval.
    """

    R_sims = np.asarray(
        R_sims,
        dtype=float,
    )

    starting_notional = np.asarray(
        starting_notional,
        dtype=float,
    )

    margin = np.asarray(
        margin,
        dtype=float,
    )

    # --------------------------------------------------------
    # SHAPE VALIDATION
    # --------------------------------------------------------

    if R_sims.ndim != 3:
        raise ValueError(
            "R_sims must have shape (N, T, K)."
        )

    N, T, K = R_sims.shape

    if starting_notional.shape != (K,):
        raise ValueError(
            "starting_notional must have shape "
            f"({K},)."
        )

    if margin.shape != (
        T,
        K,
    ):
        raise ValueError(
            "margin must have shape "
            f"({T}, {K})."
        )

    # --------------------------------------------------------
    # VALUE VALIDATION
    # --------------------------------------------------------

    if not np.all(
        np.isfinite(
            R_sims
        )
    ):
        raise ValueError(
            "R_sims contains non-finite values."
        )

    if not np.all(
        np.isfinite(
            starting_notional
        )
    ):
        raise ValueError(
            "starting_notional contains "
            "non-finite values."
        )

    if np.any(
        starting_notional <= 0.0
    ):
        raise ValueError(
            "All starting notionals must be positive."
        )

    if not np.all(
        np.isfinite(
            margin
        )
    ):
        raise ValueError(
            "margin contains non-finite values."
        )

    if np.any(
        margin <= 0.0
    ):
        raise ValueError(
            "All margin values must be positive."
        )

    if np.any(
        1.0 + R_sims <= 0.0
    ):
        raise ValueError(
            "Simulated futures returns imply a "
            "non-positive contract notional."
        )

    # --------------------------------------------------------
    # PATHWISE START-OF-INTERVAL NOTIONAL
    # --------------------------------------------------------

    pathwise_notional = np.empty(
        (
            N,
            T,
            K,
        ),
        dtype=float,
    )

    pathwise_notional[
        :,
        0,
        :
    ] = (
        starting_notional[
            None,
            :
        ]
    )

    for t in range(
        1,
        T,
    ):

        pathwise_notional[
            :,
            t,
            :
        ] = (
            pathwise_notional[
                :,
                t - 1,
                :
            ]
            * (
                1.0
                + R_sims[
                    :,
                    t - 1,
                    :
                ]
            )
        )

    # --------------------------------------------------------
    # PATHWISE EFFECTIVE LEVERAGE
    #
    # Leverage for interval t uses the notional that exists
    # at the START of interval t.
    # --------------------------------------------------------

    pathwise_leverage = (
        pathwise_notional
        / margin[
            None,
            :,
            :
        ]
    )

    # --------------------------------------------------------
    # ACCOUNT RETURN PER UNIT OF MARGIN WEIGHT
    # --------------------------------------------------------

    R_account_sims = (
        R_sims
        * pathwise_leverage
    )

    # --------------------------------------------------------
    # FINAL VALIDATION
    # --------------------------------------------------------

    expected_shape = (
        N,
        T,
        K,
    )

    if pathwise_notional.shape != expected_shape:
        raise ValueError(
            "pathwise_notional has incorrect shape."
        )

    if pathwise_leverage.shape != expected_shape:
        raise ValueError(
            "pathwise_leverage has incorrect shape."
        )

    if R_account_sims.shape != expected_shape:
        raise ValueError(
            "R_account_sims has incorrect shape."
        )

    if not np.all(
        np.isfinite(
            pathwise_notional
        )
    ):
        raise ValueError(
            "pathwise_notional contains "
            "non-finite values."
        )

    if not np.all(
        np.isfinite(
            pathwise_leverage
        )
    ):
        raise ValueError(
            "pathwise_leverage contains "
            "non-finite values."
        )

    if not np.all(
        np.isfinite(
            R_account_sims
        )
    ):
        raise ValueError(
            "R_account_sims contains "
            "non-finite values."
        )

    return (
        R_account_sims,
        pathwise_leverage,
        pathwise_notional,
    )

# ============================================================
# DOLLAR-BALANCE ACCOUNT ENGINE
#
# Optimizer decision variables remain dimensionless target
# weights, but their economic interpretation is now:
#
#     desired_margin_dollars[t, k]
#         = W[t, k] * reference_balance
#
# reference_balance is fixed at the known balance at the start
# of the optimization run. It does NOT vary across future
# Monte Carlo paths.
#
# Actual positions are path-specific because future balance
# and carried positions are path-specific.
# ============================================================


def required_margin_from_contracts(
    q,
    margin_per_contract,
):
    q = np.asarray(
        q,
        dtype=float,
    )

    margin_per_contract = np.asarray(
        margin_per_contract,
        dtype=float,
    )

    if q.shape != margin_per_contract.shape:
        raise ValueError(
            "q and margin_per_contract must have the same shape."
        )

    if np.any(~np.isfinite(q)):
        raise ValueError(
            "q must contain only finite values."
        )

    if np.any(~np.isfinite(margin_per_contract)):
        raise ValueError(
            "margin_per_contract must contain only finite values."
        )

    if np.any(margin_per_contract <= 0.0):
        raise ValueError(
            "margin_per_contract must be strictly positive."
        )

    return float(
        np.sum(
            np.abs(q)
            * margin_per_contract
        )
    )


def execute_target_contracts(
    balance,
    q_old,
    q_target,
    margin_per_contract,
    tol=1e-12,
):
    """
    Execute one desired target under grandfathered margin rules.

    Existing positions may be held, reduced, or closed even
    when the account is over-margin.

    New/increased exposure requires normal margin capacity.

    While the account remains over-margin, reductions cannot
    be redeployed into new exposure.

    Reversals close the old side first. Opposite-side exposure
    is treated as new exposure.
    """

    balance = float(balance)

    q_old = np.asarray(
        q_old,
        dtype=float,
    )

    q_target = np.asarray(
        q_target,
        dtype=float,
    )

    margin_per_contract = np.asarray(
        margin_per_contract,
        dtype=float,
    )

    if q_old.shape != q_target.shape:
        raise ValueError(
            "q_old and q_target must have the same shape."
        )

    if q_old.shape != margin_per_contract.shape:
        raise ValueError(
            "Position and margin arrays must have the same shape."
        )

    if not np.isfinite(balance):
        raise ValueError(
            "balance must be finite."
        )

    if np.any(~np.isfinite(q_old)):
        raise ValueError(
            "q_old must contain only finite values."
        )

    if np.any(~np.isfinite(q_target)):
        raise ValueError(
            "q_target must contain only finite values."
        )

    if np.any(~np.isfinite(margin_per_contract)):
        raise ValueError(
            "margin_per_contract must contain only finite values."
        )

    if np.any(margin_per_contract <= 0.0):
        raise ValueError(
            "margin_per_contract must be strictly positive."
        )

    normal_capacity = max(
        balance,
        0.0,
    )

    old_margin = required_margin_from_contracts(
        q_old,
        margin_per_contract,
    )

    target_margin = required_margin_from_contracts(
        q_target,
        margin_per_contract,
    )

    # --------------------------------------------------------
    # ACCOUNT STARTS OVER-MARGIN
    #
    # Execution occurs in two conceptual stages:
    #
    #   1. Existing exposure may move toward zero.
    #
    #   2. Only if those reductions restore the account to
    #      normal margin capacity may new/increased exposure
    #      be established, using only the capacity remaining
    #      after the reduced existing positions.
    #
    # This prevents margin released while the account is still
    # over-margin from being redeployed elsewhere.
    # --------------------------------------------------------

    if old_margin > normal_capacity + tol:

        q_reduced = q_old.copy()

        desired_additions = np.zeros_like(
            q_old
        )

        for k in range(
            len(q_old)
        ):
            old_k = q_old[k]
            target_k = q_target[k]

            # No existing position:
            # the entire target is new exposure.
            if abs(old_k) <= tol:
                q_reduced[k] = 0.0
                desired_additions[k] = target_k
                continue

            # Target is flat:
            # close the existing position.
            if abs(target_k) <= tol:
                q_reduced[k] = 0.0
                continue

            # Same side.
            if old_k * target_k > 0.0:

                # Reduction or unchanged exposure.
                if abs(target_k) <= abs(old_k):
                    q_reduced[k] = target_k

                # Increase:
                # retain the old position first and treat only
                # the incremental amount as new exposure.
                else:
                    q_reduced[k] = old_k
                    desired_additions[k] = (
                        target_k
                        - old_k
                    )

                continue

            # Reversal:
            # close the old side first. The entire opposite-side
            # target is new exposure.
            q_reduced[k] = 0.0
            desired_additions[k] = target_k

        reduced_margin = (
            required_margin_from_contracts(
                q_reduced,
                margin_per_contract,
            )
        )

        # Still over-margin after every requested permissible
        # reduction. No new exposure may be established.
        if (
            reduced_margin
            > normal_capacity + tol
        ):
            return q_reduced

        # The reductions have restored normal margin capacity.
        # Only the capacity remaining AFTER those reductions
        # can now support new exposure.
        available_for_new_margin = max(
            normal_capacity
            - reduced_margin,
            0.0,
        )

        desired_new_margin = (
            required_margin_from_contracts(
                desired_additions,
                margin_per_contract,
            )
        )

        if desired_new_margin <= tol:
            return q_reduced

        scale = min(
            1.0,
            available_for_new_margin
            / desired_new_margin,
        )

        return (
            q_reduced
            + scale * desired_additions
        )

    # --------------------------------------------------------
    # ACCOUNT STARTS WITHIN NORMAL CAPACITY
    # --------------------------------------------------------

    if target_margin <= normal_capacity + tol:
        return q_target.copy()

    q_base = q_old.copy()

    desired_additions = np.zeros_like(
        q_old
    )

    for k in range(
        len(q_old)
    ):
        old_k = q_old[k]
        target_k = q_target[k]

        if abs(old_k) <= tol:
            q_base[k] = 0.0
            desired_additions[k] = target_k
            continue

        if abs(target_k) <= tol:
            q_base[k] = 0.0
            continue

        if old_k * target_k > 0.0:

            if abs(target_k) <= abs(old_k):
                q_base[k] = target_k

            else:
                q_base[k] = old_k
                desired_additions[k] = (
                    target_k
                    - old_k
                )

            continue

        q_base[k] = 0.0
        desired_additions[k] = target_k

    base_margin = required_margin_from_contracts(
        q_base,
        margin_per_contract,
    )

    available_for_new_margin = max(
        normal_capacity
        - base_margin,
        0.0,
    )

    desired_new_margin = (
        required_margin_from_contracts(
            desired_additions,
            margin_per_contract,
        )
    )

    if desired_new_margin <= tol:
        return q_base

    scale = min(
        1.0,
        available_for_new_margin
        / desired_new_margin,
    )

    return (
        q_base
        + scale * desired_additions
    )


def build_initial_contract_position(
    current_weights,
    current_balance,
    current_margin,
):
    """
    Recover the signed carried contract-equivalent quantities
    from the observed current margin weights.

    Stage 7 defines:

        current_weight_k
            = signed_margin_dollars_k / current_balance

    Therefore:

        q_k
            = current_weight_k
              * current_balance
              / margin_per_contract_k
    """

    current_weights = np.asarray(
        current_weights,
        dtype=float,
    )

    current_margin = np.asarray(
        current_margin,
        dtype=float,
    )

    current_balance = float(
        current_balance
    )

    if current_weights.shape != current_margin.shape:
        raise ValueError(
            "current_weights and current_margin must "
            "have the same shape."
        )

    if not np.isfinite(current_balance):
        raise ValueError(
            "current_balance must be finite."
        )

    if np.any(~np.isfinite(current_weights)):
        raise ValueError(
            "current_weights must contain only finite values."
        )

    if np.any(~np.isfinite(current_margin)):
        raise ValueError(
            "current_margin must contain only finite values."
        )

    if np.any(current_margin <= 0.0):
        raise ValueError(
            "current_margin must be strictly positive."
        )

    return (
        current_weights
        * current_balance
        / current_margin
    )


def simulate_dollar_balance_account(
    W,
    raw_return_sims,
    pathwise_notional,
    margin,
    starting_balance,
    initial_contracts,
    reference_balance=None,
):
    """
    Simulate our account using additive dollar P&L.

    Parameters
    ----------
    W:
        Shape (T, K).

        Dimensionless optimizer target variables.

        Their interpretation is:

            desired signed margin dollars
                = W[t, k] * reference_balance

    raw_return_sims:
        Shape (N, T, K).

        Raw arithmetic futures returns.

    pathwise_notional:
        Shape (N, T, K).

        Start-of-interval dollar notional per contract.

    margin:
        Shape (T, K).

        Margin dollars per contract.

    starting_balance:
        Actual known balance at the start of this optimization.

    initial_contracts:
        Shape (K,).

        Actual carried contract-equivalent quantities entering
        the first remaining interval.

    reference_balance:
        Fixed positive dollar scale used to interpret W.

        If None, starting_balance is used.

    Returns
    -------
    accounting:
        Dictionary containing pathwise balances, dollar P&L,
        desired targets, and actual executed positions.
    """

    W = np.asarray(
        W,
        dtype=float,
    )

    raw_return_sims = np.asarray(
        raw_return_sims,
        dtype=float,
    )

    pathwise_notional = np.asarray(
        pathwise_notional,
        dtype=float,
    )

    margin = np.asarray(
        margin,
        dtype=float,
    )

    initial_contracts = np.asarray(
        initial_contracts,
        dtype=float,
    )

    starting_balance = float(
        starting_balance
    )

    if reference_balance is None:
        reference_balance = (
            starting_balance
        )

    reference_balance = float(
        reference_balance
    )

    if W.ndim != 2:
        raise ValueError(
            "W must have shape (T, K)."
        )

    T, K = W.shape

    if raw_return_sims.ndim != 3:
        raise ValueError(
            "raw_return_sims must have shape (N, T, K)."
        )

    N = raw_return_sims.shape[0]

    if raw_return_sims.shape != (
        N,
        T,
        K,
    ):
        raise ValueError(
            "raw_return_sims has incorrect shape."
        )

    if pathwise_notional.shape != (
        N,
        T,
        K,
    ):
        raise ValueError(
            "pathwise_notional has incorrect shape."
        )

    if margin.shape != (
        T,
        K,
    ):
        raise ValueError(
            "margin must have shape (T, K)."
        )

    if initial_contracts.shape != (K,):
        raise ValueError(
            "initial_contracts must have shape (K,)."
        )

    if not np.isfinite(starting_balance):
        raise ValueError(
            "starting_balance must be finite."
        )

    if (
        not np.isfinite(reference_balance)
        or reference_balance <= 0.0
    ):
        raise ValueError(
            "reference_balance must be a positive finite number."
        )

    if np.any(~np.isfinite(W)):
        raise ValueError(
            "W must contain only finite values."
        )

    if np.any(~np.isfinite(raw_return_sims)):
        raise ValueError(
            "raw_return_sims must contain only finite values."
        )

    if np.any(~np.isfinite(pathwise_notional)):
        raise ValueError(
            "pathwise_notional must contain only finite values."
        )

    if np.any(pathwise_notional <= 0.0):
        raise ValueError(
            "pathwise_notional must be strictly positive."
        )

    if np.any(~np.isfinite(margin)):
        raise ValueError(
            "margin must contain only finite values."
        )

    if np.any(margin <= 0.0):
        raise ValueError(
            "margin must be strictly positive."
        )

    if np.any(~np.isfinite(initial_contracts)):
        raise ValueError(
            "initial_contracts must contain only finite values."
        )

    desired_margin_dollars = (
        W
        * reference_balance
    )

    desired_contracts = (
        desired_margin_dollars
        / margin
    )

    balance_start = np.empty(
        (N, T),
        dtype=float,
    )

    balance_end = np.empty(
        (N, T),
        dtype=float,
    )

    interval_pnl = np.empty(
        (N, T),
        dtype=float,
    )

    pnl_by_future = np.empty(
        (N, T, K),
        dtype=float,
    )

    executed_contracts = np.empty(
        (N, T, K),
        dtype=float,
    )

    executed_margin = np.empty(
        (N, T),
        dtype=float,
    )

    q_old = np.broadcast_to(
        initial_contracts,
        (N, K),
    ).copy()

    balance = np.full(
        N,
        starting_balance,
        dtype=float,
    )

    for t in range(T):

        balance_start[:, t] = (
            balance
        )

        q_target_t = (
            desired_contracts[t]
        )

        margin_t = margin[t]

        q_new = np.empty(
            (N, K),
            dtype=float,
        )

        for n in range(N):
            q_new[n] = (
                execute_target_contracts(
                    balance=balance[n],
                    q_old=q_old[n],
                    q_target=q_target_t,
                    margin_per_contract=margin_t,
                )
            )

        executed_contracts[:, t, :] = (
            q_new
        )

        executed_margin[:, t] = np.sum(
            np.abs(q_new)
            * margin_t[None, :],
            axis=1,
        )

        pnl_t_by_future = (
            q_new
            * pathwise_notional[:, t, :]
            * raw_return_sims[:, t, :]
        )

        pnl_t = np.sum(
            pnl_t_by_future,
            axis=1,
        )

        pnl_by_future[:, t, :] = (
            pnl_t_by_future
        )

        interval_pnl[:, t] = (
            pnl_t
        )

        balance = (
            balance
            + pnl_t
        )

        balance_end[:, t] = (
            balance
        )

        q_old = q_new

    return {
        "reference_balance":
            reference_balance,

        "desired_margin_dollars":
            desired_margin_dollars,

        "desired_contracts":
            desired_contracts,

        "executed_contracts":
            executed_contracts,

        "executed_margin":
            executed_margin,

        "balance_start":
            balance_start,

        "balance_end":
            balance_end,

        "interval_pnl":
            interval_pnl,

        "pnl_by_future":
            pnl_by_future,

        "terminal_balance":
            balance_end[:, -1],
    }

def calculate_daily_accounting_from_dollar_account(
    account,
    B_sims,
    day_labels,
):
    """
    Aggregate the frozen dollar-balance account simulation into
    competition-day dollar P&L.

    B_sims contains FUTURE dollar changes in the two modeled
    daily leaderboard thresholds.
    """

    interval_pnl = np.asarray(
        account["interval_pnl"],
        dtype=float,
    )

    balance_start = np.asarray(
        account["balance_start"],
        dtype=float,
    )

    balance_end = np.asarray(
        account["balance_end"],
        dtype=float,
    )

    B_sims = np.asarray(
        B_sims,
        dtype=float,
    )

    day_labels = np.asarray(
        day_labels
    )

    if interval_pnl.ndim != 2:
        raise ValueError(
            "account interval_pnl must have shape (N, T)."
        )

    N, T = interval_pnl.shape

    if balance_start.shape != (N, T):
        raise ValueError(
            "account balance_start must have shape (N, T)."
        )

    if balance_end.shape != (N, T):
        raise ValueError(
            "account balance_end must have shape (N, T)."
        )

    if B_sims.ndim != 3:
        raise ValueError(
            "B_sims must have shape (N, 2, T)."
        )

    if B_sims.shape != (N, 2, T):
        raise ValueError(
            "B_sims must have shape (N, 2, T)."
        )

    if len(day_labels) != T:
        raise ValueError(
            "day_labels must have length T."
        )

    if not np.all(
        np.isfinite(
            interval_pnl
        )
    ):
        raise ValueError(
            "account interval_pnl must be finite."
        )

    if not np.all(
        np.isfinite(
            B_sims
        )
    ):
        raise ValueError(
            "B_sims must be finite."
        )

    days = list(
        dict.fromkeys(
            day_labels.tolist()
        )
    )

    D = len(days)

    our_daily_pnl = np.empty(
        (N, D),
        dtype=float,
    )

    daily_place_pnl = np.empty(
        (N, 2, D),
        dtype=float,
    )

    our_beginning_balances = np.empty(
        (N, D),
        dtype=float,
    )

    our_ending_balances = np.empty(
        (N, D),
        dtype=float,
    )

    for d, day in enumerate(days):

        indices = np.flatnonzero(
            day_labels == day
        )

        if len(indices) == 0:
            raise ValueError(
                "Each day must contain at least one interval."
            )

        first_t = int(
            indices[0]
        )

        last_t = int(
            indices[-1]
        )

        our_daily_pnl[:, d] = np.sum(
            interval_pnl[
                :,
                indices
            ],
            axis=1,
        )

        daily_place_pnl[
            :,
            :,
            d
        ] = np.sum(
            B_sims[
                :,
                :,
                indices
            ],
            axis=2,
        )

        our_beginning_balances[:, d] = (
            balance_start[
                :,
                first_t
            ]
        )

        our_ending_balances[:, d] = (
            balance_end[
                :,
                last_t
            ]
        )

    return {
        "days":
            days,

        "our_daily_pnl":
            our_daily_pnl,

        "daily_place_pnl":
            daily_place_pnl,

        "our_beginning_balances":
            our_beginning_balances,

        "our_ending_balances":
            our_ending_balances,

        "interval_pnl":
            interval_pnl,
    }

def final_prize_value_and_terminal_adjoint(
    terminal_balance,
    final_place_balance_sims,
    prizes,
    current_final_place_balances,
    current_rank=None,
    win_temperature=0.02,
):
    """
    Final-prize objective plus derivative with respect to each
    simulation's terminal balance.

    Returns:

        objective
        d_objective_d_terminal_balance
        diagnostics
    """

    A = np.asarray(
        terminal_balance,
        dtype=float,
    )

    final_place_balance_sims = np.asarray(
        final_place_balance_sims,
        dtype=float,
    )

    prizes = np.asarray(
        prizes,
        dtype=float,
    )

    current_final_place_balances = np.asarray(
        current_final_place_balances,
        dtype=float,
    )

    if A.ndim != 1:
        raise ValueError(
            "terminal_balance must have shape (N,)."
        )

    N = len(
        A
    )

    if final_place_balance_sims.ndim != 3:
        raise ValueError(
            "final_place_balance_sims must have shape "
            "(N, 6, T)."
        )

    if (
        final_place_balance_sims.shape[0] != N
        or final_place_balance_sims.shape[1] != 6
    ):
        raise ValueError(
            "final_place_balance_sims must have shape "
            "(N, 6, T)."
        )

    if prizes.shape != (
        4,
    ):
        raise ValueError(
            "prizes must contain four incremental "
            "final-prize values."
        )

    if current_final_place_balances.shape != (
        6,
    ):
        raise ValueError(
            "current_final_place_balances must have "
            "shape (6,)."
        )

    tau = float(
        win_temperature
    )

    if tau <= 0.0:
        raise ValueError(
            "win_temperature must be positive."
        )

    place_terminal_balances = (
        final_place_balance_sims[
            :,
            :,
            -1
        ]
    )

    place_pairs = [
        (
            0,
            1,
        ),
        (
            1,
            2,
        ),
        (
            2,
            3,
        ),
        (
            4,
            5,
        ),
    ]

    prize_rank_cutoffs = [
        1,
        2,
        3,
        15,
    ]

    objective = 0.0

    terminal_adjoint = np.zeros(
        N,
        dtype=float,
    )

    threshold_balances = []
    dollar_margins = []
    smooth_win_probabilities = []

    for i, (
        obtain_idx,
        maintain_idx,
    ) in enumerate(
        place_pairs
    ):
        prize_cutoff = (
            prize_rank_cutoffs[i]
        )

        if current_rank is None:
            threshold_idx = (
                obtain_idx
            )

        elif current_rank <= prize_cutoff:
            threshold_idx = (
                maintain_idx
            )

        else:
            threshold_idx = (
                obtain_idx
            )

        H = (
            place_terminal_balances[
                :,
                threshold_idx
            ]
        )

        if (
            not np.all(
                np.isfinite(
                    H
                )
            )
            or np.any(
                H <= 0.0
            )
        ):
            raise ValueError(
                "Final threshold balances must be "
                "positive and finite."
            )

        dollar_margin = (
            A
            - H
        )

        dollar_temperature = (
            tau
            * H
        )

        z = (
            dollar_margin
            / dollar_temperature
        )

        p = expit(
            z
        )

        objective += (
            prizes[i]
            * np.mean(
                p
            )
        )

        # Objective contains mean over N simulations.
        terminal_adjoint += (
            prizes[i]
            / float(
                N
            )
            * p
            * (
                1.0
                - p
            )
            / dollar_temperature
        )

        threshold_balances.append(
            H
        )

        dollar_margins.append(
            dollar_margin
        )

        smooth_win_probabilities.append(
            p
        )

    diagnostics = {
        "terminal_balance":
            A,

        "threshold_balances":
            np.stack(
                threshold_balances,
                axis=1,
            ),

        "dollar_margins":
            np.stack(
                dollar_margins,
                axis=1,
            ),

        "smooth_win_probabilities":
            np.stack(
                smooth_win_probabilities,
                axis=1,
            ),

        "expected_final_prize":
            objective,
    }

    return (
        objective,
        terminal_adjoint,
        diagnostics,
    )


def daily_prize_value_and_interval_adjoint(
    interval_pnl,
    B_sims,
    day_labels,
    daily_prize=500.0,
    competitor_temperature=1.0,
    win_temperature=1.0,
    eligible_days=None,
    daily_threshold_days=None,
    current_day=None,
    our_realized_current_day_pnl=0.0,
    current_daily_rank=None,
    current_daily_place_pnl=None,
):
    """
    Daily-prize objective plus derivative with respect to each
    simulated interval dollar P&L.

    Returns:

        objective
        d_objective_d_interval_pnl   shape (N, T)
        diagnostics
    """

    interval_pnl = np.asarray(
        interval_pnl,
        dtype=float,
    )

    B_sims = np.asarray(
        B_sims,
        dtype=float,
    )

    day_labels = np.asarray(
        day_labels
    )

    if interval_pnl.ndim != 2:
        raise ValueError(
            "interval_pnl must have shape (N, T)."
        )

    N, T = interval_pnl.shape

    if B_sims.shape != (
        N,
        2,
        T,
    ):
        raise ValueError(
            "B_sims must have shape (N, 2, T)."
        )

    if len(
        day_labels
    ) != T:
        raise ValueError(
            "day_labels must have length T."
        )

    days = list(
        dict.fromkeys(
            day_labels.tolist()
        )
    )

    D = len(
        days
    )

    our_daily_pnl = np.zeros(
        (
            N,
            D,
        ),
        dtype=float,
    )

    daily_place_pnl = np.zeros(
        (
            N,
            2,
            D,
        ),
        dtype=float,
    )

    for d, day in enumerate(
        days
    ):
        indices = np.flatnonzero(
            day_labels == day
        )

        if len(
            indices
        ) == 0:
            raise ValueError(
                "Each day must contain at least one interval."
            )

        our_daily_pnl[
            :,
            d
        ] = np.sum(
            interval_pnl[
                :,
                indices
            ],
            axis=1,
        )

        daily_place_pnl[
            :,
            :,
            d
        ] = np.sum(
            B_sims[
                :,
                :,
                indices
            ],
            axis=2,
        )

    if daily_threshold_days is None:
        if eligible_days is None:
            threshold_days = list(
                days
            )

        else:
            threshold_days = [
                day
                for day in days
                if day in eligible_days
            ]

    else:
        threshold_days = [
            day
            for day in daily_threshold_days
            if (
                day in days
                and (
                    eligible_days is None
                    or day in eligible_days
                )
            )
        ]

    interval_adjoint = np.zeros(
        (
            N,
            T,
        ),
        dtype=float,
    )

    if len(
        threshold_days
    ) == 0:
        return (
            0.0,
            interval_adjoint,
            {
                "threshold_days":
                    [],

                "probability_win_at_least_once":
                    0.0,

                "expected_daily_prize":
                    0.0,
            },
        )

    day_to_index = {
        day: i
        for i, day in enumerate(
            days
        )
    }

    day_indices = np.array(
        [
            day_to_index[
                day
            ]
            for day in threshold_days
        ],
        dtype=int,
    )

    our_selected_pnl = (
        our_daily_pnl[
            :,
            day_indices
        ].copy()
    )

    if current_day in threshold_days:
        j = threshold_days.index(
            current_day
        )

        our_selected_pnl[
            :,
            j
        ] += float(
            our_realized_current_day_pnl
        )

    daily_first_pnl = (
        daily_place_pnl[
            :,
            0,
            day_indices
        ].copy()
    )

    daily_second_pnl = (
        daily_place_pnl[
            :,
            1,
            day_indices
        ].copy()
    )

    if current_day in threshold_days:
        if current_daily_place_pnl is None:
            raise ValueError(
                "current_daily_place_pnl must be supplied "
                "for the active current day."
            )

        current_daily_place_pnl = np.asarray(
            current_daily_place_pnl,
            dtype=float,
        )

        if current_daily_place_pnl.shape != (
            2,
        ):
            raise ValueError(
                "current_daily_place_pnl must have shape (2,)."
            )

        j = threshold_days.index(
            current_day
        )

        daily_first_pnl[
            :,
            j
        ] += (
            current_daily_place_pnl[
                0
            ]
        )

        daily_second_pnl[
            :,
            j
        ] += (
            current_daily_place_pnl[
                1
            ]
        )

    tau_c = float(
        competitor_temperature
    )

    tau_w = float(
        win_temperature
    )

    if tau_c <= 0.0:
        raise ValueError(
            "competitor_temperature must be positive."
        )

    if tau_w <= 0.0:
        raise ValueError(
            "win_temperature must be positive."
        )

    rank_gate = expit(
        (
            our_selected_pnl
            - daily_first_pnl
        )
        / tau_c
    )

    place_gap = (
        daily_second_pnl
        - daily_first_pnl
    )

    selected_threshold_pnl = (
        daily_first_pnl
        + rank_gate
        * place_gap
    )

    # --------------------------------------------------------
    # dm / d(our daily P&L)
    #
    # margin
    #   = our - threshold
    #
    # threshold
    #   = first + g * (second - first)
    #
    # dg / dour
    #   = g(1-g) / tau_c
    # --------------------------------------------------------

    dmargin_dour = (
        1.0
        - place_gap
        * rank_gate
        * (
            1.0
            - rank_gate
        )
        / tau_c
    )

    # --------------------------------------------------------
    # Current observed day:
    #
    # rank 1 -> observed second
    # else   -> observed first
    #
    # The selected observed threshold is locally independent
    # of our W, so dmargin/dour = 1.
    # --------------------------------------------------------

    if current_day in threshold_days:
        current_day_column = (
            threshold_days.index(
                current_day
            )
        )

        if current_daily_rank is not None:
            if current_daily_rank == 1:
                selected_threshold_pnl[
                    :,
                    current_day_column
                ] = (
                    daily_second_pnl[
                        :,
                        current_day_column
                    ]
                )

            else:
                selected_threshold_pnl[
                    :,
                    current_day_column
                ] = (
                    daily_first_pnl[
                        :,
                        current_day_column
                    ]
                )

            dmargin_dour[
                :,
                current_day_column
            ] = 1.0

    margins = (
        our_selected_pnl
        - selected_threshold_pnl
    )

    q = expit(
        margins
        / tau_w
    )

    q_any = (
        1.0
        - np.prod(
            1.0 - q,
            axis=1,
        )
    )

    objective = (
        float(
            daily_prize
        )
        * np.mean(
            q_any
        )
    )

    D_selected = len(
        threshold_days
    )

    # --------------------------------------------------------
    # d objective / d our_selected_pnl[n,d]
    #
    # Objective:
    #
    #     daily_prize / N * sum_n q_any[n]
    #
    # q_any:
    #
    #     1 - product_d(1-q_d)
    #
    # --------------------------------------------------------

    selected_daily_adjoint = np.zeros(
        (
            N,
            D_selected,
        ),
        dtype=float,
    )

    for d in range(
        D_selected
    ):
        if D_selected == 1:
            other_product = np.ones(
                N,
                dtype=float,
            )

        else:
            other_indices = [
                j
                for j in range(
                    D_selected
                )
                if j != d
            ]

            other_product = np.prod(
                1.0
                - q[
                    :,
                    other_indices
                ],
                axis=1,
            )

        dq_dmargin = (
            q[
                :,
                d
            ]
            * (
                1.0
                - q[
                    :,
                    d
                ]
            )
            / tau_w
        )

        selected_daily_adjoint[
            :,
            d
        ] = (
            float(
                daily_prize
            )
            / float(
                N
            )
            * other_product
            * dq_dmargin
            * dmargin_dour[
                :,
                d
            ]
        )

    # --------------------------------------------------------
    # Every interval belonging to a selected day contributes
    # one-for-one to that day's simulated P&L sum.
    # --------------------------------------------------------

    for d, day in enumerate(
        threshold_days
    ):
        indices = np.flatnonzero(
            day_labels == day
        )

        interval_adjoint[
            :,
            indices
        ] = (
            selected_daily_adjoint[
                :,
                d
            ][
                :,
                None
            ]
        )

    diagnostics = {
        "threshold_days":
            threshold_days,

        "our_selected_pnl":
            our_selected_pnl,

        "daily_first_pnl":
            daily_first_pnl,

        "daily_second_pnl":
            daily_second_pnl,

        "rank_gate":
            rank_gate,

        "daily_threshold_pnl":
            selected_threshold_pnl,

        "smooth_min_margin":
            margins,

        "smooth_daily_win":
            q,

        "smooth_any_win":
            q_any,

        "probability_win_at_least_once":
            np.mean(
                q_any
            ),

        "expected_daily_prize":
            objective,
    }

    return (
        objective,
        interval_adjoint,
        diagnostics,
    )


def reverse_dollar_account_gradient(
    account,
    margin,
    reference_balance,
    terminal_balance_adjoint,
    interval_pnl_adjoint,
):
    """
    Reverse the complete account state dynamics.

    State before interval t:

        balance_t
        q_old_t

    Control:

        W_t

    Execution:

        q_new_t
            = f(
                balance_t,
                q_old_t,
                q_target_t
            )

    P&L:

        pnl_t
            = q_new_t dot pnl_per_contract_t

    Balance:

        balance_{t+1}
            = balance_t + pnl_t

    Position state:

        q_old_{t+1}
            = q_new_t
    """

    margin = np.asarray(
        margin,
        dtype=float,
    )

    terminal_balance_adjoint = np.asarray(
        terminal_balance_adjoint,
        dtype=float,
    )

    interval_pnl_adjoint = np.asarray(
        interval_pnl_adjoint,
        dtype=float,
    )

    pnl_per_contract = np.asarray(
        account[
            "pnl_per_contract"
        ],
        dtype=float,
    )

    dq_dbalance = np.asarray(
        account[
            "dq_dbalance"
        ],
        dtype=float,
    )

    dq_dq_old = np.asarray(
        account[
            "dq_dq_old"
        ],
        dtype=float,
    )

    dq_dq_target = np.asarray(
        account[
            "dq_dq_target"
        ],
        dtype=float,
    )

    N, T, K = pnl_per_contract.shape

    if margin.shape != (
        T,
        K,
    ):
        raise ValueError(
            "margin must have shape (T, K)."
        )

    if terminal_balance_adjoint.shape != (
        N,
    ):
        raise ValueError(
            "terminal_balance_adjoint must have shape (N,)."
        )

    if interval_pnl_adjoint.shape != (
        N,
        T,
    ):
        raise ValueError(
            "interval_pnl_adjoint must have shape (N, T)."
        )

    if dq_dbalance.shape != (
        N,
        T,
        K,
    ):
        raise ValueError(
            "dq_dbalance has incompatible shape."
        )

    if dq_dq_old.shape != (
        N,
        T,
        K,
        K,
    ):
        raise ValueError(
            "dq_dq_old has incompatible shape."
        )

    if dq_dq_target.shape != (
        N,
        T,
        K,
        K,
    ):
        raise ValueError(
            "dq_dq_target has incompatible shape."
        )

    reference_balance = float(
        reference_balance
    )

    gradient = np.zeros(
        (
            T,
            K,
        ),
        dtype=float,
    )

    # --------------------------------------------------------
    # Adjoint of state AFTER the current reverse step.
    #
    # lambda_balance corresponds to balance_{t+1}.
    #
    # lambda_q corresponds to q_old_{t+1}, which is q_new_t.
    # --------------------------------------------------------

    lambda_balance = (
        terminal_balance_adjoint.copy()
    )

    lambda_q = np.zeros(
        (
            N,
            K,
        ),
        dtype=float,
    )

    for t in range(
        T - 1,
        -1,
        -1,
    ):
        # ----------------------------------------------------
        # pnl_t affects:
        #
        # 1. the daily-prize objective directly;
        # 2. balance_{t+1}, whose adjoint is lambda_balance.
        # ----------------------------------------------------

        lambda_pnl = (
            interval_pnl_adjoint[
                :,
                t
            ]
            + lambda_balance
        )

        # ----------------------------------------------------
        # q_new_t affects:
        #
        # 1. pnl_t through pnl_per_contract;
        # 2. next interval's q_old state.
        # ----------------------------------------------------

        lambda_q_new = (
            lambda_q
            + lambda_pnl[
                :,
                None
            ]
            * pnl_per_contract[
                :,
                t,
                :
            ]
        )

        # ----------------------------------------------------
        # q_target_t adjoint:
        #
        # C^T * lambda_q_new
        # ----------------------------------------------------

        lambda_q_target = np.einsum(
            "nki,nk->ni",
            dq_dq_target[
                :,
                t,
                :,
                :
            ],
            lambda_q_new,
        )

        # q_target_t,k
        #     = W_t,k * reference_balance / margin_t,k

        gradient[
            t,
            :
        ] = np.sum(
            lambda_q_target,
            axis=0,
        ) * (
            reference_balance
            / margin[
                t,
                :
            ]
        )

        # ----------------------------------------------------
        # Previous balance adjoint:
        #
        # balance_{t+1}
        #   = balance_t + pnl_t
        #
        # plus q_new's dependence on balance_t.
        # ----------------------------------------------------

        lambda_balance_previous = (
            lambda_balance
            + np.einsum(
                "nk,nk->n",
                dq_dbalance[
                    :,
                    t,
                    :
                ],
                lambda_q_new,
            )
        )

        # ----------------------------------------------------
        # Previous q_old adjoint:
        #
        # B^T * lambda_q_new
        # ----------------------------------------------------

        lambda_q_previous = np.einsum(
            "nki,nk->ni",
            dq_dq_old[
                :,
                t,
                :,
                :
            ],
            lambda_q_new,
        )

        lambda_balance = (
            lambda_balance_previous
        )

        lambda_q = (
            lambda_q_previous
        )

    return gradient

def batched_analytic_local_execution_jacobians(
    balance,
    q_old,
    q_target,
    margin_per_contract,
    tol=1e-12,
):
    """
    Batched equivalent of:

        analytic_local_execution_jacobians()

    Inputs
    ------
    balance:
        shape (N,)

    q_old:
        shape (N, K)

    q_target:
        shape (K,)

    margin_per_contract:
        shape (K,)

    Returns
    -------
    q_executed:
        shape (N, K)

    dq_dbalance:
        shape (N, K)

    dq_dq_old:
        shape (N, K, K)

    dq_dq_target:
        shape (N, K, K)
    """

    balance = np.asarray(
        balance,
        dtype=float,
    )

    q_old = np.asarray(
        q_old,
        dtype=float,
    )

    q_target = np.asarray(
        q_target,
        dtype=float,
    )

    margin_per_contract = np.asarray(
        margin_per_contract,
        dtype=float,
    )

    if balance.ndim != 1:
        raise ValueError(
            "balance must have shape (N,)."
        )

    if q_old.ndim != 2:
        raise ValueError(
            "q_old must have shape (N, K)."
        )

    N, K = q_old.shape

    if balance.shape != (
        N,
    ):
        raise ValueError(
            "balance must have shape (N,)."
        )

    if q_target.shape != (
        K,
    ):
        raise ValueError(
            "q_target must have shape (K,)."
        )

    if margin_per_contract.shape != (
        K,
    ):
        raise ValueError(
            "margin_per_contract must have shape (K,)."
        )

    if not np.all(
        np.isfinite(
            balance
        )
    ):
        raise ValueError(
            "balance must be finite."
        )

    if not np.all(
        np.isfinite(
            q_old
        )
    ):
        raise ValueError(
            "q_old must be finite."
        )

    if not np.all(
        np.isfinite(
            q_target
        )
    ):
        raise ValueError(
            "q_target must be finite."
        )

    if not np.all(
        np.isfinite(
            margin_per_contract
        )
    ):
        raise ValueError(
            "margin_per_contract must be finite."
        )

    if np.any(
        margin_per_contract <= 0.0
    ):
        raise ValueError(
            "margin_per_contract must be positive."
        )

    # --------------------------------------------------------
    # Common arrays.
    # --------------------------------------------------------

    normal_capacity = np.maximum(
        balance,
        0.0,
    )

    d_capacity_dbalance = (
        balance > 0.0
    ).astype(
        float
    )

    old_margin = np.sum(
        np.abs(
            q_old
        )
        * margin_per_contract[
            None,
            :
        ],
        axis=1,
    )

    target_margin = float(
        np.sum(
            np.abs(
                q_target
            )
            * margin_per_contract
        )
    )

    # --------------------------------------------------------
    # Build q_base and additions for every path.
    #
    # This is the batched equivalent of the frozen
    # build_base_and_additions() helper.
    # --------------------------------------------------------

    q_base = np.zeros(
        (
            N,
            K,
        ),
        dtype=float,
    )

    additions = np.zeros(
        (
            N,
            K,
        ),
        dtype=float,
    )

    db_base = np.zeros(
        (
            N,
            K,
        ),
        dtype=float,
    )

    dold_base = np.zeros(
        (
            N,
            K,
            K,
        ),
        dtype=float,
    )

    dtarget_base = np.zeros(
        (
            N,
            K,
            K,
        ),
        dtype=float,
    )

    db_add = np.zeros(
        (
            N,
            K,
        ),
        dtype=float,
    )

    dold_add = np.zeros(
        (
            N,
            K,
            K,
        ),
        dtype=float,
    )

    dtarget_add = np.zeros(
        (
            N,
            K,
            K,
        ),
        dtype=float,
    )

    target_matrix = np.broadcast_to(
        q_target[
            None,
            :
        ],
        (
            N,
            K,
        ),
    )

    old_zero = (
        np.abs(
            q_old
        )
        <= tol
    )

    target_zero = (
        np.abs(
            target_matrix
        )
        <= tol
    )

    same_side = (
        q_old
        * target_matrix
        > 0.0
    )

    same_side_reduction = (
        (~old_zero)
        & (~target_zero)
        & same_side
        & (
            np.abs(
                target_matrix
            )
            <= np.abs(
                q_old
            )
        )
    )

    same_side_increase = (
        (~old_zero)
        & (~target_zero)
        & same_side
        & (
            np.abs(
                target_matrix
            )
            > np.abs(
                q_old
            )
        )
    )

    reversal = (
        (~old_zero)
        & (~target_zero)
        & (~same_side)
    )

    # --------------------------------------------------------
    # No existing position:
    #
    # base = 0
    # additions = target
    # --------------------------------------------------------

    additions[
        old_zero
    ] = target_matrix[
        old_zero
    ]

    # --------------------------------------------------------
    # Same-side reduction:
    #
    # base = target
    # additions = 0
    # --------------------------------------------------------

    q_base[
        same_side_reduction
    ] = target_matrix[
        same_side_reduction
    ]

    # --------------------------------------------------------
    # Same-side increase:
    #
    # base = old
    # additions = target - old
    # --------------------------------------------------------

    q_base[
        same_side_increase
    ] = q_old[
        same_side_increase
    ]

    additions[
        same_side_increase
    ] = (
        target_matrix[
            same_side_increase
        ]
        - q_old[
            same_side_increase
        ]
    )

    # --------------------------------------------------------
    # Reversal:
    #
    # base = 0
    # additions = target
    # --------------------------------------------------------

    additions[
        reversal
    ] = target_matrix[
        reversal
    ]

    # --------------------------------------------------------
    # Diagonal derivative entries.
    #
    # The component rules are separable before the common
    # margin-scaling step, so these local derivative matrices
    # are diagonal here.
    # --------------------------------------------------------

    diag = np.arange(
        K
    )

    dtarget_base[
        :,
        diag,
        diag
    ] = same_side_reduction.astype(
        float
    )

    dold_base[
        :,
        diag,
        diag
    ] = same_side_increase.astype(
        float
    )

    dold_add[
        :,
        diag,
        diag
    ] = -same_side_increase.astype(
        float
    )

    dtarget_add[
        :,
        diag,
        diag
    ] = (
        old_zero
        | same_side_increase
        | reversal
    ).astype(
        float
    )

    # --------------------------------------------------------
    # Margin derivative helper.
    # --------------------------------------------------------

    def batched_margin_derivatives(
        q,
        db_q,
        dold_q,
        dtarget_q,
    ):
        margin_gradient_q = (
            np.sign(
                q
            )
            * margin_per_contract[
                None,
                :
            ]
        )

        db_margin = np.einsum(
            "ni,ni->n",
            margin_gradient_q,
            db_q,
        )

        dold_margin = np.einsum(
            "ni,nij->nj",
            margin_gradient_q,
            dold_q,
        )

        dtarget_margin = np.einsum(
            "ni,nij->nj",
            margin_gradient_q,
            dtarget_q,
        )

        return (
            db_margin,
            dold_margin,
            dtarget_margin,
        )

    # --------------------------------------------------------
    # Output arrays.
    # --------------------------------------------------------

    q_executed = np.zeros(
        (
            N,
            K,
        ),
        dtype=float,
    )

    dq_dbalance = np.zeros(
        (
            N,
            K,
        ),
        dtype=float,
    )

    dq_dq_old = np.zeros(
        (
            N,
            K,
            K,
        ),
        dtype=float,
    )

    dq_dq_target = np.zeros(
        (
            N,
            K,
            K,
        ),
        dtype=float,
    )

    assigned = np.zeros(
        N,
        dtype=bool,
    )

    # --------------------------------------------------------
    # Assignment helper for q_base-only branches.
    # --------------------------------------------------------

    def assign_base(mask):
        if not np.any(
            mask
        ):
            return

        q_executed[
            mask
        ] = q_base[
            mask
        ]

        dq_dbalance[
            mask
        ] = db_base[
            mask
        ]

        dq_dq_old[
            mask
        ] = dold_base[
            mask
        ]

        dq_dq_target[
            mask
        ] = dtarget_base[
            mask
        ]

        assigned[
            mask
        ] = True

    # --------------------------------------------------------
    # Assignment helper for:
    #
    # q_new = q_base + scale * additions
    # --------------------------------------------------------

    def assign_scaled(
        mask,
        scale,
        db_scale,
        dold_scale,
        dtarget_scale,
    ):
        if not np.any(
            mask
        ):
            return

        add_m = additions[
            mask
        ]

        scale_m = scale[
            mask
        ]

        q_executed[
            mask
        ] = (
            q_base[
                mask
            ]
            + scale_m[
                :,
                None
            ]
            * add_m
        )

        dq_dbalance[
            mask
        ] = (
            db_base[
                mask
            ]
            + scale_m[
                :,
                None
            ]
            * db_add[
                mask
            ]
            + add_m
            * db_scale[
                mask
            ][
                :,
                None
            ]
        )

        dq_dq_old[
            mask
        ] = (
            dold_base[
                mask
            ]
            + scale_m[
                :,
                None,
                None
            ]
            * dold_add[
                mask
            ]
            + add_m[
                :,
                :,
                None
            ]
            * dold_scale[
                mask
            ][
                :,
                None,
                :
            ]
        )

        dq_dq_target[
            mask
        ] = (
            dtarget_base[
                mask
            ]
            + scale_m[
                :,
                None,
                None
            ]
            * dtarget_add[
                mask
            ]
            + add_m[
                :,
                :,
                None
            ]
            * dtarget_scale[
                mask
            ][
                :,
                None,
                :
            ]
        )

        assigned[
            mask
        ] = True

    # ========================================================
    # CASE 1
    # Account starts over-margin.
    # ========================================================

    over_margin = (
        old_margin
        > normal_capacity + tol
    )

    reduced_margin = np.sum(
        np.abs(
            q_base
        )
        * margin_per_contract[
            None,
            :
        ],
        axis=1,
    )

    desired_new_margin = np.sum(
        np.abs(
            additions
        )
        * margin_per_contract[
            None,
            :
        ],
        axis=1,
    )

    # --------------------------------------------------------
    # Still over-margin after requested reductions.
    # --------------------------------------------------------

    mask = (
        over_margin
        & (
            reduced_margin
            > normal_capacity + tol
        )
    )

    assign_base(
        mask
    )

    # --------------------------------------------------------
    # Reductions restored normal capacity.
    # --------------------------------------------------------

    restored = (
        over_margin
        & (~assigned)
    )

    available = np.maximum(
        normal_capacity
        - reduced_margin,
        0.0,
    )

    # No requested new exposure.
    mask = (
        restored
        & (
            desired_new_margin
            <= tol
        )
    )

    assign_base(
        mask
    )

    # All requested new exposure fits.
    mask = (
        restored
        & (~assigned)
        & (
            desired_new_margin
            <= available + tol
        )
    )

    if np.any(
        mask
    ):
        scale = np.ones(
            N,
            dtype=float,
        )

        db_scale = np.zeros(
            N,
            dtype=float,
        )

        dold_scale = np.zeros(
            (
                N,
                K,
            ),
            dtype=float,
        )

        dtarget_scale = np.zeros(
            (
                N,
                K,
            ),
            dtype=float,
        )

        assign_scaled(
            mask=mask,
            scale=scale,
            db_scale=db_scale,
            dold_scale=dold_scale,
            dtarget_scale=dtarget_scale,
        )

    # --------------------------------------------------------
    # Requested new exposure must be clipped.
    # --------------------------------------------------------

    mask = (
        restored
        & (~assigned)
    )

    if np.any(
        mask
    ):
        (
            db_reduced_margin,
            dold_reduced_margin,
            dtarget_reduced_margin,
        ) = batched_margin_derivatives(
            q=q_base,
            db_q=db_base,
            dold_q=dold_base,
            dtarget_q=dtarget_base,
        )

        (
            db_desired_margin,
            dold_desired_margin,
            dtarget_desired_margin,
        ) = batched_margin_derivatives(
            q=additions,
            db_q=db_add,
            dold_q=dold_add,
            dtarget_q=dtarget_add,
        )

        db_available = (
            d_capacity_dbalance
            - db_reduced_margin
        )

        dold_available = (
            -dold_reduced_margin
        )

        dtarget_available = (
            -dtarget_reduced_margin
        )

        denominator = (
            desired_new_margin ** 2
        )

        safe_denominator = np.where(
            desired_new_margin > tol,
            denominator,
            1.0,
        )

        scale = (
            available
            / np.where(
                desired_new_margin > tol,
                desired_new_margin,
                1.0,
            )
        )

        db_scale = (
            (
                db_available
                * desired_new_margin
                - available
                * db_desired_margin
            )
            / safe_denominator
        )

        dold_scale = (
            (
                dold_available
                * desired_new_margin[
                    :,
                    None
                ]
                - available[
                    :,
                    None
                ]
                * dold_desired_margin
            )
            / safe_denominator[
                :,
                None
            ]
        )

        dtarget_scale = (
            (
                dtarget_available
                * desired_new_margin[
                    :,
                    None
                ]
                - available[
                    :,
                    None
                ]
                * dtarget_desired_margin
            )
            / safe_denominator[
                :,
                None
            ]
        )

        assign_scaled(
            mask=mask,
            scale=scale,
            db_scale=db_scale,
            dold_scale=dold_scale,
            dtarget_scale=dtarget_scale,
        )

    # ========================================================
    # CASE 2
    # Account starts within normal capacity.
    # ========================================================

    normal_start = (
        ~over_margin
    )

    # --------------------------------------------------------
    # Entire target is affordable.
    #
    # Exact frozen branch:
    #
    # q_executed = q_target
    # dq/dbalance = 0
    # dq/dq_old = 0
    # dq/dq_target = I
    # --------------------------------------------------------

    mask = (
        normal_start
        & (
            target_margin
            <= normal_capacity + tol
        )
    )

    if np.any(
        mask
    ):
        q_executed[
            mask
        ] = q_target[
            None,
            :
        ]

        dq_dbalance[
            mask
        ] = 0.0

        dq_dq_old[
            mask
        ] = 0.0

        dq_dq_target[
            mask
        ] = np.eye(
            K,
            dtype=float,
        )[
            None,
            :,
            :
        ]

        assigned[
            mask
        ] = True

    # --------------------------------------------------------
    # Target not fully affordable.
    # --------------------------------------------------------

    clipped_normal = (
        normal_start
        & (~assigned)
    )

    # No additions after componentwise reduction.
    mask = (
        clipped_normal
        & (
            desired_new_margin
            <= tol
        )
    )

    assign_base(
        mask
    )

    # --------------------------------------------------------
    # Remaining normal-start paths are clipped additions.
    # --------------------------------------------------------

    mask = (
        clipped_normal
        & (~assigned)
    )

    if np.any(
        mask
    ):
        (
            db_base_margin,
            dold_base_margin,
            dtarget_base_margin,
        ) = batched_margin_derivatives(
            q=q_base,
            db_q=db_base,
            dold_q=dold_base,
            dtarget_q=dtarget_base,
        )

        (
            db_desired_margin,
            dold_desired_margin,
            dtarget_desired_margin,
        ) = batched_margin_derivatives(
            q=additions,
            db_q=db_add,
            dold_q=dold_add,
            dtarget_q=dtarget_add,
        )

        base_margin = reduced_margin

        available_normal = np.maximum(
            normal_capacity
            - base_margin,
            0.0,
        )

        db_available = (
            d_capacity_dbalance
            - db_base_margin
        )

        dold_available = (
            -dold_base_margin
        )

        dtarget_available = (
            -dtarget_base_margin
        )

        denominator = (
            desired_new_margin ** 2
        )

        safe_denominator = np.where(
            desired_new_margin > tol,
            denominator,
            1.0,
        )

        scale = (
            available_normal
            / np.where(
                desired_new_margin > tol,
                desired_new_margin,
                1.0,
            )
        )

        db_scale = (
            (
                db_available
                * desired_new_margin
                - available_normal
                * db_desired_margin
            )
            / safe_denominator
        )

        dold_scale = (
            (
                dold_available
                * desired_new_margin[
                    :,
                    None
                ]
                - available_normal[
                    :,
                    None
                ]
                * dold_desired_margin
            )
            / safe_denominator[
                :,
                None
            ]
        )

        dtarget_scale = (
            (
                dtarget_available
                * desired_new_margin[
                    :,
                    None
                ]
                - available_normal[
                    :,
                    None
                ]
                * dtarget_desired_margin
            )
            / safe_denominator[
                :,
                None
            ]
        )

        assign_scaled(
            mask=mask,
            scale=scale,
            db_scale=db_scale,
            dold_scale=dold_scale,
            dtarget_scale=dtarget_scale,
        )

    if not np.all(
        assigned
    ):
        bad = np.flatnonzero(
            ~assigned
        )

        raise RuntimeError(
            "Batched execution failed to assign all paths. "
            f"Unassigned indices: {bad[:10]}"
        )

    return {
        "q_executed":
            q_executed,

        "dq_dbalance":
            dq_dbalance,

        "dq_dq_old":
            dq_dq_old,

        "dq_dq_target":
            dq_dq_target,
    }


def simulate_dollar_balance_account_with_batched_reverse_tape(
    W,
    raw_return_sims,
    pathwise_notional,
    margin,
    starting_balance,
    initial_contracts,
    reference_balance=None,
):
    """
    Batched Stage 5B.4D.2 forward pass.

    The loop over time remains because account balance and
    carried contracts are sequential state variables.

    There is NO Python loop over simulations.
    """

    W = np.asarray(
        W,
        dtype=float,
    )

    raw_return_sims = np.asarray(
        raw_return_sims,
        dtype=float,
    )

    pathwise_notional = np.asarray(
        pathwise_notional,
        dtype=float,
    )

    margin = np.asarray(
        margin,
        dtype=float,
    )

    initial_contracts = np.asarray(
        initial_contracts,
        dtype=float,
    )

    if W.ndim != 2:
        raise ValueError(
            "W must have shape (T, K)."
        )

    T, K = W.shape

    if raw_return_sims.ndim != 3:
        raise ValueError(
            "raw_return_sims must have shape (N, T, K)."
        )

    N = raw_return_sims.shape[0]

    if raw_return_sims.shape != (
        N,
        T,
        K,
    ):
        raise ValueError(
            "raw_return_sims has incompatible shape."
        )

    if pathwise_notional.shape != (
        N,
        T,
        K,
    ):
        raise ValueError(
            "pathwise_notional has incompatible shape."
        )

    if margin.shape != (
        T,
        K,
    ):
        raise ValueError(
            "margin must have shape (T, K)."
        )

    if initial_contracts.shape != (
        K,
    ):
        raise ValueError(
            "initial_contracts must have shape (K,)."
        )

    if not np.all(
        np.isfinite(
            W
        )
    ):
        raise ValueError(
            "W must be finite."
        )

    if not np.all(
        np.isfinite(
            raw_return_sims
        )
    ):
        raise ValueError(
            "raw_return_sims must be finite."
        )

    if not np.all(
        np.isfinite(
            pathwise_notional
        )
    ):
        raise ValueError(
            "pathwise_notional must be finite."
        )

    if not np.all(
        np.isfinite(
            margin
        )
    ):
        raise ValueError(
            "margin must be finite."
        )

    if np.any(
        margin <= 0.0
    ):
        raise ValueError(
            "margin must be positive."
        )

    if not np.all(
        np.isfinite(
            initial_contracts
        )
    ):
        raise ValueError(
            "initial_contracts must be finite."
        )

    starting_balance = float(
        starting_balance
    )

    if not np.isfinite(
        starting_balance
    ):
        raise ValueError(
            "starting_balance must be finite."
        )

    if reference_balance is None:
        reference_balance = (
            starting_balance
        )

    reference_balance = float(
        reference_balance
    )

    if not np.isfinite(
        reference_balance
    ):
        raise ValueError(
            "reference_balance must be finite."
        )

    target_contracts = (
        W
        * reference_balance
        / margin
    )

    balance_start = np.zeros(
        (
            N,
            T,
        ),
        dtype=float,
    )

    balance_end = np.zeros(
        (
            N,
            T,
        ),
        dtype=float,
    )

    interval_pnl = np.zeros(
        (
            N,
            T,
        ),
        dtype=float,
    )

    executed_contracts = np.zeros(
        (
            N,
            T,
            K,
        ),
        dtype=float,
    )

    pnl_per_contract = (
        pathwise_notional
        * raw_return_sims
    )

    dq_dbalance = np.zeros(
        (
            N,
            T,
            K,
        ),
        dtype=float,
    )

    dq_dq_old = np.zeros(
        (
            N,
            T,
            K,
            K,
        ),
        dtype=float,
    )

    dq_dq_target = np.zeros(
        (
            N,
            T,
            K,
            K,
        ),
        dtype=float,
    )

    balance_now = np.full(
        N,
        starting_balance,
        dtype=float,
    )

    q_old = np.broadcast_to(
        initial_contracts[
            None,
            :
        ],
        (
            N,
            K,
        ),
    ).copy()

    for t in range(
        T
    ):
        balance_start[
            :,
            t
        ] = balance_now

        local = (
            batched_analytic_local_execution_jacobians(
                balance=
                    balance_now,
                q_old=
                    q_old,
                q_target=
                    target_contracts[
                        t,
                        :
                    ],
                margin_per_contract=
                    margin[
                        t,
                        :
                    ],
            )
        )

        q_new = local[
            "q_executed"
        ]

        executed_contracts[
            :,
            t,
            :
        ] = q_new

        dq_dbalance[
            :,
            t,
            :
        ] = local[
            "dq_dbalance"
        ]

        dq_dq_old[
            :,
            t,
            :,
            :
        ] = local[
            "dq_dq_old"
        ]

        dq_dq_target[
            :,
            t,
            :,
            :
        ] = local[
            "dq_dq_target"
        ]

        interval_pnl[
            :,
            t
        ] = np.sum(
            q_new
            * pnl_per_contract[
                :,
                t,
                :
            ],
            axis=1,
        )

        balance_now = (
            balance_now
            + interval_pnl[
                :,
                t
            ]
        )

        balance_end[
            :,
            t
        ] = balance_now

        q_old = q_new

    return {
        "balance_start":
            balance_start,

        "balance_end":
            balance_end,

        "interval_pnl":
            interval_pnl,

        "target_contracts":
            target_contracts,

        "executed_contracts":
            executed_contracts,

        "terminal_balance":
            balance_end[
                :,
                -1
            ],

        "pnl_per_contract":
            pnl_per_contract,

        "dq_dbalance":
            dq_dbalance,

        "dq_dq_old":
            dq_dq_old,

        "dq_dq_target":
            dq_dq_target,
    }

def simulate_dollar_balance_account_value_only_batched(
    W,
    raw_return_sims,
    pathwise_notional,
    margin,
    starting_balance,
    initial_contracts,
    reference_balance=None,
):
    """
    Value-only batched account simulation.

    Same execution/account dynamics as the production
    gradient-capable simulator, but does not construct or
    store execution Jacobians.
    """

    W = np.asarray(W, dtype=float)
    raw_return_sims = np.asarray(
        raw_return_sims,
        dtype=float,
    )
    pathwise_notional = np.asarray(
        pathwise_notional,
        dtype=float,
    )
    margin = np.asarray(
        margin,
        dtype=float,
    )
    initial_contracts = np.asarray(
        initial_contracts,
        dtype=float,
    )

    T, K_local = W.shape
    N = raw_return_sims.shape[0]

    if reference_balance is None:
        reference_balance = float(
            starting_balance
        )

    reference_balance = float(
        reference_balance
    )

    target_contracts = (
        W
        * reference_balance
        / margin
    )

    balance = np.full(
        N,
        float(starting_balance),
        dtype=float,
    )

    q_old = np.broadcast_to(
        initial_contracts,
        (N, K_local),
    ).copy()

    interval_pnl = np.empty(
        (N, T),
        dtype=float,
    )

    for t in range(T):

        q_target_t = target_contracts[
            t,
            :
        ][
            None,
            :
        ]

        q_new = (
            execute_target_contracts_batched_value_only(
                balance=balance[
                    None,
                    :
                ],
                q_old=q_old[
                    None,
                    :,
                    :
                ],
                q_target=q_target_t,
                margin_per_contract=margin[
                    t,
                    :
                ],
                tol=1e-12,
            )[
                0,
                :,
                :
            ]
        )

        pnl_t = np.sum(
            q_new
            * pathwise_notional[
                :,
                t,
                :
            ]
            * raw_return_sims[
                :,
                t,
                :
            ],
            axis=1,
        )

        interval_pnl[
            :,
            t
        ] = pnl_t

        balance = (
            balance
            + pnl_t
        )

        q_old = q_new

    return {
        "terminal_balance":
            balance,

        "interval_pnl":
            interval_pnl,

        "final_contracts":
            q_old,
    }


def total_expected_prize_value_only_from_dollar_account(
    W,
    raw_return_sims,
    pathwise_notional,
    margin,
    initial_contracts,
    B_sims,
    final_place_balance_sims,
    day_labels,
    prizes,
    starting_balance,
    current_final_place_balances,
    current_rank=None,
    daily_prize=500.0,
    final_win_temperature=0.02,
    daily_competitor_temperature=1.0,
    daily_win_temperature=1.0,
    eligible_days=None,
    daily_threshold_days=None,
    current_day=None,
    our_realized_current_day_pnl=0.0,
    current_daily_rank=None,
    current_daily_place_pnl=None,
    daily_prize_already_won=False,
    reference_balance=None,
):
    """
    Objective-only version used by line search.

    Computes the same prize objective without constructing
    execution Jacobians or running the reverse gradient.
    """

    if reference_balance is None:
        reference_balance = float(
            starting_balance
        )

    account = (
        simulate_dollar_balance_account_value_only_batched(
            W=W,
            raw_return_sims=
                raw_return_sims,
            pathwise_notional=
                pathwise_notional,
            margin=margin,
            starting_balance=
                starting_balance,
            initial_contracts=
                initial_contracts,
            reference_balance=
                reference_balance,
        )
    )

    (
        final_objective,
        _,
        _,
    ) = (
        final_prize_value_and_terminal_adjoint(
            terminal_balance=
                account[
                    "terminal_balance"
                ],
            final_place_balance_sims=
                final_place_balance_sims,
            prizes=prizes,
            current_final_place_balances=
                current_final_place_balances,
            current_rank=
                current_rank,
            win_temperature=
                final_win_temperature,
        )
    )

    no_daily_days_remaining = (
        daily_threshold_days is not None
        and len(
            daily_threshold_days
        ) == 0
    )

    if (
        daily_prize_already_won
        or no_daily_days_remaining
    ):
        daily_objective = 0.0

    else:
        (
            daily_objective,
            _,
            _,
        ) = (
            daily_prize_value_and_interval_adjoint(
                interval_pnl=
                    account[
                        "interval_pnl"
                    ],
                B_sims=B_sims,
                day_labels=
                    day_labels,
                daily_prize=
                    daily_prize,
                competitor_temperature=
                    daily_competitor_temperature,
                win_temperature=
                    daily_win_temperature,
                eligible_days=
                    eligible_days,
                daily_threshold_days=
                    daily_threshold_days,
                current_day=
                    current_day,
                our_realized_current_day_pnl=
                    our_realized_current_day_pnl,
                current_daily_rank=
                    current_daily_rank,
                current_daily_place_pnl=
                    current_daily_place_pnl,
            )
        )

    return float(
        final_objective
        + daily_objective
    )

def total_expected_prize_and_batched_reverse_gradient_from_dollar_account(
    W,
    raw_return_sims,
    pathwise_notional,
    margin,
    initial_contracts,
    B_sims,
    final_place_balance_sims,
    day_labels,
    prizes,
    starting_balance,
    current_final_place_balances,
    current_rank=None,
    daily_prize=500.0,
    final_win_temperature=0.02,
    daily_competitor_temperature=1.0,
    daily_win_temperature=1.0,
    eligible_days=None,
    daily_threshold_days=None,
    current_day=None,
    our_realized_current_day_pnl=0.0,
    current_daily_rank=None,
    current_daily_place_pnl=None,
    daily_prize_already_won=False,
    reference_balance=None,
):
    """
    Complete Stage 5B.4D.3 objective.

    Same reverse pass as frozen 5B.4D.2.
    Only the forward execution/Jacobian tape is batched.
    """

    W = np.asarray(
        W,
        dtype=float,
    )

    if reference_balance is None:
        reference_balance = float(
            starting_balance
        )

    reference_balance = float(
        reference_balance
    )

    account = (
        simulate_dollar_balance_account_with_batched_reverse_tape(
            W=W,
            raw_return_sims=
                raw_return_sims,
            pathwise_notional=
                pathwise_notional,
            margin=margin,
            starting_balance=
                starting_balance,
            initial_contracts=
                initial_contracts,
            reference_balance=
                reference_balance,
        )
    )

    (
        final_objective,
        terminal_balance_adjoint,
        final_diagnostics,
    ) = (
        final_prize_value_and_terminal_adjoint(
            terminal_balance=
                account[
                    "terminal_balance"
                ],
            final_place_balance_sims=
                final_place_balance_sims,
            prizes=prizes,
            current_final_place_balances=
                current_final_place_balances,
            current_rank=
                current_rank,
            win_temperature=
                final_win_temperature,
        )
    )

    no_daily_days_remaining = (
        daily_threshold_days is not None
        and len(
            daily_threshold_days
        ) == 0
    )

    if (
        daily_prize_already_won
        or no_daily_days_remaining
    ):
        daily_objective = 0.0

        interval_pnl_adjoint = np.zeros_like(
            account[
                "interval_pnl"
            ],
            dtype=float,
        )

        daily_diagnostics = {
            "threshold_days":
                [],

            "probability_win_at_least_once":
                0.0,

            "expected_daily_prize":
                0.0,
        }

    else:
        (
            daily_objective,
            interval_pnl_adjoint,
            daily_diagnostics,
        ) = (
            daily_prize_value_and_interval_adjoint(
                interval_pnl=
                    account[
                        "interval_pnl"
                    ],
                B_sims=B_sims,
                day_labels=
                    day_labels,
                daily_prize=
                    daily_prize,
                competitor_temperature=
                    daily_competitor_temperature,
                win_temperature=
                    daily_win_temperature,
                eligible_days=
                    eligible_days,
                daily_threshold_days=
                    daily_threshold_days,
                current_day=
                    current_day,
                our_realized_current_day_pnl=
                    our_realized_current_day_pnl,
                current_daily_rank=
                    current_daily_rank,
                current_daily_place_pnl=
                    current_daily_place_pnl,
            )
        )

    gradient = (
        reverse_dollar_account_gradient(
            account=account,
            margin=margin,
            reference_balance=
                reference_balance,
            terminal_balance_adjoint=
                terminal_balance_adjoint,
            interval_pnl_adjoint=
                interval_pnl_adjoint,
        )
    )

    total_objective = (
        final_objective
        + daily_objective
    )

    diagnostics = {
        "account":
            account,

        "final":
            final_diagnostics,

        "daily":
            daily_diagnostics,

        "final_expected_prize":
            final_objective,

        "daily_expected_prize":
            daily_objective,

        "total_expected_prize":
            total_objective,
    }

    return (
        total_objective,
        gradient,
        diagnostics,
    )

def run_single_problem(
    T,
    K,
    prizes,
    n_starts,
    weight_bound,
    seed,
    maxiter,
    day_labels,
    standard_equivalent_contract_limit=None,
    standard_contract_display_factor=None,
    n_sims=10000,
    simulation_model="distributional",
    distributional_innovation_system=None,
    distributional_mapping_system=None,
    starting_balance=25000.0,
    daily_prize=500.0,
    n_daily_competitors=None,
    daily_threshold_days=("Monday", "Tuesday", "Wednesday", "Thursday"),
    daily_temperature_schedule=(50.0, 25.0, 10.0, 5.0),
    final_rank_temperature=250.0,
    final_win_temperature=0.02,
    eligible_days=None,
    initial_step=1.0,
    weight_tolerance=1e-6,
    objective_tolerance=1e-5,

    # Current competition state
    current_t=0,
    current_balance=None,
    optimization_horizon=None,
    
    # Our current overall leaderboard rank.
    # None is allowed only at t=0.
    current_rank=None,
    
    # Current carried futures margin weights:
    # [Future_1, Future_2, ..., Future_K]
    current_weights=None,
    
    # Current contract notionals at the beginning of the
    # optimization horizon:
    # [Future_1, Future_2, ..., Future_K]
    #
    # These are dollar notionals per contract, not portfolio
    # dollar exposures.
    current_contract_notional=None,
    
    # Overall leaderboard balances:
    # [1st, 2nd, 3rd, 4th, 15th, 16th]
    current_final_place_balances=None,

    # Our realized P&L so far in the current competition day
    our_current_daily_pnl=None,

    # Our current rank on today's daily leaderboard.
    # Relevant only during an active daily-prize day.
    current_daily_rank=None,

    # Current daily leaderboard P&Ls:
    # [daily 1st-place P&L, daily 2nd-place P&L]
    current_daily_place_pnl=None,

    # Whether we have already won the one allowed daily prize
    daily_prize_already_won=False,
):
    """Optimize total prize with separate final thresholds and day-specific daily hurdles."""
    C_final_objectives = len(prizes)   # 4: 1st, 2nd, 3rd, top 15
    C_final_places = 6                 # 1st, 2nd, 3rd, 4th, 15th, 16th
    
    if C_final_objectives != 4:
        raise ValueError(
            "This implementation expects four final prize objectives: "
            "1st, top 2, top 3, and top 15."
        )

    # ============================================================
    # CURRENT CONTRACT NOTIONAL
    #
    # This is the notional of one contract for each future at
    # the beginning of the current optimization horizon.
    # ============================================================
    
    if current_contract_notional is None:
    
        raise ValueError(
            "current_contract_notional must be supplied."
        )
    
    
    current_contract_notional = np.asarray(
        current_contract_notional,
        dtype=float,
    )
    
    
    if current_contract_notional.shape != (K,):
    
        raise ValueError(
            "current_contract_notional must have shape "
            f"({K},)."
        )
    
    
    if not np.all(
        np.isfinite(
            current_contract_notional
        )
    ):
    
        raise ValueError(
            "current_contract_notional must contain "
            "only finite values."
        )
    
    
    if np.any(
        current_contract_notional <= 0.0
    ):
    
        raise ValueError(
            "All current contract notionals "
            "must be positive."
        )

    # ============================================================
    # STEP 7:
    # DAILY PRIZE STATE
    # ============================================================

    if not isinstance(
        daily_prize_already_won,
        (bool, np.bool_)
    ):
        raise TypeError(
            "daily_prize_already_won must be boolean."
        )

    # ============================================================
    # CURRENT-T / REMAINING-HORIZON SETUP
    # ============================================================
    
    if not isinstance(current_t, (int, np.integer)):
        raise TypeError(
            "current_t must be an integer."
        )
    
    if current_t < 0 or current_t >= T:
        raise ValueError(
            f"current_t must satisfy 0 <= current_t < {T}."
        )
    
    T_full = T
    
    remaining_T = (
        T_full - current_t
    )
    
    day_labels = np.asarray(
        day_labels
    )
    
    if len(day_labels) != T_full:
        raise ValueError(
            "day_labels must have length equal to the full horizon T."
        )

    # ============================================================
    # CURRENT FUTURES POSITION STATE
    #
    # current_weights contains the ACTUAL carried futures
    # margin weights entering current_t.
    #
    # This is observed state, not an optimizer decision.
    #
    # At t=0 there are no pre-existing positions, so the
    # default is zero exposure.
    #
    # For t>0 the actual carried position must be supplied.
    #
    # IMPORTANT:
    # Do NOT impose gross <= 1 here. Market movement may have
    # caused an existing position to drift above 100% of equity.
    # That grandfathering logic is handled separately.
    # ============================================================

    if current_t == 0:

        if current_weights is None:
            current_weights = np.zeros(
                K,
                dtype=float,
            )

    else:

        if current_weights is None:
            raise ValueError(
                "current_weights must be supplied when "
                "current_t > 0."
            )

    current_weights = np.asarray(
        current_weights,
        dtype=float,
    )

    if current_weights.shape != (K,):
        raise ValueError(
            "current_weights must have shape "
            f"({K},)."
        )

    if not np.all(
        np.isfinite(
            current_weights
        )
    ):
        raise ValueError(
            "current_weights must contain only "
            "finite values."
        )

    current_gross_weight = float(
        np.sum(
            np.abs(
                current_weights
            )
        )
    )
    
    # ============================================================
    # CURRENT ACCOUNT STATE
    #
    # For live/receding-horizon optimization, completed history
    # does not need to be reconstructed. The future problem only
    # needs our actual account balance at current_t.
    # ============================================================
    
    if current_t == 0:
    
        # At the beginning of the competition, our balance is the
        # common starting balance.
        if current_balance is None:
            current_balance = starting_balance
    
    else:
    
        if current_balance is None:
            raise ValueError(
                "current_balance must be supplied when current_t > 0."
            )
    
    current_balance = float(
        current_balance
    )
    
    if (
        not np.isfinite(current_balance)
        or current_balance <= 0.0
    ):
        raise ValueError(
            "current_balance must be a positive finite number."
        )
    
    # Future simulations begin from our actual current balance.
    future_starting_balance = current_balance

    # ============================================================
    # CURRENT OVERALL RANK
    #
    # At t=0 everyone begins from the common starting state,
    # so there is no meaningful realized rank yet.
    #
    # For t>0 we must know our actual current rank so that
    # our own leaderboard slot is never treated as a competitor.
    # ============================================================

    if current_t == 0:

        current_rank = None

    else:

        if current_rank is None:
            raise ValueError(
                "current_rank must be supplied "
                "when current_t > 0."
            )

        if not isinstance(
            current_rank,
            (int, np.integer),
        ):
            raise TypeError(
                "current_rank must be an integer."
            )

        if current_rank < 1:
            raise ValueError(
                "current_rank must be at least 1."
            )

        current_rank = int(
            current_rank
        )

    # ============================================================
    # ============================================================
    # CURRENT FINAL-PLACE BALANCES
    #
    # We now model SIX leaderboard places:
    #   1st, 2nd, 3rd, 4th, 15th, 16th
    #
    # These current balances are the starting levels from which
    # the six future place-return simulations will be compounded.
    # ============================================================

    final_place_labels = [
        "1st",
        "2nd",
        "3rd",
        "4th",
        "15th",
        "16th",
    ]

    # ============================================================
    # CURRENT OVERALL LEADERBOARD STATE
    # ============================================================

    if current_t == 0:

        if current_final_place_balances is None:
            current_final_place_balances = np.full(
                C_final_places,
                starting_balance,
                dtype=float,
            )

    else:

        if current_final_place_balances is None:
            raise ValueError(
                "current_final_place_balances must be supplied "
                "when current_t > 0. Expected balances for "
                "[1st, 2nd, 3rd, 4th, 15th, 16th]."
            )

    current_final_place_balances = np.asarray(
        current_final_place_balances,
        dtype=float,
    )

    if current_final_place_balances.shape != (6,):
        raise ValueError(
            "current_final_place_balances must have shape (6,) "
            "in the order "
            "[1st, 2nd, 3rd, 4th, 15th, 16th]."
        )

    if (
        not np.all(
            np.isfinite(
                current_final_place_balances
            )
        )
        or np.any(
            current_final_place_balances <= 0.0
        )
    ):
        raise ValueError(
            "current_final_place_balances must contain "
            "positive finite balances."
        )

    # ============================================================
    # CURRENT DAILY-PRIZE STATE
    # ============================================================

    current_day = day_labels[current_t]

    active_daily_day = (
        current_day in daily_threshold_days
    )

    if active_daily_day:

        # At the first interval of a prize day, no P&L has yet
        # accumulated. Otherwise use the observed leaderboard state.
        current_day_start_t = np.flatnonzero(
            day_labels == current_day
        )[0]

        if current_t == current_day_start_t:

            current_daily_rank = None
            
            if our_current_daily_pnl is None:
                our_current_daily_pnl = 0.0

            if current_daily_place_pnl is None:
                current_daily_place_pnl = np.zeros(
                    2,
                    dtype=float,
                )

        else:

            if our_current_daily_pnl is None:
                raise ValueError(
                    "our_current_daily_pnl must be supplied "
                    "during an active daily-prize day after "
                    "the day's first interval."
                )

            if current_daily_place_pnl is None:
                raise ValueError(
                    "current_daily_place_pnl must be supplied "
                    "during an active daily-prize day. Expected "
                    "[daily_1st_pnl, daily_2nd_pnl]."
                )

        if (
            current_t != current_day_start_t
            and current_daily_rank is None
        ):
            raise ValueError(
                "current_daily_rank must be supplied "
                "during an active daily-prize day."
            )

        if current_daily_rank is not None:

            if not isinstance(
                current_daily_rank,
                (int, np.integer),
            ):
                raise TypeError(
                    "current_daily_rank must be an integer."
                )

            if current_daily_rank < 1:
                raise ValueError(
                    "current_daily_rank must be at least 1."
                )

            current_daily_rank = int(
                current_daily_rank
            )
        
        our_realized_current_day_pnl = float(
            our_current_daily_pnl
        )

        current_daily_place_pnl = np.asarray(
            current_daily_place_pnl,
            dtype=float,
        )

        if current_daily_place_pnl.shape != (2,):
            raise ValueError(
                "current_daily_place_pnl must have shape (2,)."
            )

        if (
            not np.isfinite(
                our_realized_current_day_pnl
            )
            or not np.all(
                np.isfinite(
                    current_daily_place_pnl
                )
            )
        ):
            raise ValueError(
                "Current daily P&L state must contain "
                "finite values."
            )

    else:

        # Sunday and Friday have no daily prize.
        # These values are irrelevant to the objective.
        our_realized_current_day_pnl = 0.0

        current_daily_rank = None

        current_daily_place_pnl = np.zeros(
            2,
            dtype=float,
        )

    # Diagnostic reconstruction only.
    current_day_start_balance = (
        future_starting_balance
        - our_realized_current_day_pnl
    )
    
    daily_threshold_days = tuple(
        daily_threshold_days
    )

    # There are ALWAYS two stochastic daily place processes:
    #
    #   0 = daily 1st place
    #   1 = daily 2nd place
    #
    # Monday-Thursday are prize opportunities, not process rows.
    C_daily = 2

    if (
        n_daily_competitors is not None
        and n_daily_competitors != C_daily
    ):
        raise ValueError(
            "n_daily_competitors must equal 2 "
            "because the daily model contains "
            "1st-place and 2nd-place processes."
        )

    if eligible_days is None:
        eligible_days = list(daily_threshold_days)

    daily_temperature_schedule = tuple(
        float(tau)
        for tau in daily_temperature_schedule
    )

    if len(daily_temperature_schedule) == 0:
        raise ValueError(
            "daily_temperature_schedule must contain "
            "at least one value."
        )

    if any(
        tau <= 0.0
        for tau in daily_temperature_schedule
    ):
        raise ValueError(
            "All daily temperatures must be positive."
        )

    final_rank_temperature = float(
        final_rank_temperature
    )

    final_win_temperature = float(
        final_win_temperature
    )

    if (
        final_rank_temperature <= 0.0
        or final_win_temperature <= 0.0
    ):
        raise ValueError(
            "Final-prize temperatures must be positive."
        )

    (
        mu,
        Sigma,
        state_mean_competition,
        state_cov_competition,
        state_futures_cov_competition,
    ) = make_synthetic_split_problem(
        T=T,
        K=K,
        C_final=C_final_places,
        C_daily=C_daily,
        seed=seed,
    )

    # ============================================================
    # TIME-VARYING FUTURES MARGIN
    #
    # Shape:
    #   (T_full, K)
    #
    # margin[t,k] is the forecast dollar margin required
    # per contract.
    #
    # Effective leverage will be derived pathwise after the
    # raw futures Monte Carlo paths are simulated.
    # ============================================================
    
    margin = load_margin_forecasts(
        T=T_full,
        K=K,
    )

    # ============================================================
    # SLICE TO THE OPTIMIZATION HORIZON
    #
    # optimization_horizon=None preserves the original behavior:
    # optimize every period from current_t through T_full - 1.
    #
    # Otherwise, optimization_horizon is the maximum number of
    # periods included in the rolling optimization window.
    # ============================================================

    if optimization_horizon is None:
        optimization_end_t = T_full
    else:
        optimization_horizon = int(
            optimization_horizon
        )

        if optimization_horizon < 1:
            raise ValueError(
                "optimization_horizon must be at least 1 "
                "or None."
            )

        optimization_end_t = min(
            current_t + optimization_horizon,
            T_full,
        )

    mu = mu[
        current_t:optimization_end_t
    ]

    Sigma = Sigma[
        current_t:optimization_end_t
    ]

    state_mean_competition = (
        state_mean_competition[
            current_t:optimization_end_t,
            :
        ]
    )

    state_cov_competition = (
        state_cov_competition[
            current_t:optimization_end_t,
            :,
            :
        ]
    )

    state_futures_cov_competition = (
        state_futures_cov_competition[
            :,
            current_t:optimization_end_t,
            :
        ]
    )

    margin = margin[
        current_t:optimization_end_t,
        :
    ]

    day_labels_remaining = (
        day_labels[
            current_t:optimization_end_t
        ]
    )

    # ============================================================
    # ORIGINAL PRODUCTION TIME INDICES
    #
    # The forecast arrays below have been sliced to the remaining
    # horizon, but the frozen 50.15 coupling is indexed by the
    # original production interval.
    #
    # Example:
    #
    #   current_t = 4
    #
    # remaining array index 0 corresponds to original t = 4.
    # ============================================================

    t_original_values_remaining = np.arange(
        current_t,
        optimization_end_t,
        dtype=int,
    )

    remaining_T = (
        optimization_end_t
        - current_t
    )


    if mu.shape != (
        remaining_T,
        K,
    ):

        raise ValueError(
            "mu has incorrect remaining-horizon shape."
        )


    if Sigma.shape != (
        remaining_T,
        K,
        K,
    ):

        raise ValueError(
            "Sigma has incorrect remaining-horizon shape."
        )


    if state_mean_competition.shape != (
        remaining_T,
        8,
    ):

        raise ValueError(
            "state_mean_competition has incorrect "
            "remaining-horizon shape."
        )


    if state_cov_competition.shape != (
        remaining_T,
        8,
        8,
    ):

        raise ValueError(
            "state_cov_competition has incorrect "
            "remaining-horizon shape."
        )


    if state_futures_cov_competition.shape != (
        8,
        remaining_T,
        K,
    ):

        raise ValueError(
            "state_futures_cov_competition has "
            "incorrect remaining-horizon shape."
        )


    if margin.shape != (
        remaining_T,
        K,
    ):
    
        raise ValueError(
            "margin must have shape "
            f"({remaining_T}, {K})."
        )

    if not np.all(
        np.isfinite(
            margin
        )
    ):
    
        raise ValueError(
            "Remaining-horizon margin forecasts "
            "must all be finite."
        )
    
    
    if np.any(
        margin <= 0.0
    ):
    
        raise ValueError(
            "Remaining-horizon margin forecasts "
            "must all be positive."
        )

    if len(
        day_labels_remaining
    ) != remaining_T:

        raise ValueError(
            "day_labels_remaining has "
            "incorrect length."
        )

    if t_original_values_remaining.shape != (
        remaining_T,
    ):
        raise ValueError(
            "t_original_values_remaining has "
            "incorrect shape."
        )

    # ============================================================
    # KEEP ONLY DAILY-PRIZE DAYS THAT STILL EXIST
    # IN THE REMAINING HORIZON
    # ============================================================

    remaining_days = set(
        day_labels_remaining.tolist()
    )

    active_daily_threshold_days = tuple(
        day
        for day in daily_threshold_days
        if day in remaining_days
    )

    # Once the one-time daily prize has already been won,
    # there are no remaining daily-prize opportunities.
    if daily_prize_already_won:
        active_daily_threshold_days = ()

    daily_threshold_days = (
        active_daily_threshold_days
    )

    eligible_days = list(
        active_daily_threshold_days
    )
    
    # From this point onward, the optimizer works only
    # with the remaining horizon.
    T = remaining_T


    # ============================================================
    # ONE JOINT COMPETITION MONTE CARLO
    #
    # The SAME simulated world now determines:
    #
    #   - futures returns
    #   - six final leaderboard places
    #   - daily 1st place
    #   - daily 2nd place
    #
    # Therefore we preserve:
    #
    #   final <-> final dependence
    #   daily <-> daily dependence
    #   final <-> daily dependence
    #   competitor-state <-> futures dependence
    # ============================================================

    (
        R_sims,
        final_place_balance_sims,
        B_sims,
    ) = simulate_competition_paths(
        mu=mu,
        Sigma=Sigma,

        state_mean_competition=
            state_mean_competition,

        state_cov_competition=
            state_cov_competition,

        state_futures_cov_competition=
            state_futures_cov_competition,

        current_final_place_balances=
            current_final_place_balances,

        current_daily_place_pnl=
            current_daily_place_pnl,

        day_labels=
            day_labels_remaining,

        t_original_values=
            t_original_values_remaining,

        n_sims=n_sims,

        simulation_model=
            simulation_model,

        innovation_system=
            distributional_innovation_system,

        mapping_system=
            distributional_mapping_system,

        seed=
            seed + 88888,
    )


    # ============================================================
    # CONVERT RAW FUTURES RETURNS TO ACCOUNT RETURNS
    #
    # Contract notional evolves separately on every Monte Carlo
    # path. Margin is currently deterministic across paths.
    #
    # The leverage used during interval t is based on the
    # START-OF-INTERVAL notional, so the return at t affects
    # leverage beginning at t+1.
    # ============================================================
    
    (
        R_account_sims,
        pathwise_leverage,
        pathwise_notional,
    ) = build_pathwise_leverage_and_account_returns(
        R_sims=R_sims,
        starting_notional=current_contract_notional,
        margin=margin,
    )

    # ============================================================
    # STAGE 5B.4C - PRODUCTION DOLLAR-ACCOUNT INITIAL STATE
    #
    # Recover the actual optimizer-contract position entering the
    # remaining horizon exactly once.
    #
    # Frozen Stage 5A.1 identity:
    #
    #     current_weight
    #         = q * current_margin / current_balance
    #
    # therefore:
    #
    #     q
    #         = current_weight * current_balance / current_margin
    #
    # These contracts are state. W[0] remains the optimizer's
    # requested target for the first future interval.
    # ============================================================
    
    initial_contracts = (
        current_weights
        * future_starting_balance
        / margin[
            0,
            :
        ]
    )
    
    if initial_contracts.shape != (
        K,
    ):
        raise ValueError(
            "initial_contracts must have shape (K,)."
        )
    
    if not np.all(
        np.isfinite(
            initial_contracts
        )
    ):
        raise ValueError(
            "initial_contracts must contain only finite values."
        )
    
    reference_balance = float(
        future_starting_balance
    )

    # ============================================================
    # STANDARD-EQUIVALENT CONTRACT LIMITS
    #
    # Convert each underlying's maximum standard-equivalent
    # contract exposure into the optimizer's margin-weight units.
    #
    # For future k at time t:
    #
    #     q_impl
    #         = W[t,k] * reference_balance / margin[t,k]
    #
    #     q_standard
    #         = q_impl * standard_contract_display_factor[k]
    #
    # Therefore:
    #
    #     |W[t,k]|
    #         <= contract_limit[k] * margin[t,k]
    #            / (
    #                reference_balance
    #                * standard_contract_display_factor[k]
    #            )
    #
    # The existing scalar weight_bound remains active as well.
    # ============================================================

    if standard_equivalent_contract_limit is None:
        raise ValueError(
            "standard_equivalent_contract_limit must be supplied."
        )

    if standard_contract_display_factor is None:
        raise ValueError(
            "standard_contract_display_factor must be supplied."
        )

    standard_equivalent_contract_limit = np.asarray(
        standard_equivalent_contract_limit,
        dtype=float,
    )

    standard_contract_display_factor = np.asarray(
        standard_contract_display_factor,
        dtype=float,
    )

    if standard_equivalent_contract_limit.shape != (K,):
        raise ValueError(
            "standard_equivalent_contract_limit must "
            f"have shape ({K},)."
        )

    if standard_contract_display_factor.shape != (K,):
        raise ValueError(
            "standard_contract_display_factor must "
            f"have shape ({K},)."
        )

    if not np.all(
        np.isfinite(
            standard_equivalent_contract_limit
        )
    ):
        raise ValueError(
            "standard_equivalent_contract_limit contains "
            "non-finite values."
        )

    if not np.all(
        np.isfinite(
            standard_contract_display_factor
        )
    ):
        raise ValueError(
            "standard_contract_display_factor contains "
            "non-finite values."
        )

    if np.any(
        standard_equivalent_contract_limit <= 0.0
    ):
        raise ValueError(
            "All standard-equivalent contract limits "
            "must be positive."
        )

    if np.any(
        standard_contract_display_factor <= 0.0
    ):
        raise ValueError(
            "All standard-contract display factors "
            "must be positive."
        )

    contract_limit_weight_bound = (
        standard_equivalent_contract_limit[
            None,
            :
        ]
        * margin
        / (
            reference_balance
            * standard_contract_display_factor[
                None,
                :
            ]
        )
    )

    contract_limit_weight_bound = np.minimum(
        contract_limit_weight_bound,
        float(
            weight_bound
        ),
    )

    if contract_limit_weight_bound.shape != (
        T,
        K,
    ):
        raise ValueError(
            "contract_limit_weight_bound has "
            "incorrect shape."
        )

    if R_account_sims.shape != (
        n_sims,
        T,
        K,
    ):

        raise ValueError(
            "R_account_sims has incorrect shape."
        )


    # ============================================================
    # BACKWARD-COMPATIBILITY ALIAS
    #
    # The existing final-prize function calls its futures input
    # R_final_sims.
    #
    # It is now deliberately the EXACT SAME Monte Carlo array
    # used by the daily-prize objective.
    # ============================================================

    R_final_account_sims = (
        R_account_sims
    )

    records = []

    # ============================================================
    # BUILD OPTIMIZATION CASES
    #
    # There is now ONE target-space formulation.
    #
    # We no longer solve separate "normal" and "grandfathered"
    # current-row regimes.
    #
    # The optimizer proposes desired positions. The frozen
    # execute_target_contracts() layer determines what can
    # actually execute path by path.
    # ============================================================
    
    rng = np.random.default_rng(
        seed + 30000
    )
    
    optimization_cases = []
    
    for start in range(
        1,
        n_starts + 1,
    ):
    
        W_initial = rng.uniform(
            -0.05,
            0.05,
            size=(T, K),
        )
    
        W_initial = (
            project_weight_matrix_for_execution(
                W_initial,
                current_weights=
                    current_weights,
                gross_limit=1.0,
                weight_bound=
                    weight_bound,
                weight_bound_matrix=
                    contract_limit_weight_bound,
            )
        )
    
        optimization_cases.append(
            (
                start,
                "execution_target",
                W_initial,
            )
        )

    # ============================================================
    # OPTIMIZE EACH REGIME SEPARATELY
    # ============================================================

    for (
        start,
        current_regime,
        W,
    ) in optimization_cases:

        start_time = time.perf_counter()
        function_evals = 0
        main_gradient_evals = 0
        trial_evals = 0
        total_backtracks = 0
        total_iterations = 0
        stage_records = []
        all_stages_converged = True
        final_weight_change = np.nan
        final_objective_change = np.nan

        for tau in daily_temperature_schedule:
            step_size = initial_step
            stage_converged = False

            for stage_iteration in range(1, maxiter + 1):
                objective, G, diagnostics = (
                    total_expected_prize_and_batched_reverse_gradient_from_dollar_account(
                        W=W,
                        raw_return_sims=
                            R_sims,
                        pathwise_notional=
                            pathwise_notional,
                        margin=margin,
                        initial_contracts=
                            initial_contracts,
                        B_sims=B_sims,
                        final_place_balance_sims=
                            final_place_balance_sims,
                        day_labels=
                            day_labels_remaining,
                        prizes=prizes,
                        starting_balance=
                            future_starting_balance,
                        current_final_place_balances=
                            current_final_place_balances,
                        current_rank=
                            current_rank,
                        daily_prize=
                            daily_prize,
                        final_win_temperature=
                            final_win_temperature,
                        daily_competitor_temperature=
                            tau,
                        daily_win_temperature=
                            tau,
                        eligible_days=
                            eligible_days,
                        daily_threshold_days=
                            daily_threshold_days,
                        current_day=
                            current_day,
                        our_realized_current_day_pnl=
                            our_realized_current_day_pnl,
                        current_daily_rank=
                            current_daily_rank,
                        current_daily_place_pnl=
                            current_daily_place_pnl,
                        daily_prize_already_won=
                            daily_prize_already_won,
                        reference_balance=
                            reference_balance,
                    )
                )
                function_evals += 1
                main_gradient_evals += 1

                trial_step = step_size
                accepted = False
                iteration_trial_evals = 0

                for _ in range(50):
                    W_trial = (
                        W
                        + trial_step * G
                    )

                    W_trial = (
                        project_weight_matrix_for_execution(
                            W_trial,
                            current_weights=
                                current_weights,
                            gross_limit=1.0,
                            weight_bound=
                                weight_bound,
                            weight_bound_matrix=
                                contract_limit_weight_bound,
                        )
                    )

                    function_evals += 1
                    trial_evals += 1
                    iteration_trial_evals += 1

                    trial_objective = (
                        total_expected_prize_value_only_from_dollar_account(
                            W=W_trial,
                            raw_return_sims=
                                R_sims,
                            pathwise_notional=
                                pathwise_notional,
                            margin=margin,
                            initial_contracts=
                                initial_contracts,
                            B_sims=B_sims,
                            final_place_balance_sims=
                                final_place_balance_sims,
                            day_labels=
                                day_labels_remaining,
                            prizes=prizes,
                            starting_balance=
                                future_starting_balance,
                            current_final_place_balances=
                                current_final_place_balances,
                            current_rank=
                                current_rank,
                            daily_prize=
                                daily_prize,
                            final_win_temperature=
                                final_win_temperature,
                            daily_competitor_temperature=
                                tau,
                            daily_win_temperature=
                                tau,
                            eligible_days=
                                eligible_days,
                            daily_threshold_days=
                                daily_threshold_days,
                            current_day=
                                current_day,
                            our_realized_current_day_pnl=
                                our_realized_current_day_pnl,
                            current_daily_rank=
                                current_daily_rank,
                            current_daily_place_pnl=
                                current_daily_place_pnl,
                            daily_prize_already_won=
                                daily_prize_already_won,
                            reference_balance=
                                reference_balance,
                        )
                    )

                    if trial_objective >= objective:
                        accepted = True
                        break

                    total_backtracks += 1
                    trial_step *= 0.5

                if not accepted:
                    elapsed_now = time.perf_counter() - start_time

                    print(
                        f"[OPT] LINE SEARCH FAILED | "
                        f"iter={stage_iteration} | "
                        f"this_trials={iteration_trial_evals} | "
                        f"evals={function_evals} | "
                        f"main={main_gradient_evals} | "
                        f"trial={trial_evals} | "
                        f"backtracks={total_backtracks} | "
                        f"elapsed={elapsed_now:.1f}s",
                        flush=True,
                    )
                    break

                final_weight_change = np.max(np.abs(W_trial - W))
                final_objective_change = trial_objective - objective
                W = W_trial
                step_size = min(trial_step * 1.2, 10.0)

                if (
                    stage_iteration == 1
                    or stage_iteration % 10 == 0
                ):
                    elapsed_now = time.perf_counter() - start_time

                    print(
                        f"[OPT] iter={stage_iteration:3d} | "
                        f"evals={function_evals:4d} | "
                        f"main={main_gradient_evals:3d} | "
                        f"trial={trial_evals:4d} | "
                        f"this_trials={iteration_trial_evals:2d} | "
                        f"backtracks={total_backtracks:4d} | "
                        f"obj={trial_objective:.6f} | "
                        f"dObj={final_objective_change:.3e} | "
                        f"dW={final_weight_change:.3e} | "
                        f"step={trial_step:.3e} | "
                        f"elapsed={elapsed_now:.1f}s",
                        flush=True,
                    )

                if (
                    final_weight_change < weight_tolerance
                    and abs(final_objective_change) < objective_tolerance
                ):
                    stage_converged = True
                    break

            total_iterations += stage_iteration

            print(
                "\n"
                f"[OPT STAGE END] tau={tau} | "
                f"iterations={stage_iteration} | "
                f"converged={stage_converged} | "
                f"evals_so_far={function_evals} | "
                f"main={main_gradient_evals} | "
                f"trial={trial_evals} | "
                f"backtracks={total_backtracks} | "
                f"elapsed={time.perf_counter() - start_time:.1f}s"
                "\n",
                flush=True,
            )

            stage_objective, _, stage_diagnostics = (
                total_expected_prize_and_batched_reverse_gradient_from_dollar_account(
                    W=W,
                    raw_return_sims=
                        R_sims,
                    pathwise_notional=
                        pathwise_notional,
                    margin=margin,
                    initial_contracts=
                        initial_contracts,
                    B_sims=B_sims,
                    final_place_balance_sims=
                        final_place_balance_sims,
                    day_labels=
                        day_labels_remaining,
                    prizes=prizes,
                    starting_balance=
                        future_starting_balance,
                    current_final_place_balances=
                        current_final_place_balances,
                    current_rank=
                        current_rank,
                    daily_prize=
                        daily_prize,
                    final_win_temperature=
                        final_win_temperature,
                    daily_competitor_temperature=
                        tau,
                    daily_win_temperature=
                        tau,
                    eligible_days=
                        eligible_days,
                    daily_threshold_days=
                        daily_threshold_days,
                    current_day=
                        current_day,
                    our_realized_current_day_pnl=
                        our_realized_current_day_pnl,
                    current_daily_rank=
                        current_daily_rank,
                    current_daily_place_pnl=
                        current_daily_place_pnl,
                    daily_prize_already_won=
                        daily_prize_already_won,
                    reference_balance=
                        reference_balance,
                )
            )
            function_evals += 1

            stage_records.append({
                "temperature": tau,
                "converged": stage_converged,
                "iterations": stage_iteration,
                "expected_prize": stage_objective,
                "final_expected_prize": stage_diagnostics["final_expected_prize"],
                "daily_expected_prize": stage_diagnostics["daily_expected_prize"],
                "smooth_daily_win_probability": stage_diagnostics["daily"][
                    "probability_win_at_least_once"
                ],
            })

            if not stage_converged:
                all_stages_converged = False

        runtime = time.perf_counter() - start_time
        final_daily_tau = (
            daily_temperature_schedule[-1]
        )

        final_objective, _, final_diagnostics = (
            total_expected_prize_and_batched_reverse_gradient_from_dollar_account(
                W=W,
                raw_return_sims=
                    R_sims,
                pathwise_notional=
                    pathwise_notional,
                margin=margin,
                initial_contracts=
                    initial_contracts,
                B_sims=B_sims,
                final_place_balance_sims=
                    final_place_balance_sims,
                day_labels=
                    day_labels_remaining,
                prizes=prizes,
                starting_balance=
                    future_starting_balance,
                current_final_place_balances=
                    current_final_place_balances,
                current_rank=
                    current_rank,
                daily_prize=
                    daily_prize,
                final_win_temperature=
                    final_win_temperature,
                daily_competitor_temperature=
                    final_daily_tau,
                daily_win_temperature=
                    final_daily_tau,
                eligible_days=
                    eligible_days,
                daily_threshold_days=
                    daily_threshold_days,
                current_day=
                    current_day,
                our_realized_current_day_pnl=
                    our_realized_current_day_pnl,
                current_daily_rank=
                    current_daily_rank,
                current_daily_place_pnl=
                    current_daily_place_pnl,
                daily_prize_already_won=
                    daily_prize_already_won,
                reference_balance=
                    reference_balance,
            )
        )
        function_evals += 1

        hard_accounting = (
            calculate_daily_accounting_from_dollar_account(
                account=
                    final_diagnostics["account"],
                B_sims=B_sims,
                day_labels=
                    day_labels_remaining,
            )
        )
        # ============================================================
        # STEP 7:
        # HARD DAILY EVALUATION
        # ============================================================

        if (
            daily_prize_already_won
            or (
                daily_threshold_days is not None
                and len(daily_threshold_days) == 0
            )
        ):

            hard_daily_result = {
                "days": [],
                "daily_threshold_pnl": None,
                "daily_win_indicators": None,
                "any_daily_win": None,
                "daily_win_probabilities":
                    np.empty(0),
                "probability_win_at_least_once":
                    0.0,
                "expected_daily_prize":
                    0.0,
            }

        else:

            hard_daily_result = calculate_hard_daily_prize(
                accounting=hard_accounting,
                daily_prize=daily_prize,
                eligible_days=eligible_days,
                daily_threshold_days=
                    daily_threshold_days,
                current_day=current_day,
                our_realized_current_day_pnl=
                    our_realized_current_day_pnl,
                current_daily_rank=
                    current_daily_rank,
                current_daily_place_pnl=
                    current_daily_place_pnl,
            )

        # ============================================================
        # REBUILD FULL-TIMELINE WEIGHT MATRIX
        # ============================================================

        # ========================================================
        # FINAL SEQUENTIAL GRANDFATHERING VALIDATION
        # ========================================================

        previous_validation_weights = (
            current_weights.copy()
        )

        for validation_t in range(
            W.shape[0]
        ):

            validation_weights = W[
                validation_t,
                :
            ]

            validation_gross = float(
                np.sum(
                    np.abs(
                        validation_weights
                    )
                )
            )

            previous_validation_gross = float(
                np.sum(
                    np.abs(
                        previous_validation_weights
                    )
                )
            )

            if validation_gross > 1.0 + 1e-10:

                if previous_validation_gross <= 1.0 + 1e-10:
                    raise RuntimeError(
                        "Final optimized path illegally returned "
                        "above 100% gross after reaching the "
                        "normal regime."
                    )

                if np.any(
                    np.abs(
                        validation_weights
                    )
                    >
                    np.abs(
                        previous_validation_weights
                    )
                    + 1e-10
                ):
                    raise RuntimeError(
                        "Final optimized path violates the "
                        "sequential grandfathered gross-exposure "
                        "constraint."
                    )

                nonzero_validation = (
                    np.abs(
                        validation_weights
                    )
                    > 1e-10
                )

                if np.any(
                    nonzero_validation
                    &
                    (
                        np.sign(
                            validation_weights
                        )
                        !=
                        np.sign(
                            previous_validation_weights
                        )
                    )
                ):
                    raise RuntimeError(
                        "Final optimized path reverses exposure "
                        "while remaining above 100% gross."
                    )

            previous_validation_weights = (
                validation_weights.copy()
            )

        # Completed intervals are no longer reconstructed.
        # Mark them NaN; rows current_t onward contain the
        # optimized remaining-horizon weights.
        W_full = np.full(
            (T_full, K),
            np.nan,
            dtype=float,
        )

        W_full[
            current_t:optimization_end_t,
            :
        ] = W

        assert W_full.shape == (
            T_full,
            K
        )

        records.append({
            "T": T,
            "K": K,
            "full_T":
                T_full,

            "remaining_T":
                remaining_T,

            "K":
                K,

            "variables":
                remaining_T * K,

            "start":
                start,

            "current_regime":
                current_regime,

            "success":
                all_stages_converged,
            "iterations": total_iterations,
            "function_evals": function_evals,
            "runtime_sec": runtime,
            "final_daily_temperature":
                final_daily_tau,

            "final_rank_temperature":
                final_rank_temperature,

            "final_win_temperature":
                final_win_temperature,
            "expected_prize": final_objective,
            "final_expected_prize": final_diagnostics["final_expected_prize"],
            "daily_expected_prize": final_diagnostics["daily_expected_prize"],
            "daily_win_probability": final_diagnostics["daily"][
                "probability_win_at_least_once"
            ],
            "hard_daily_expected_prize": hard_daily_result[
                "expected_daily_prize"
            ],
            "hard_daily_win_probability": hard_daily_result[
                "probability_win_at_least_once"
            ],
            "final_weight_change": final_weight_change,
            "final_objective_change": final_objective_change,
            "stage_results": stage_records,
            # ------------------------------------------------
            # DISCRETE EXECUTION EVALUATION STATE
            #
            # Preserve the exact Monte Carlo worlds and
            # dollar-account inputs used by this optimizer run.
            #
            # These are required so the discrete current-period
            # candidates can be compared on identical worlds.
            # ------------------------------------------------

            "evaluation_R_sims":
                R_sims.copy(),

            "evaluation_pathwise_notional":
                pathwise_notional.copy(),

            "evaluation_margin":
                margin.copy(),

            "evaluation_initial_contracts":
                initial_contracts.copy(),

            "evaluation_B_sims":
                B_sims.copy(),

            "evaluation_final_place_balance_sims":
                final_place_balance_sims.copy(),

            "evaluation_day_labels":
                np.asarray(
                    day_labels_remaining,
                    dtype=object,
                ).copy(),

            "evaluation_reference_balance":
                reference_balance,

            "evaluation_final_daily_temperature":
                final_daily_tau,
            "weights_remaining":
                W.copy(),

            "weights_full":
                W_full.copy(),

            "current_t":
                current_t,

            "remaining_intervals":
                remaining_T,
            
            # STEP 3
            "future_starting_balance": 
                future_starting_balance,
            
            # Backward-compatible alias
            "weights":
                W_full.copy(),
            
            "current_balance":
                future_starting_balance,

            # Actual carried positions entering current_t.
            "current_position_weights":
                current_weights.copy(),

            "current_gross_weight":
                current_gross_weight,
            
            "current_final_place_balances":
                current_final_place_balances.copy(),

            "current_rank":
                current_rank,
            
            "current_day":
                current_day,

            "current_day_start_balance":
                current_day_start_balance,

            "our_realized_current_day_pnl":
                our_realized_current_day_pnl,

            "current_daily_rank":
                current_daily_rank,
            
            "current_daily_place_pnl":
                current_daily_place_pnl.copy(),
            
            "daily_prize_already_won":
                daily_prize_already_won,

            "active_daily_threshold_days":
                daily_threshold_days,
            
            "remaining_time_indices":
                np.arange(
                    current_t,
                    optimization_end_t,
                ),
            
            "current_weights":
                W[0].copy(),
                        
            "message": (
                "All temperature stages converged"
                if all_stages_converged
                else "One or more temperature stages stopped before tolerance"
            ),
        })

    results_df = pd.DataFrame(
        records
    )

    results_df = (
        results_df
        .sort_values(
            by="expected_prize",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )

    return results_df

def build_summary(results):

    summary = (
        results
        .groupby(
            [
                "T",
                "K",
                "variables"
            ],
            as_index=False
        )
        .agg(
            success_rate=(
                "success",
                "mean"
            ),

            avg_runtime_sec=(
                "runtime_sec",
                "mean"
            ),

            median_runtime_sec=(
                "runtime_sec",
                "median"
            ),

            avg_iterations=(
                "iterations",
                "mean"
            ),

            best_expected_prize=(
                "expected_prize",
                "max"
            ),

            worst_expected_prize=(
                "expected_prize",
                "min"
            ),

            avg_expected_prize=(
                "expected_prize",
                "mean"
            ),

            std_expected_prize=(
                "expected_prize",
                "std"
            ),

            avg_final_expected_prize=(
                "final_expected_prize",
                "mean"
            ),

            avg_daily_expected_prize=(
                "daily_expected_prize",
                "mean"
            ),

            avg_daily_win_probability=(
                "daily_win_probability",
                "mean"
            ),
        )
    )


    summary[
        "success_rate"
    ] *= 100.0


    summary[
        "objective_spread"
    ] = (
        summary[
            "best_expected_prize"
        ]
        - summary[
            "worst_expected_prize"
        ]
    )


    return summary


def main():
    parser = argparse.ArgumentParser(
        description="Benchmark high-dimensional prize-maximization optimizer."
    )

    parser.add_argument(
        "--K",
        type=int,
        default=8,
        help="Number of futures/assets. Default: 8",
    )

    parser.add_argument(
        "--T",
        nargs="+",
        type=int,
        default=[20, 60, 120],
        help="Horizons to test. Default: 20 60 120",
    )

    parser.add_argument(
        "--starts",
        type=int,
        default=5,
        help="Random starts per horizon. Default: 5",
    )

    parser.add_argument(
        "--bound",
        type=float,
        default=1.0,
        help="Absolute bound on each individual weight. Default: 1.00",
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=20260813,
        help="Random seed.",
    )

    parser.add_argument(
        "--maxiter",
        type=int,
        default=1000,
        help="Maximum optimizer iterations.",
    )

    parser.add_argument(
        "--output-prefix",
        type=str,
        default="optimizer_benchmark",
        help="Prefix for output CSV files.",
    )

    args = parser.parse_args(args=[])

    # Placeholder prize terms for the benchmark.
    # These now encode exact mutually-exclusive rank probabilities.
    prizes = np.array(
        [750.0, 500.0, 1150.0, 100.0]
    )

    all_results = []

    print("\nRunning dimensionality benchmark...\n")

    for idx, T in enumerate(args.T):
        print(
            f"T={T:>4} | "
            f"K={args.K:>2} | "
            f"variables={T * args.K:>5}"
        )

        result_df = run_single_problem(
            T=T,
            K=args.K,
            prizes=prizes,
            n_starts=args.starts,
            weight_bound=args.bound,
            seed=args.seed + idx * 1000,
            maxiter=args.maxiter,
        )

        all_results.append(result_df)

    results = pd.concat(
        all_results,
        ignore_index=True,
    )

    summary = build_summary(results)

    print("\nSUMMARY")
    print("=" * 100)
    print(summary.to_string(index=False))

    print("\nALL RUNS")
    print("=" * 100)

    columns = [
        "T",
        "variables",
        "start",
        "success",
        "iterations",
        "function_evals",
        "runtime_sec",
        "expected_prize",
    ]

    print(results[columns].to_string(index=False))

    results_file = (
        f"{args.output_prefix}_all_runs.csv"
    )

    summary_file = (
        f"{args.output_prefix}_summary.csv"
    )

    results.to_csv(
        results_file,
        index=False,
    )

    summary.to_csv(
        summary_file,
        index=False,
    )

    print("\nSaved:")
    print(f"  {results_file}")
    print(f"  {summary_file}")

    print("\nWhat to look for:")
    print(
        "1. success_rate near 100%."
    )
    print(
        "2. Runtime as T and the number of variables increase."
    )
    print(
        "3. objective_spread across random starts."
    )
    print(
        "   A large spread can indicate local-optimum sensitivity."
    )

# ------------------------------------------------------------
# HELPER 1
# Enumerate every attainable exposure for ONE underlying.
#
# We enumerate product combinations only within an underlying.
# At most WTI has 3 products:
#
#   21^3 = 9261
#
# which is tiny.
#
# Multiple actual-product combinations can produce the same
# economic exposure. For now, retain the decomposition with
# the smallest total absolute number of contracts.
#
# Margin tie-breaking is added below.
# ------------------------------------------------------------

def build_underlying_discrete_exposure_map(
    future_name,
    implementation_margin,
    product_specification,
    max_abs_position=10,
    exposure_round_digits=10,
):

    if future_name not in product_specification:
        raise KeyError(
            f"Missing product specification for {future_name}."
        )

    products = product_specification[
        future_name
    ]

    product_names = list(
        products.keys()
    )

    position_ranges = [
        range(
            -int(
                products[name][
                    "max_abs_contracts"
                ]
            ),
            int(
                products[name][
                    "max_abs_contracts"
                ]
            ) + 1,
        )
        for name in product_names
    ]

    exposure_map = {}

    for positions in product(
        *position_ranges
    ):

        positions = np.asarray(
            positions,
            dtype=int,
        )

        standard_exposure = 0.0
        implementation_units = 0.0
        required_margin = 0.0

        for j, product_name in enumerate(
            product_names
        ):

            spec = products[
                product_name
            ]

            q = int(
                positions[j]
            )

            impl_units = float(
                spec[
                    "implementation_units"
                ]
            )

            std_equivalent = float(
                spec[
                    "standard_equivalent"
                ]
            )

            standard_exposure += (
                q
                * std_equivalent
            )

            implementation_units += (
                q
                * impl_units
            )

            required_margin += (
                abs(q)
                * impl_units
                * float(
                    implementation_margin
                )
            )

        exposure_key = round(
            float(
                standard_exposure
            ),
            exposure_round_digits,
        )

        total_abs_contracts = int(
            np.sum(
                np.abs(
                    positions
                )
            )
        )

        candidate = {
            "future": future_name,
            "product_names": tuple(
                product_names
            ),
            "positions": tuple(
                int(x)
                for x in positions
            ),
            "standard_exposure": float(
                standard_exposure
            ),
            "implementation_units": float(
                implementation_units
            ),
            "required_margin": float(
                required_margin
            ),
            "total_abs_contracts": (
                total_abs_contracts
            ),
        }

        if exposure_key not in exposure_map:

            exposure_map[
                exposure_key
            ] = candidate

        else:

            old = exposure_map[
                exposure_key
            ]

            new_score = (
                candidate[
                    "required_margin"
                ],
                candidate[
                    "total_abs_contracts"
                ],
            )

            old_score = (
                old[
                    "required_margin"
                ],
                old[
                    "total_abs_contracts"
                ],
            )

            if new_score < old_score:

                exposure_map[
                    exposure_key
                ] = candidate

    return exposure_map


# ------------------------------------------------------------
# HELPER 2
# Return the nearest discrete exposures for one continuous
# standard-equivalent target.
#
# We keep candidates on BOTH sides of the continuous target.
# This matters because the prize objective may prefer slightly
# more or slightly less exposure.
# ------------------------------------------------------------

def get_nearby_underlying_candidates(
    desired_standard_exposure,
    exposure_map,
    n_each_side=3,
):

    desired_standard_exposure = float(
        desired_standard_exposure
    )

    exposures = np.array(
        sorted(
            exposure_map.keys()
        ),
        dtype=float,
    )

    lower = exposures[
        exposures
        <= desired_standard_exposure
    ]

    upper = exposures[
        exposures
        >= desired_standard_exposure
    ]

    selected = []

    if lower.size > 0:

        selected.extend(
            lower[
                -n_each_side:
            ].tolist()
        )

    if upper.size > 0:

        selected.extend(
            upper[
                :n_each_side
            ].tolist()
        )

    nearest_index = int(
        np.argmin(
            np.abs(
                exposures
                - desired_standard_exposure
            )
        )
    )

    selected.append(
        float(
            exposures[
                nearest_index
            ]
        )
    )

    # Always include zero as an executable alternative.
    if 0.0 in exposure_map:

        selected.append(
            0.0
        )

    selected = sorted(
        set(
            round(
                float(x),
                10,
            )
            for x in selected
        )
    )

    candidates = [
        exposure_map[
            exposure
        ]
        for exposure in selected
    ]

    candidates = sorted(
        candidates,
        key=lambda x: (
            abs(
                x[
                    "standard_exposure"
                ]
                - desired_standard_exposure
            ),
            x[
                "required_margin"
            ],
            x[
                "total_abs_contracts"
            ],
        ),
    )

    return candidates


# ------------------------------------------------------------
# HELPER 3
# Convert a continuous weight into the standard-equivalent
# contract target used by the discrete layer.
#
# This uses exactly the same dimensional conversion as the
# validated continuous contract-limit work.
# ------------------------------------------------------------

def weight_to_standard_contract_target(
    weight,
    reference_balance,
    implementation_margin,
    display_factor,
):

    implementation_contracts = (
        float(weight)
        * float(reference_balance)
        / float(
            implementation_margin
        )
    )

    standard_contracts = (
        implementation_contracts
        * float(
            display_factor
        )
    )

    return float(
        standard_contracts
    )

# ------------------------------------------------------------
# HELPER 1
# Convert one Stage-1 candidate into canonical implementation
# contract quantity.
# ------------------------------------------------------------

def discrete_candidate_to_implementation_contracts(
    candidate,
):

    return float(
        candidate[
            "implementation_units"
        ]
    )


# ------------------------------------------------------------
# HELPER 2
# Convert an 8-future implementation-contract target into the
# corresponding canonical optimizer weight vector.
# ------------------------------------------------------------

def implementation_contracts_to_weights(
    q,
    margin_per_contract,
    reference_balance,
):

    q = np.asarray(
        q,
        dtype=float,
    )

    margin_per_contract = np.asarray(
        margin_per_contract,
        dtype=float,
    )

    reference_balance = float(
        reference_balance
    )

    if q.shape != (K,):
        raise ValueError(
            "q must have shape (K,)."
        )

    if margin_per_contract.shape != (K,):
        raise ValueError(
            "margin_per_contract must have shape (K,)."
        )

    if (
        not np.isfinite(reference_balance)
        or reference_balance <= 0.0
    ):
        raise ValueError(
            "reference_balance must be positive and finite."
        )

    return (
        q
        * margin_per_contract
        / reference_balance
    )


# ------------------------------------------------------------
# HELPER 3
# Squared distance from the continuous target.
#
# We use WEIGHT space because the continuous optimizer itself
# operates in weight space.
#
# This score is NOT prize EV. It is only used to decide which
# feasible discrete candidates are worth carrying into Stage 3.
# ------------------------------------------------------------

def discrete_weight_distance(
    candidate_weights,
    continuous_weights,
):

    candidate_weights = np.asarray(
        candidate_weights,
        dtype=float,
    )

    continuous_weights = np.asarray(
        continuous_weights,
        dtype=float,
    )

    return float(
        np.sum(
            (
                candidate_weights
                - continuous_weights
            ) ** 2
        )
    )

# ------------------------------------------------------------
# HELPER 4
# Validate an actual-product transition.
#
# Product positions are integer positions on the platform.
#
# A transition is product-feasible when:
#
#   1. every target product position is an integer;
#   2. every target product position is within its own
#      +/- max_abs_contracts limit;
#   3. the target dictionary contains exactly the known
#      tradable products.
#
# Closing, reducing, increasing, opening, and reversing a
# product are all represented directly by old -> target.
#
# Margin/grandfathering feasibility remains the responsibility
# of execute_target_contracts() in canonical implementation
# units. This helper handles the product-specific platform
# position constraints.
# ------------------------------------------------------------

def validate_actual_product_transition(
    current_positions,
    target_positions,
    product_specification,
    future_names,
    tol=1e-12,
):

    expected_products = set()

    for future_name in future_names:

        expected_products.update(
            product_specification[
                future_name
            ].keys()
        )

    if set(current_positions.keys()) != expected_products:
        raise ValueError(
            "current_positions does not contain exactly "
            "the expected actual products."
        )

    if set(target_positions.keys()) != expected_products:
        raise ValueError(
            "target_positions does not contain exactly "
            "the expected actual products."
        )

    trades = {}

    for future_name in future_names:

        products = product_specification[
            future_name
        ]

        for product_name, product_spec in products.items():

            q_old_raw = current_positions[
                product_name
            ]

            q_target_raw = target_positions[
                product_name
            ]

            if (
                not np.isfinite(q_old_raw)
                or not np.isfinite(q_target_raw)
            ):
                return {
                    "feasible": False,
                    "trades": {},
                    "reason": "nonfinite_position",
                }

            q_old = int(
                np.rint(
                    q_old_raw
                )
            )

            q_target = int(
                np.rint(
                    q_target_raw
                )
            )

            if not np.isclose(
                float(q_old_raw),
                float(q_old),
                atol=tol,
                rtol=0.0,
            ):
                return {
                    "feasible": False,
                    "trades": {},
                    "reason": "noninteger_current_position",
                }

            if not np.isclose(
                float(q_target_raw),
                float(q_target),
                atol=tol,
                rtol=0.0,
            ):
                return {
                    "feasible": False,
                    "trades": {},
                    "reason": "noninteger_target_position",
                }

            limit = int(
                product_spec[
                    "max_abs_contracts"
                ]
            )

            if abs(q_old) > limit:
                return {
                    "feasible": False,
                    "trades": {},
                    "reason": "current_position_over_limit",
                }

            if abs(q_target) > limit:
                return {
                    "feasible": False,
                    "trades": {},
                    "reason": "target_position_over_limit",
                }

            trades[
                product_name
            ] = (
                q_target
                - q_old
            )

    return {
        "feasible": True,
        "trades": trades,
        "reason": "ok",
    }

# ------------------------------------------------------------
# HELPER
# Batched path-dependent contract execution.
#
# Inputs:
#
#   balance:
#       shape (C, S)
#
#   q_old:
#       shape (C, S, K)
#
#   q_target:
#       shape (C, K)
#
#   margin_per_contract:
#       shape (K,)
#
# Output:
#
#   q_new:
#       shape (C, S, K)
#
# This reproduces the frozen execution specification:
#
#   - holding existing exposure is allowed
#   - reductions are allowed
#   - closing is allowed
#   - increases require available margin
#   - reversal closes old side first
#   - new opposite-side exposure uses remaining capacity
#   - balance <= 0 gives zero capacity for increases
#
# ============================================================

def execute_target_contracts_batched_value_only(
    balance,
    q_old,
    q_target,
    margin_per_contract,
    tol=1e-12,
):

    balance = np.asarray(
        balance,
        dtype=float,
    )

    q_old = np.asarray(
        q_old,
        dtype=float,
    )

    q_target = np.asarray(
        q_target,
        dtype=float,
    )

    margin_per_contract = np.asarray(
        margin_per_contract,
        dtype=float,
    )

    if balance.ndim != 2:
        raise ValueError(
            "balance must have shape (C, S)."
        )

    if q_old.ndim != 3:
        raise ValueError(
            "q_old must have shape (C, S, K)."
        )

    C, S = balance.shape

    if q_old.shape != (
        C,
        S,
        K,
    ):
        raise ValueError(
            "q_old shape does not match balance."
        )

    if q_target.shape != (
        C,
        K,
    ):
        raise ValueError(
            "q_target must have shape (C, K)."
        )

    if margin_per_contract.shape != (
        K,
    ):
        raise ValueError(
            "margin_per_contract must have shape (K,)."
        )

    if np.any(
        ~np.isfinite(
            balance
        )
    ):
        raise ValueError(
            "balance must be finite."
        )

    if np.any(
        ~np.isfinite(
            q_old
        )
    ):
        raise ValueError(
            "q_old must be finite."
        )

    if np.any(
        ~np.isfinite(
            q_target
        )
    ):
        raise ValueError(
            "q_target must be finite."
        )

    if np.any(
        ~np.isfinite(
            margin_per_contract
        )
    ):
        raise ValueError(
            "margin_per_contract must be finite."
        )

    if np.any(
        margin_per_contract <= 0.0
    ):
        raise ValueError(
            "margin_per_contract must be positive."
        )


    # --------------------------------------------------------
    # Flatten candidate x simulation into one state dimension.
    #
    # This lets us reproduce the exact production execution
    # branches without constructing derivative arrays.
    # --------------------------------------------------------

    M = (
        C
        * S
    )

    balance_flat = balance.reshape(
        M
    )

    q_old_flat = q_old.reshape(
        M,
        K,
    )

    q_target_flat = np.broadcast_to(
        q_target[
            :,
            None,
            :
        ],
        (
            C,
            S,
            K,
        ),
    ).reshape(
        M,
        K,
    )


    # --------------------------------------------------------
    # Production capacity definition.
    # --------------------------------------------------------

    normal_capacity = np.maximum(
        balance_flat,
        0.0,
    )

    old_margin = np.sum(
        np.abs(
            q_old_flat
        )
        * margin_per_contract[
            None,
            :
        ],
        axis=1,
    )

    target_margin = np.sum(
        np.abs(
            q_target_flat
        )
        * margin_per_contract[
            None,
            :
        ],
        axis=1,
    )


    # --------------------------------------------------------
    # Reproduce production q_base / additions construction.
    # --------------------------------------------------------

    q_base = np.zeros(
        (
            M,
            K,
        ),
        dtype=float,
    )

    additions = np.zeros(
        (
            M,
            K,
        ),
        dtype=float,
    )

    old_zero = (
        np.abs(
            q_old_flat
        )
        <= tol
    )

    target_zero = (
        np.abs(
            q_target_flat
        )
        <= tol
    )

    same_side = (
        q_old_flat
        * q_target_flat
        > 0.0
    )

    same_side_reduction = (
        (~old_zero)
        & (~target_zero)
        & same_side
        & (
            np.abs(
                q_target_flat
            )
            <= np.abs(
                q_old_flat
            )
        )
    )

    same_side_increase = (
        (~old_zero)
        & (~target_zero)
        & same_side
        & (
            np.abs(
                q_target_flat
            )
            > np.abs(
                q_old_flat
            )
        )
    )

    reversal = (
        (~old_zero)
        & (~target_zero)
        & (~same_side)
    )


    # No existing position:
    #
    # base = 0
    # additions = target

    additions[
        old_zero
    ] = q_target_flat[
        old_zero
    ]


    # Same-side reduction:
    #
    # base = target
    # additions = 0

    q_base[
        same_side_reduction
    ] = q_target_flat[
        same_side_reduction
    ]


    # Same-side increase:
    #
    # base = old
    # additions = target - old

    q_base[
        same_side_increase
    ] = q_old_flat[
        same_side_increase
    ]

    additions[
        same_side_increase
    ] = (
        q_target_flat[
            same_side_increase
        ]
        - q_old_flat[
            same_side_increase
        ]
    )


    # Reversal:
    #
    # base = 0
    # additions = target

    additions[
        reversal
    ] = q_target_flat[
        reversal
    ]


    # --------------------------------------------------------
    # Margins after the reduction / close leg and for all
    # desired new exposure.
    # --------------------------------------------------------

    base_margin = np.sum(
        np.abs(
            q_base
        )
        * margin_per_contract[
            None,
            :
        ],
        axis=1,
    )

    desired_new_margin = np.sum(
        np.abs(
            additions
        )
        * margin_per_contract[
            None,
            :
        ],
        axis=1,
    )


    # --------------------------------------------------------
    # Output.
    # --------------------------------------------------------

    q_executed = np.zeros(
        (
            M,
            K,
        ),
        dtype=float,
    )

    assigned = np.zeros(
        M,
        dtype=bool,
    )


    # ========================================================
    # CASE 1
    # Account starts over-margin.
    # ========================================================

    over_margin = (
        old_margin
        > normal_capacity + tol
    )


    # --------------------------------------------------------
    # Still over-margin after all requested permissible
    # reductions.
    #
    # Production behavior:
    #
    #   return q_base
    # --------------------------------------------------------

    mask = (
        over_margin
        & (
            base_margin
            > normal_capacity + tol
        )
    )

    q_executed[
        mask
    ] = q_base[
        mask
    ]

    assigned[
        mask
    ] = True


    # --------------------------------------------------------
    # Reductions restored normal capacity.
    # --------------------------------------------------------

    restored = (
        over_margin
        & (~assigned)
    )

    available = np.maximum(
        normal_capacity
        - base_margin,
        0.0,
    )


    # No desired additions.

    mask = (
        restored
        & (
            desired_new_margin
            <= tol
        )
    )

    q_executed[
        mask
    ] = q_base[
        mask
    ]

    assigned[
        mask
    ] = True


    # All desired additions fit.

    mask = (
        restored
        & (~assigned)
        & (
            desired_new_margin
            <= available + tol
        )
    )

    q_executed[
        mask
    ] = (
        q_base[
            mask
        ]
        + additions[
            mask
        ]
    )

    assigned[
        mask
    ] = True


    # --------------------------------------------------------
    # Desired additions must be clipped.
    #
    # CRITICAL:
    #
    # Production uses ONE COMMON SCALE across all additions.
    # It does not allocate margin future-by-future.
    # --------------------------------------------------------

    mask = (
        restored
        & (~assigned)
    )

    if np.any(
        mask
    ):

        scale = (
            available[
                mask
            ]
            / desired_new_margin[
                mask
            ]
        )

        q_executed[
            mask
        ] = (
            q_base[
                mask
            ]
            + scale[
                :,
                None
            ]
            * additions[
                mask
            ]
        )

        assigned[
            mask
        ] = True


    # ========================================================
    # CASE 2
    # Account starts within normal capacity.
    # ========================================================

    normal_start = (
        ~over_margin
    )


    # --------------------------------------------------------
    # Entire target affordable.
    #
    # Exact production branch:
    #
    #   q_executed = q_target
    # --------------------------------------------------------

    mask = (
        normal_start
        & (
            target_margin
            <= normal_capacity + tol
        )
    )

    q_executed[
        mask
    ] = q_target_flat[
        mask
    ]

    assigned[
        mask
    ] = True


    # --------------------------------------------------------
    # Target not fully affordable.
    # --------------------------------------------------------

    clipped_normal = (
        normal_start
        & (~assigned)
    )


    # No additions after componentwise reduction.

    mask = (
        clipped_normal
        & (
            desired_new_margin
            <= tol
        )
    )

    q_executed[
        mask
    ] = q_base[
        mask
    ]

    assigned[
        mask
    ] = True


    # --------------------------------------------------------
    # Remaining normal-start states require proportional
    # clipping of desired additions.
    # --------------------------------------------------------

    mask = (
        clipped_normal
        & (~assigned)
    )

    if np.any(
        mask
    ):

        available_normal = np.maximum(
            normal_capacity[
                mask
            ]
            - base_margin[
                mask
            ],
            0.0,
        )

        scale = (
            available_normal
            / desired_new_margin[
                mask
            ]
        )

        q_executed[
            mask
        ] = (
            q_base[
                mask
            ]
            + scale[
                :,
                None
            ]
            * additions[
                mask
            ]
        )

        assigned[
            mask
        ] = True


    if not np.all(
        assigned
    ):

        bad = np.flatnonzero(
            ~assigned
        )

        raise RuntimeError(
            "Value-only batched execution failed to assign "
            "all candidate-world states. "
            f"Unassigned indices: {bad[:10]}"
        )


    return q_executed.reshape(
        C,
        S,
        K,
    )

def execute_target_contracts_batched_value_only_fast(
    balance,
    q_old,
    q_target,
    margin_per_contract,
    tol=1e-12,
):
    """
    Faster value-only implementation of the exact same
    production execution rules.

    Keeps candidate x simulation dimensions intact and avoids
    materializing the broadcast target across simulations.
    """

    balance = np.asarray(
        balance,
        dtype=float,
    )

    q_old = np.asarray(
        q_old,
        dtype=float,
    )

    q_target = np.asarray(
        q_target,
        dtype=float,
    )

    margin_per_contract = np.asarray(
        margin_per_contract,
        dtype=float,
    )

    C, S = balance.shape
    K_local = q_old.shape[2]

    if q_old.shape != (
        C,
        S,
        K_local,
    ):
        raise ValueError(
            "q_old shape does not match balance."
        )

    if q_target.shape != (
        C,
        K_local,
    ):
        raise ValueError(
            "q_target must have shape (C, K)."
        )

    if margin_per_contract.shape != (
        K_local,
    ):
        raise ValueError(
            "margin_per_contract must have shape (K,)."
        )

    normal_capacity = np.maximum(
        balance,
        0.0,
    )

    margin_broadcast = (
        margin_per_contract[
            None,
            None,
            :
        ]
    )

    target_3d = q_target[
        :,
        None,
        :
    ]

    old_abs = np.abs(
        q_old
    )

    target_abs = np.abs(
        q_target
    )

    old_margin = np.sum(
        old_abs
        * margin_broadcast,
        axis=2,
    )

    target_margin_candidate = np.sum(
        target_abs
        * margin_per_contract[
            None,
            :
        ],
        axis=1,
    )

    old_zero = (
        old_abs
        <= tol
    )

    target_zero = (
        target_abs
        <= tol
    )[
        :,
        None,
        :
    ]

    same_side = (
        q_old
        * target_3d
        > 0.0
    )

    target_abs_3d = target_abs[
        :,
        None,
        :
    ]

    same_side_reduction = (
        (~old_zero)
        & (~target_zero)
        & same_side
        & (
            target_abs_3d
            <= old_abs
        )
    )

    same_side_increase = (
        (~old_zero)
        & (~target_zero)
        & same_side
        & (
            target_abs_3d
            > old_abs
        )
    )

    reversal = (
        (~old_zero)
        & (~target_zero)
        & (~same_side)
    )

    q_base = np.where(
        same_side_reduction,
        target_3d,
        0.0,
    )

    q_base = np.where(
        same_side_increase,
        q_old,
        q_base,
    )

    additions = np.where(
        old_zero | reversal,
        target_3d,
        0.0,
    )

    additions = np.where(
        same_side_increase,
        target_3d - q_old,
        additions,
    )

    base_margin = np.sum(
        np.abs(
            q_base
        )
        * margin_broadcast,
        axis=2,
    )

    desired_new_margin = np.sum(
        np.abs(
            additions
        )
        * margin_broadcast,
        axis=2,
    )

    available = np.maximum(
        normal_capacity
        - base_margin,
        0.0,
    )

    over_margin = (
        old_margin
        > normal_capacity + tol
    )

    base_still_over = (
        base_margin
        > normal_capacity + tol
    )

    target_affordable = (
        target_margin_candidate[
            :,
            None
        ]
        <= normal_capacity + tol
    )

    no_additions = (
        desired_new_margin
        <= tol
    )

    additions_fit = (
        desired_new_margin
        <= available + tol
    )

    q_executed = q_base.copy()

    # --------------------------------------------------------
    # Over-margin states whose reductions restore capacity.
    # --------------------------------------------------------

    restored = (
        over_margin
        & (~base_still_over)
    )

    restored_full = (
        restored
        & (~no_additions)
        & additions_fit
    )

    q_executed = np.where(
        restored_full[
            :,
            :,
            None
        ],
        q_base + additions,
        q_executed,
    )

    restored_clip = (
        restored
        & (~no_additions)
        & (~additions_fit)
    )

    safe_desired = np.where(
        desired_new_margin > tol,
        desired_new_margin,
        1.0,
    )

    scale = (
        available
        / safe_desired
    )

    q_executed = np.where(
        restored_clip[
            :,
            :,
            None
        ],
        q_base
        + scale[
            :,
            :,
            None
        ]
        * additions,
        q_executed,
    )

    # --------------------------------------------------------
    # Normal-start states.
    # --------------------------------------------------------

    normal_start = (
        ~over_margin
    )

    normal_full = (
        normal_start
        & target_affordable
    )

    q_executed = np.where(
        normal_full[
            :,
            :,
            None
        ],
        target_3d,
        q_executed,
    )

    normal_clip = (
        normal_start
        & (~target_affordable)
        & (~no_additions)
    )

    q_executed = np.where(
        normal_clip[
            :,
            :,
            None
        ],
        q_base
        + scale[
            :,
            :,
            None
        ]
        * additions,
        q_executed,
    )

    return q_executed

# ============================================================
# FULL BATCHED VALUE-ONLY ACCOUNT SIMULATOR
# ============================================================

def simulate_discrete_candidates_value_only(
    candidate_weights,
    continuous_weights,
    raw_return_sims,
    pathwise_notional,
    margin,
    starting_balance,
    initial_contracts,
    reference_balance,
):

    candidate_weights = np.asarray(
        candidate_weights,
        dtype=float,
    )

    continuous_weights = np.asarray(
        continuous_weights,
        dtype=float,
    )

    raw_return_sims = np.asarray(
        raw_return_sims,
        dtype=float,
    )

    pathwise_notional = np.asarray(
        pathwise_notional,
        dtype=float,
    )

    margin = np.asarray(
        margin,
        dtype=float,
    )

    initial_contracts = np.asarray(
        initial_contracts,
        dtype=float,
    )

    C = candidate_weights.shape[
        0
    ]

    S = raw_return_sims.shape[
        0
    ]

    T = raw_return_sims.shape[
        1
    ]

    if candidate_weights.shape != (
        C,
        K,
    ):
        raise ValueError(
            "candidate_weights must have shape (C, K)."
        )

    if continuous_weights.shape != (
        T,
        K,
    ):
        raise ValueError(
            "continuous_weights must have shape (T, K)."
        )

    if pathwise_notional.shape != (
        S,
        T,
        K,
    ):
        raise ValueError(
            "pathwise_notional shape mismatch."
        )

    if margin.shape != (
        T,
        K,
    ):
        raise ValueError(
            "margin shape mismatch."
        )

    if initial_contracts.shape != (
        K,
    ):
        raise ValueError(
            "initial_contracts must have shape (K,)."
        )


    # --------------------------------------------------------
    # Account state
    # --------------------------------------------------------

    balance = np.full(
        (
            C,
            S,
        ),
        float(
            starting_balance
        ),
        dtype=float,
    )

    q_old = np.broadcast_to(
        initial_contracts[
            None,
            None,
            :
        ],
        (
            C,
            S,
            K,
        ),
    ).copy()

    interval_pnl = np.zeros(
        (
            C,
            S,
            T,
        ),
        dtype=float,
    )


    # --------------------------------------------------------
    # Forward simulation
    # --------------------------------------------------------

    execution_runtime = 0.0
    pnl_runtime = 0.0

    for t in range(T):

        if t == 0:

            W_target_t = (
                candidate_weights
            )

        else:

            W_target_t = np.broadcast_to(
                continuous_weights[
                    t,
                    :
                ][
                    None,
                    :
                ],
                (
                    C,
                    K,
                ),
            )

        q_target_t = (
            W_target_t
            * float(
                reference_balance
            )
            / margin[
                t,
                :
            ][
                None,
                :
            ]
        )

        execution_start = time.perf_counter()

        q_new = (
            execute_target_contracts_batched_value_only_fast(
                balance=balance,
                q_old=q_old,
                q_target=q_target_t,
                margin_per_contract=(
                    margin[
                        t,
                        :
                    ]
                ),
                tol=1e-12,
            )
        )

        execution_runtime += (
            time.perf_counter()
            - execution_start
        )

        pnl_start = time.perf_counter()

        pnl_t = np.sum(
            q_new
            * pathwise_notional[
                None,
                :,
                t,
                :
            ]
            * raw_return_sims[
                None,
                :,
                t,
                :
            ],
            axis=2,
        )

        interval_pnl[
            :,
            :,
            t
        ] = pnl_t

        balance = (
            balance
            + pnl_t
        )

        q_old = q_new

        pnl_runtime += (
            time.perf_counter()
            - pnl_start
        )

    print(
        "[STAGE 3 ACCOUNT BREAKDOWN] "
        f"C={C} | "
        f"execution={execution_runtime:.3f}s | "
        f"pnl_update={pnl_runtime:.3f}s | "
        f"total={execution_runtime + pnl_runtime:.3f}s"
    )

    return {
        "terminal_balance": (
            balance
        ),
        "interval_pnl": (
            interval_pnl
        ),
        "final_contracts": (
            q_old
        ),
    }


# ============================================================
# VALUE-ONLY PRIZE EVALUATOR FOR ONE BATCH
# ============================================================

def evaluate_discrete_candidate_batch_value_only(
    candidate_weights,
):

    candidate_weights = np.asarray(
        candidate_weights,
        dtype=float,
    )

    stage3_account_start = time.perf_counter()

    account = (
        simulate_discrete_candidates_value_only(
            candidate_weights=(
                candidate_weights
            ),
            continuous_weights=(
                weights_remaining
            ),
            raw_return_sims=(
                evaluation_R_sims
            ),
            pathwise_notional=(
                evaluation_pathwise_notional
            ),
            margin=(
                evaluation_margin
            ),
            starting_balance=(
                stage3_starting_balance
            ),
            initial_contracts=(
                evaluation_initial_contracts
            ),
            reference_balance=(
                evaluation_reference_balance
            ),
        )
    )

    stage3_account_runtime = (
        time.perf_counter()
        - stage3_account_start
    )

    stage3_prize_start = time.perf_counter()

    C = candidate_weights.shape[
        0
    ]

    total_ev = np.zeros(
        C,
        dtype=float,
    )

    final_ev = np.zeros(
        C,
        dtype=float,
    )

    daily_ev = np.zeros(
        C,
        dtype=float,
    )


    # --------------------------------------------------------
    # Prize-value functions are currently written for one
    # candidate at a time.
    #
    # That is fine: Stage 3A showed prize calculation is tiny.
    # The expensive account simulation is now batched.
    # --------------------------------------------------------

    for c in range(C):

        (
            final_value,
            _,
            _,
        ) = (
            final_prize_value_and_terminal_adjoint(
                terminal_balance=(
                    account[
                        "terminal_balance"
                    ][
                        c,
                        :
                    ]
                ),
                final_place_balance_sims=(
                    evaluation_final_place_balance_sims
                ),
                prizes=prizes,
                current_final_place_balances=(
                    stage3_current_final_place_balances
                ),
                current_rank=(
                    stage3_current_rank
                ),
                win_temperature=0.02,
            )
        )

        if (
            stage3_daily_prize_already_won
            or len(
                stage3_daily_threshold_days
            ) == 0
        ):

            daily_value = 0.0

        else:

            (
                daily_value,
                _,
                _,
            ) = (
                daily_prize_value_and_interval_adjoint(
                    interval_pnl=(
                        account[
                            "interval_pnl"
                        ][
                            c,
                            :,
                            :
                        ]
                    ),
                    B_sims=(
                        evaluation_B_sims
                    ),
                    day_labels=(
                        evaluation_day_labels
                    ),
                    daily_prize=500.0,
                    competitor_temperature=(
                        evaluation_final_daily_temperature
                    ),
                    win_temperature=(
                        evaluation_final_daily_temperature
                    ),
                    eligible_days=(
                        stage3_eligible_days
                    ),
                    daily_threshold_days=(
                        stage3_daily_threshold_days
                    ),
                    current_day=(
                        stage3_current_day
                    ),
                    our_realized_current_day_pnl=(
                        stage3_our_realized_current_day_pnl
                    ),
                    current_daily_rank=(
                        stage3_current_daily_rank
                    ),
                    current_daily_place_pnl=(
                        stage3_current_daily_place_pnl
                    ),
                )
            )

        final_ev[
            c
        ] = float(
            final_value
        )

        daily_ev[
            c
        ] = float(
            daily_value
        )

        total_ev[
            c
        ] = (
            final_ev[
                c
            ]
            + daily_ev[
                c
            ]
        )

    stage3_prize_runtime = (
        time.perf_counter()
        - stage3_prize_start
    )

    print(
        "[STAGE 3D BREAKDOWN] "
        f"C={C} | "
        f"account={stage3_account_runtime:.3f}s | "
        f"prize={stage3_prize_runtime:.3f}s | "
        f"total="
        f"{stage3_account_runtime + stage3_prize_runtime:.3f}s"
    )

    return {
        "total_ev": (
            total_ev
        ),
        "final_ev": (
            final_ev
        ),
        "daily_ev": (
            daily_ev
        ),
        "account": (
            account
        ),
    }
    
# End of Optimizer Cell

In [ ]:
# ============================================================
# REUSABLE OPTIMIZER RUN — ANY t
# ============================================================

# ============================================================
# LIVE RUN CONTROLS
#
# These are the only competition-state controls that are
# manually changed here.
#
# All balances, daily P&Ls, and carried futures positions
# come from current_state.csv.
# ============================================================

# H = 100 settings
maxiter_toggle = 200
n_sims_toggle = 3000
max_joint_candidates_to_keep_toggle = 50
stage3d_batch_size_toggle = 50

# # H = ??? settings
# maxiter_toggle = 250
# n_sims_toggle = 3000
# max_joint_candidates_to_keep_toggle = 250
# stage3d_batch_size_toggle = 50

optimization_horizon_toggle = 2
seed_toggle = 62611481

current_t = 2

daily_prize_already_won = False

# ------------------------------------------------------------
# CURRENT ACTUAL-PRODUCT POSITIONS
#
# Authoritative live record of the actual tradable contracts
# currently carried in the competition account.
#
# Enter INTEGER contract quantities exactly as shown on the
# platform:
#
#   positive = long
#   negative = short
#   zero     = no position
#
# This product-level state is used by the discrete execution
# layer so that per-product +/-10 contract limits can be
# enforced across rebalances.
#
# The continuous optimizer still receives the existing
# underlying-level carried state from current_state.csv.
#
# IMPORTANT:
# For now this is a manual live input. The validation cell
# below will verify that these actual products reconstruct the
# same underlying exposure as current_state.csv.
# ------------------------------------------------------------

current_actual_product_positions = {
    "CL": 0,
    "QM": 0,
    "MCL": 0,
    "GC": 0,
    "MGC": 0,
    "1OZ": 0,
    "SR3": 0,
    "ZN": 0,
    "KE": 0,
    "HO": 0,
    "ZW": 0,
    "MZW": 0,
    "NQ": 0,
    "MNQ": 0,
}

# ------------------------------------------------------------
# FUTURE IDENTITIES AND EXECUTION CONTROLS
#
# The optimizer's contract quantity is measured in units of
# the implementation contract associated with each Future_k.
#
# Examples:
#   Future_1 CL -> MCL
#
#   Future_2 GC -> MGC
#       1.0 optimizer contract = 1 MGC
#       0.1 optimizer contract = 1 1OZ contract
#
#   Future_8 NQ -> MNQ
#       1.0 optimizer contract = 1 MNQ
#
# execution_increment is therefore measured in IMPLEMENTATION-
# CONTRACT units, not standard-contract equivalents.
#
# IMPORTANT:
# This rounding is display-only. It does not yet solve the
# discrete portfolio optimization problem or enforce total
# executable margin after rounding.
# ------------------------------------------------------------

underlying_symbols = {
    "Future_1": "CL",
    "Future_2": "GC",
    "Future_3": "SR3",
    "Future_4": "ZN",
    "Future_5": "KE",
    "Future_6": "HO",
    "Future_7": "ZW",
    "Future_8": "NQ",
}

implementation_symbols = {
    "Future_1": "MCL",
    "Future_2": "MGC",
    "Future_3": "SR3",
    "Future_4": "ZN",
    "Future_5": "KE",
    "Future_6": "HO",
    "Future_7": "ZW",
    "Future_8": "MNQ",
}

# Standard-contract-equivalent display conversion.
#
# The optimizer continues to work entirely in implementation-
# contract units. These factors are used ONLY to make the live
# recommendation easier to read consistently.
#
# Examples:
#   10 MCL = 1 CL
#   10 MGC = 1 GC
#   10 MNQ = 1 NQ
#
# Direct implementation contracts already equal one standard
# contract, so their factor is 1.

standard_contract_display_factor = {
    "Future_1": 0.10,
    "Future_2": 0.10,
    "Future_3": 1.00,
    "Future_4": 1.00,
    "Future_5": 1.00,
    "Future_6": 1.00,
    "Future_7": 1.00,
    "Future_8": 0.10,
}

execution_increment = {
    "Future_1": 1.00,
    "Future_2": 0.10,
    "Future_3": 1.00,
    "Future_4": 1.00,
    "Future_5": 1.00,
    "Future_6": 1.00,
    "Future_7": 0.10,
    "Future_8": 1.00,
}

# Maximum absolute exposure to each underlying, expressed in
# standard-contract-equivalent units.
#
# The competition permits at most 10 contracts long or short
# in EACH individual tradable contract product.
#
# Therefore:
#   CL:  10 CL + 10 QM + 10 MCL
#        = 10*1.0 + 10*0.5 + 10*0.1 = 16.0 CL
#
#   GC:  10 GC + 10 MGC + 10 1OZ
#        = 10*1.0 + 10*0.1 + 10*0.01 = 11.1 GC
#
#   SR3: 10 SR3 = 10.0 SR3
#   ZN:  10 ZN  = 10.0 ZN
#   KE:  10 KE  = 10.0 KE
#   HO:  10 HO  = 10.0 HO
#
#   ZW:  10 ZW + 10 MZW
#        = 10*1.0 + 10*0.1 = 11.0 ZW
#
#   NQ:  10 NQ + 10 MNQ
#        = 10*1.0 + 10*0.1 = 11.0 NQ
#
# These are USER CONTROLS. Change them here if the competition's
# permitted contract products or per-product contract limits change.

standard_equivalent_contract_limit = {
    "Future_1": 16.0,
    "Future_2": 11.1,
    "Future_3": 10.0,
    "Future_4": 10.0,
    "Future_5": 10.0,
    "Future_6": 10.0,
    "Future_7": 11.0,
    "Future_8": 11.0,
}

# ------------------------------------------------------------
# ACTUAL TRADABLE CONTRACT PRODUCTS
#
# implementation_units:
#   Number of canonical implementation contracts represented
#   by one actual tradable contract.
#
# standard_equivalent:
#   Number of standard contracts represented by one actual
#   tradable contract.
#
# max_abs_contracts:
#   Competition position limit for that individual product.
#
# Product margin is derived later from:
#
#   implementation_units * canonical implementation margin
#
# This keeps the discrete layer consistent with the existing
# production margin methodology.
# ------------------------------------------------------------

tradable_contract_products = {
    "Future_1": {
        "CL": {
            "implementation_units": 10.0,
            "standard_equivalent": 1.0,
            "max_abs_contracts": 10,
        },
        "QM": {
            "implementation_units": 5.0,
            "standard_equivalent": 0.5,
            "max_abs_contracts": 10,
        },
        "MCL": {
            "implementation_units": 1.0,
            "standard_equivalent": 0.1,
            "max_abs_contracts": 10,
        },
    },
    "Future_2": {
        "GC": {
            "implementation_units": 10.0,
            "standard_equivalent": 1.0,
            "max_abs_contracts": 10,
        },
        "MGC": {
            "implementation_units": 1.0,
            "standard_equivalent": 0.1,
            "max_abs_contracts": 10,
        },
        "1OZ": {
            "implementation_units": 0.1,
            "standard_equivalent": 0.01,
            "max_abs_contracts": 10,
        },
    },
    "Future_3": {
        "SR3": {
            "implementation_units": 1.0,
            "standard_equivalent": 1.0,
            "max_abs_contracts": 10,
        },
    },
    "Future_4": {
        "ZN": {
            "implementation_units": 1.0,
            "standard_equivalent": 1.0,
            "max_abs_contracts": 10,
        },
    },
    "Future_5": {
        "KE": {
            "implementation_units": 1.0,
            "standard_equivalent": 1.0,
            "max_abs_contracts": 10,
        },
    },
    "Future_6": {
        "HO": {
            "implementation_units": 1.0,
            "standard_equivalent": 1.0,
            "max_abs_contracts": 10,
        },
    },
    "Future_7": {
        "ZW": {
            "implementation_units": 1.0,
            "standard_equivalent": 1.0,
            "max_abs_contracts": 10,
        },
        "MZW": {
            "implementation_units": 0.1,
            "standard_equivalent": 0.1,
            "max_abs_contracts": 10,
        },
    },
    "Future_8": {
        "NQ": {
            "implementation_units": 10.0,
            "standard_equivalent": 1.0,
            "max_abs_contracts": 10,
        },
        "MNQ": {
            "implementation_units": 1.0,
            "standard_equivalent": 0.1,
            "max_abs_contracts": 10,
        },
    },
}

# ------------------------------------------------------------
# INPUT FOLDER
# ------------------------------------------------------------

ACTIVE_FORECAST_DIR = str(
    (Path.cwd().parent if Path.cwd().name == "optimizer" else Path.cwd())
    / "examples"
    / "sample_inputs"
)

CURRENT_STATE_FILE = os.path.join(
    ACTIVE_FORECAST_DIR,
    "current_state.csv",
)


# ------------------------------------------------------------
# STEP 7C - LOAD FILE-BASED DISTRIBUTIONAL SYSTEM
#
# These are reconstructed entirely from the serialized practice
# files written by Step 7A.
#
# The reusable run cell therefore does not depend on the
# PRACTICE_HETEROGENEOUS_* in-memory objects created by the
# practice-generator cell.
# ------------------------------------------------------------

practice_innovation_system = (
    load_python_innovation_system(
        ACTIVE_FORECAST_DIR
    )
)

practice_mapping_system = (
    load_python_arithmetic_mapping(
        ACTIVE_FORECAST_DIR
    )
)


print(
    "Loaded production distributional system from files: TRUE"
)


# ------------------------------------------------------------
# COMPETITION SETUP
# ------------------------------------------------------------

T = 147
K = 8

prizes = np.array([
    750.0,
    500.0,
    1150.0,
    100.0,
])

day_labels = np.array(
    ["Sunday"] * 14
    + ["Monday"] * 31
    + ["Tuesday"] * 31
    + ["Wednesday"] * 31
    + ["Thursday"] * 31
    + ["Friday"] * 9
)


# ------------------------------------------------------------
# VALIDATE current_t
# ------------------------------------------------------------

if not (0 <= current_t < T):
    raise ValueError(
        f"current_t must be between 0 and {T - 1}."
    )

current_day = day_labels[current_t]

print(
    f"Running optimizer for t={current_t} "
    f"({current_day})"
)


# ------------------------------------------------------------
# BUILD CURRENT-STATE ARGUMENTS
# ------------------------------------------------------------

state_kwargs = {}

state_df = pd.read_csv(
    CURRENT_STATE_FILE
)

if len(state_df) != 1:
    raise ValueError(
        "current_state.csv must contain "
        "exactly one row."
    )

state = state_df.iloc[0]


# At t=0, run_single_problem automatically uses:
#
#   our balance = $25,000
#   all six overall place balances = $25,000
#   daily P&L state = $0
#
# Therefore we do not need current_state.csv.

if current_t == 0:

    # At competition start there are no carried positions.
    current_position_weights = np.zeros(
        K,
        dtype=float,
    )

else:

    # --------------------------------------------------------
    # CURRENT CARRIED FUTURES POSITIONS
    # --------------------------------------------------------

    weight_columns = [
        f"weight_Future_{i}"
        for i in range(1, K + 1)
    ]

    missing_weight_columns = [
        column
        for column in weight_columns
        if column not in state_df.columns
    ]

    if missing_weight_columns:
        raise ValueError(
            "current_state.csv is missing "
            "position-weight columns: "
            f"{missing_weight_columns}"
        )

    current_position_weights = np.array(
        [
            state[column]
            for column in weight_columns
        ],
        dtype=float,
    )

    # --------------------------------------------------------
    # OUR CURRENT ACCOUNT BALANCE
    # --------------------------------------------------------

    state_kwargs[
        "current_balance"
    ] = float(
        state["our_balance"]
    )

    # --------------------------------------------------------
    # OUR CURRENT OVERALL RANK
    # --------------------------------------------------------

    rank_value = float(
        state["our_rank"]
    )

    if (
        not np.isfinite(rank_value)
        or rank_value < 1
        or not rank_value.is_integer()
    ):
        raise ValueError(
            "our_rank in current_state.csv "
            "must be a positive integer."
        )

    state_kwargs[
        "current_rank"
    ] = int(
        rank_value
    )

    # --------------------------------------------------------
    # CURRENT OVERALL LEADERBOARD BALANCES
    #
    # [1st, 2nd, 3rd, 4th, 15th, 16th]
    # --------------------------------------------------------

    state_kwargs[
        "current_final_place_balances"
    ] = np.array([
        state["balance_1st"],
        state["balance_2nd"],
        state["balance_3rd"],
        state["balance_4th"],
        state["balance_15th"],
        state["balance_16th"],
    ], dtype=float)

    # --------------------------------------------------------
    # DAILY-PRIZE STATE
    #
    # Only relevant Monday through Thursday.
    # Sunday and Friday are ignored.
    # --------------------------------------------------------

    if current_day in [
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
    ]:

        state_kwargs[
            "our_current_daily_pnl"
        ] = float(
            state["our_daily_pnl"]
        )

        daily_rank_value = float(
            state["our_daily_rank"]
        )

        if (
            not np.isfinite(daily_rank_value)
            or daily_rank_value < 1
            or not daily_rank_value.is_integer()
        ):
            raise ValueError(
                "our_daily_rank in current_state.csv "
                "must be a positive integer."
            )

        state_kwargs[
            "current_daily_rank"
        ] = int(
            daily_rank_value
        )

        state_kwargs[
            "current_daily_place_pnl"
        ] = np.array([
            state["daily_1st_pnl"],
            state["daily_2nd_pnl"],
        ], dtype=float)

# ------------------------------------------------------------
# CURRENT CONTRACT NOTIONAL
#
# Dollar notional of one contract for each future at the
# beginning of the current optimization horizon.
# ------------------------------------------------------------

notional_columns = [
    f"notional_Future_{i}"
    for i in range(1, K + 1)
]

missing_notional_columns = [
    column
    for column in notional_columns
    if column not in state_df.columns
]

if missing_notional_columns:
    raise ValueError(
        "current_state.csv is missing "
        "contract-notional columns: "
        f"{missing_notional_columns}"
    )

current_contract_notional = np.array(
    [
        state[column]
        for column in notional_columns
    ],
    dtype=float,
)

if current_contract_notional.shape != (K,):
    raise ValueError(
        "current_contract_notional must have "
        f"shape ({K},)."
    )

if not np.all(
    np.isfinite(
        current_contract_notional
    )
):
    raise ValueError(
        "current_contract_notional must contain "
        "only finite values."
    )

if np.any(
    current_contract_notional <= 0.0
):
    raise ValueError(
        "All current contract notionals "
        "must be positive."
    )

# ------------------------------------------------------------
# VALIDATE CURRENT CARRIED POSITIONS
# ------------------------------------------------------------

current_position_weights = np.asarray(
    current_position_weights,
    dtype=float,
)

if current_position_weights.shape != (K,):
    raise ValueError(
        f"current_position_weights must have shape ({K},)."
    )

if not np.all(
    np.isfinite(
        current_position_weights
    )
):
    raise ValueError(
        "current_position_weights must contain "
        "only finite values."
    )

current_position_gross = float(
    np.sum(
        np.abs(
            current_position_weights
        )
    )
)


# ------------------------------------------------------------
# SHOW CURRENT STATE BEING USED
# ------------------------------------------------------------

print(
    "Current carried position weights:"
)

display(
    pd.Series(
        current_position_weights,
        index=[
            f"Future_{i}"
            for i in range(1, K + 1)
        ],
        name=f"Position entering t={current_t}",
    )
)

print(
    f"Current gross carried weight: "
    f"{current_position_gross:.4f}"
)

print(
    "Current contract notionals:"
)

display(
    pd.Series(
        current_contract_notional,
        index=[
            f"Future_{i}"
            for i in range(1, K + 1)
        ],
        name=f"Notional entering t={current_t}",
    )
)

if current_t == 0:

    print(
        "Using automatic competition-start state."
    )

else:

    print(
        f"Our current overall rank: "
        f"{state_kwargs['current_rank']}"
    )
    
    print(
        f"Our current balance: "
        f"${state_kwargs['current_balance']:,.2f}"
    )

    print(
        "Overall place balances:",
        state_kwargs[
            "current_final_place_balances"
        ],
    )

    if (
        "our_current_daily_pnl"
        in state_kwargs
    ):

        print(
            f"Our current daily P&L: "
            f"${state_kwargs['our_current_daily_pnl']:,.2f}"
        )

        print(
            f"Our current daily rank: "
            f"{state_kwargs['current_daily_rank']}"
        )

        print(
            "Daily 1st/2nd P&Ls:",
            state_kwargs[
                "current_daily_place_pnl"
            ],
        )

    else:

        print(
            "Daily P&L state not required "
            f"on {current_day}."
        )

# ------------------------------------------------------------
# RUN OPTIMIZER
# ------------------------------------------------------------

results = run_single_problem(
    T=T,
    K=K,
    prizes=prizes,
    n_starts=1,
    weight_bound=1.0,

    standard_equivalent_contract_limit=np.array(
        [
            standard_equivalent_contract_limit[
                f"Future_{i}"
            ]
            for i in range(
                1,
                K + 1
            )
        ],
        dtype=float,
    ),
    standard_contract_display_factor=np.array(
        [
            standard_contract_display_factor[
                f"Future_{i}"
            ]
            for i in range(
                1,
                K + 1
            )
        ],
        dtype=float,
    ),
    
    seed=seed_toggle,
    maxiter=maxiter_toggle,
    day_labels=day_labels,
    daily_prize=500.0,
    daily_prize_already_won=daily_prize_already_won,

    current_t=current_t,
    optimization_horizon=optimization_horizon_toggle,
    current_weights=current_position_weights,
    current_contract_notional=current_contract_notional,
    
    n_sims=n_sims_toggle,
    simulation_model="distributional",
    distributional_innovation_system=practice_innovation_system,
    distributional_mapping_system=practice_mapping_system,
    daily_temperature_schedule=(10.0,),

    **state_kwargs,
)


# ------------------------------------------------------------
# FULL OPTIMIZED FUTURE WEIGHT PATH
#
# This is the complete open-loop plan selected using the
# information available at current_t.
#
# In live use, only the first row is the actionable current
# recommendation. Future rows are displayed for inspection;
# the optimizer should be rerun as new information arrives.
# ------------------------------------------------------------

weights_remaining = np.asarray(
    results.iloc[0]["weights_remaining"],
    dtype=float,
)

remaining_time_indices = np.asarray(
    results.iloc[0]["remaining_time_indices"],
    dtype=int,
)

if weights_remaining.shape != (
    len(
        remaining_time_indices
    ),
    K,
):
    raise ValueError(
        "weights_remaining has an unexpected shape."
    )

future_weight_path = pd.DataFrame(
    weights_remaining,
    columns=[
        f"Future_{i}"
        for i in range(1, K + 1)
    ],
)

future_weight_path.insert(
    0,
    "Day",
    [
        day_labels[t]
        for t in remaining_time_indices
    ],
)

future_weight_path.insert(
    0,
    "Global_t",
    remaining_time_indices,
)

future_weight_path[
    "Gross"
] = (
    future_weight_path[
        [
            f"Future_{i}"
            for i in range(1, K + 1)
        ]
    ]
    .abs()
    .sum(
        axis=1
    )
)

print(
    "\nFULL OPTIMIZED FUTURE WEIGHT PATH"
)

with pd.option_context(
    "display.max_rows",
    None,
):
    display(
        future_weight_path
    )


# ------------------------------------------------------------
# FINAL ACTIONABLE RECOMMENDATION
# ------------------------------------------------------------

current_recommended_weights = (
    weights_remaining[
        0,
        :
    ]
)

current_recommended_gross = float(
    np.sum(
        np.abs(
            current_recommended_weights
        )
    )
)

print(
    "\n"
    + "=" * 70
)

print(
    "NEXT-PERIOD RECOMMENDATION"
)

print(
    "=" * 70
)

print(
    f"Decision interval global t: "
    f"{remaining_time_indices[0]}"
)

print(
    f"Day: "
    f"{day_labels[remaining_time_indices[0]]}"
)

# ------------------------------------------------------------
# BUILD EXECUTION-FRIENDLY RECOMMENDATION TABLE
# ------------------------------------------------------------

future_names = [
    f"Future_{i}"
    for i in range(1, K + 1)
]

if set(underlying_symbols.keys()) != set(future_names):
    raise ValueError(
        "underlying_symbols must contain exactly "
        "Future_1 through Future_8."
    )

if set(execution_increment.keys()) != set(future_names):
    raise ValueError(
        "execution_increment must contain exactly "
        "Future_1 through Future_8."
    )

if any(
    (
        not np.isfinite(execution_increment[name])
        or execution_increment[name] <= 0.0
    )
    for name in future_names
):
    raise ValueError(
        "Every contract increment must be finite and positive."
    )

margin_full = load_margin_forecasts(
    T=T,
    K=K,
)

current_margin = margin_full[
    current_t,
    :
]

if not np.all(
    np.isfinite(
        current_margin
    )
):
    raise ValueError(
        "Current-period margins must all be finite."
    )

if np.any(
    current_margin <= 0.0
):
    raise ValueError(
        "Current-period margins must all be positive."
    )

if current_t == 0:
    execution_balance = 25000.0
else:
    execution_balance = float(
        state_kwargs["current_balance"]
    )

raw_implementation_contracts = (
    execution_balance
    * current_recommended_weights
    / current_margin
)

rounded_implementation_contracts = np.array(
    [
        (
            round(
                raw_implementation_contracts[i]
                / execution_increment[future_names[i]]
            )
            * execution_increment[future_names[i]]
        )
        for i in range(K)
    ],
    dtype=float,
)

rounded_margin_dollars = (
    np.abs(
        rounded_implementation_contracts
    )
    * current_margin
)

rounded_margin_weight = (
    rounded_margin_dollars
    / execution_balance
)

raw_standard_contracts = np.array(
    [
        (
            raw_implementation_contracts[i]
            * standard_contract_display_factor[future_names[i]]
        )
        for i in range(K)
    ],
    dtype=float,
)

rounded_standard_contracts = np.array(
    [
        (
            rounded_implementation_contracts[i]
            * standard_contract_display_factor[future_names[i]]
        )
        for i in range(K)
    ],
    dtype=float,
)

recommendation_table = pd.DataFrame(
    {
        "Future": future_names,
        "Contract": [
            underlying_symbols[name]
            for name in future_names
        ],
        "Weight": [
            f"{100.0 * weight:.2f}%"
            for weight in current_recommended_weights
        ],
        "Raw Contract Amount": raw_standard_contracts,
        "Executable Contract Amount": rounded_standard_contracts,
        "Executable Margin Weight": [
            f"{100.0 * weight:.2f}%"
            for weight in rounded_margin_weight
        ],
    }
)

recommendation_table["Raw Contract Amount"] = (
    recommendation_table["Raw Contract Amount"]
    .map(
        lambda x: f"{x:.3f}"
    )
)

recommendation_table["Executable Contract Amount"] = (
    recommendation_table["Executable Contract Amount"]
    .map(
        lambda x: f"{x:.2f}"
    )
)

display(
    recommendation_table
)

print(
    f"Recommended gross exposure: "
    f"{current_recommended_gross:.6f}"
)


# ------------------------------------------------------------
# EXPECTED PRIZE
# ------------------------------------------------------------

expected_prize = float(
    results.iloc[0][
        "expected_prize"
    ]
)

final_expected_prize = float(
    results.iloc[0][
        "final_expected_prize"
    ]
)

daily_expected_prize = float(
    results.iloc[0][
        "daily_expected_prize"
    ]
)

hard_daily_expected_prize = float(
    results.iloc[0][
        "hard_daily_expected_prize"
    ]
)

print(
    "\nEXPECTED PRIZE"
)

print(
    f"Total expected prize: "
    f"${expected_prize:,.2f}"
)

print(
    f"Final-leaderboard expected prize: "
    f"${final_expected_prize:,.2f}"
)

print(
    f"Daily expected prize: "
    f"${daily_expected_prize:,.2f}"
)

print(
    f"Hard daily expected prize: "
    f"${hard_daily_expected_prize:,.2f}"
)


# ------------------------------------------------------------
# OPTIMIZER STATUS
# ------------------------------------------------------------

print(
    "\nOPTIMIZER STATUS"
)

print(
    f"Converged: "
    f"{bool(results.iloc[0]['success'])}"
)

print(
    f"Iterations: "
    f"{int(results.iloc[0]['iterations'])}"
)

print(
    f"Function evaluations: "
    f"{int(results.iloc[0]['function_evals'])}"
)

print(
    f"Optimizer runtime: "
    f"{float(results.iloc[0]['runtime_sec']):,.2f} seconds"
)

print(
    f"Message: "
    f"{results.iloc[0]['message']}"
)

# ============================================================
# DISCRETE EXECUTION - LIVE PRODUCTION PIPELINE
# ============================================================
#
# Everything below this point converts the continuous optimum
# into the best currently executable actual-product portfolio
# on the SAME Monte Carlo worlds used by run_single_problem().
# ============================================================


# ------------------------------------------------------------
# EXTRACT AUTHORITATIVE CONTINUOUS EVALUATION STATE
# ------------------------------------------------------------

best_result = results.iloc[0]

continuous_current_weights = np.asarray(
    best_result[
        "current_weights"
    ],
    dtype=float,
).copy()

evaluation_R_sims = np.asarray(
    best_result[
        "evaluation_R_sims"
    ],
    dtype=float,
)

evaluation_pathwise_notional = np.asarray(
    best_result[
        "evaluation_pathwise_notional"
    ],
    dtype=float,
)

evaluation_margin = np.asarray(
    best_result[
        "evaluation_margin"
    ],
    dtype=float,
)

evaluation_initial_contracts = np.asarray(
    best_result[
        "evaluation_initial_contracts"
    ],
    dtype=float,
)

evaluation_B_sims = np.asarray(
    best_result[
        "evaluation_B_sims"
    ],
    dtype=float,
)

evaluation_final_place_balance_sims = np.asarray(
    best_result[
        "evaluation_final_place_balance_sims"
    ],
    dtype=float,
)

evaluation_day_labels = np.asarray(
    best_result[
        "evaluation_day_labels"
    ],
    dtype=object,
)

evaluation_reference_balance = float(
    best_result[
        "evaluation_reference_balance"
    ]
)

evaluation_final_daily_temperature = float(
    best_result[
        "evaluation_final_daily_temperature"
    ]
)


# ------------------------------------------------------------
# RECONSTRUCT CURRENT COMPETITION STATE
# ------------------------------------------------------------

stage3_starting_balance = float(
    best_result[
        "future_starting_balance"
    ]
)

stage3_current_final_place_balances = np.asarray(
    best_result[
        "current_final_place_balances"
    ],
    dtype=float,
).copy()

stage3_current_rank_raw = best_result[
    "current_rank"
]

stage3_current_rank = (
    None
    if stage3_current_rank_raw is None
    else int(stage3_current_rank_raw)
)

stage3_current_day = best_result[
    "current_day"
]

stage3_our_realized_current_day_pnl = float(
    best_result[
        "our_realized_current_day_pnl"
    ]
)

stage3_current_daily_rank = (
    None
    if best_result["current_daily_rank"] is None
    else int(
        best_result[
            "current_daily_rank"
        ]
    )
)

stage3_current_daily_place_pnl = np.asarray(
    best_result[
        "current_daily_place_pnl"
    ],
    dtype=float,
).copy()

stage3_daily_threshold_days = best_result[
    "active_daily_threshold_days"
]

stage3_daily_prize_already_won = bool(
    best_result[
        "daily_prize_already_won"
    ]
)

stage3_eligible_days = (
    None
    if stage3_daily_prize_already_won
    else stage3_daily_threshold_days
)


# ------------------------------------------------------------
# CURRENT-PERIOD DISCRETE REFERENCE STATE
# ------------------------------------------------------------

future_names_product_test = [
    f"Future_{i}"
    for i in range(
        1,
        K + 1
    )
]

margin_full = load_margin_forecasts(
    T=T,
    K=K,
)

current_margin_discrete = np.asarray(
    margin_full[
        current_t,
        :
    ],
    dtype=float,
)

if current_t == 0:
    discrete_reference_balance = 25000.0
else:
    discrete_reference_balance = float(
        state_kwargs[
            "current_balance"
        ]
    )

if not np.isclose(
    discrete_reference_balance,
    evaluation_reference_balance,
    rtol=0.0,
    atol=1e-10,
):
    raise RuntimeError(
        "Discrete reference balance does not match "
        "the optimizer evaluation reference balance."
    )


# ------------------------------------------------------------
# CONTINUOUS BENCHMARK
# ------------------------------------------------------------

stage3d_continuous_ev = float(
    best_result[
        "expected_prize"
    ]
)

stage3d_continuous_final_ev = float(
    best_result[
        "final_expected_prize"
    ]
)

stage3d_continuous_daily_ev = float(
    best_result[
        "daily_expected_prize"
    ]
)


print(
    "\nDiscrete production state loaded from "
    "run_single_problem(): TRUE"
)

if current_t == 0:

    discrete_reference_balance = (
        25000.0
    )

else:

    discrete_reference_balance = float(
        state_kwargs[
            "current_balance"
        ]
    )

current_margin_discrete = np.asarray(
    margin_full[
        current_t,
        :
    ],
    dtype=float,
)

# ============================================================
# CURRENT ACTUAL-PRODUCT POSITION VALIDATION
# ============================================================
#
# This cell validates the manually entered actual-product
# positions and converts them into the canonical implementation
# contract quantities used by the existing optimizer.
#
# It does NOT change the continuous optimizer.
# ============================================================


# ------------------------------------------------------------
# BUILD PRODUCT -> FUTURE LOOKUP
# ------------------------------------------------------------

actual_product_to_future = {}

for future_name in future_names_product_test:

    for product_name in tradable_contract_products[
        future_name
    ]:

        if product_name in actual_product_to_future:
            raise ValueError(
                f"Duplicate actual product name: {product_name}."
            )

        actual_product_to_future[
            product_name
        ] = future_name


expected_actual_products = set(
    actual_product_to_future.keys()
)

entered_actual_products = set(
    current_actual_product_positions.keys()
)

if entered_actual_products != expected_actual_products:

    missing_products = sorted(
        expected_actual_products
        - entered_actual_products
    )

    extra_products = sorted(
        entered_actual_products
        - expected_actual_products
    )

    raise ValueError(
        "current_actual_product_positions does not exactly "
        "match tradable_contract_products. "
        f"Missing={missing_products}, "
        f"Extra={extra_products}."
    )


# ------------------------------------------------------------
# VALIDATE INTEGER PRODUCT POSITIONS AND PRODUCT LIMITS
# ------------------------------------------------------------

current_actual_product_positions_validated = {}

for product_name in sorted(
    expected_actual_products
):

    q_raw = current_actual_product_positions[
        product_name
    ]

    if not np.isfinite(
        q_raw
    ):
        raise ValueError(
            f"{product_name}: position must be finite."
        )

    q_rounded = int(
        np.rint(
            q_raw
        )
    )

    if not np.isclose(
        float(q_raw),
        float(q_rounded),
        rtol=0.0,
        atol=1e-12,
    ):
        raise ValueError(
            f"{product_name}: actual-product position "
            "must be an integer."
        )

    future_name = actual_product_to_future[
        product_name
    ]

    max_abs_contracts = int(
        tradable_contract_products[
            future_name
        ][
            product_name
        ][
            "max_abs_contracts"
        ]
    )

    if abs(
        q_rounded
    ) > max_abs_contracts:
        raise ValueError(
            f"{product_name}: position {q_rounded} exceeds "
            f"+/-{max_abs_contracts} product limit."
        )

    current_actual_product_positions_validated[
        product_name
    ] = q_rounded


# ------------------------------------------------------------
# RECONSTRUCT CANONICAL IMPLEMENTATION-CONTRACT POSITIONS
# ------------------------------------------------------------

current_actual_q_implementation = np.zeros(
    K,
    dtype=float,
)

current_actual_standard_equivalent = np.zeros(
    K,
    dtype=float,
)

current_actual_product_rows = []

for k, future_name in enumerate(
    future_names_product_test
):

    products = tradable_contract_products[
        future_name
    ]

    for product_name, product_spec in products.items():

        q_product = int(
            current_actual_product_positions_validated[
                product_name
            ]
        )

        implementation_units = float(
            product_spec[
                "implementation_units"
            ]
        )

        standard_equivalent = float(
            product_spec[
                "standard_equivalent"
            ]
        )

        implementation_position = (
            q_product
            * implementation_units
        )

        standard_position = (
            q_product
            * standard_equivalent
        )

        current_actual_q_implementation[
            k
        ] += implementation_position

        current_actual_standard_equivalent[
            k
        ] += standard_position

        current_actual_product_rows.append(
            {
                "Future": future_name,
                "Underlying": underlying_symbols[
                    future_name
                ],
                "Product": product_name,
                "Actual Contracts": q_product,
                "Implementation Units Per Product": (
                    implementation_units
                ),
                "Implementation Position": (
                    implementation_position
                ),
                "Standard Equivalent Position": (
                    standard_position
                ),
            }
        )


# ------------------------------------------------------------
# RECONSTRUCT UNDERLYING WEIGHTS
#
# Use the exact same canonical relationship as the existing
# optimizer:
#
#   weight = q_implementation * margin / balance
# ------------------------------------------------------------

current_actual_reconstructed_weights = (
    current_actual_q_implementation
    * current_margin_discrete
    / discrete_reference_balance
)


# ------------------------------------------------------------
# COMPARE AGAINST EXISTING CURRENT-STATE PIPELINE
# ------------------------------------------------------------

current_state_q_implementation = (
    np.asarray(
        current_position_weights,
        dtype=float,
    )
    * discrete_reference_balance
    / current_margin_discrete
)

current_actual_q_error = (
    current_actual_q_implementation
    - current_state_q_implementation
)

current_actual_weight_error = (
    current_actual_reconstructed_weights
    - np.asarray(
        current_position_weights,
        dtype=float,
    )
)


current_actual_validation_rows = []

for k, future_name in enumerate(
    future_names_product_test
):

    current_actual_validation_rows.append(
        {
            "Future": future_name,
            "Underlying": underlying_symbols[
                future_name
            ],
            "Actual Product Impl Q": float(
                current_actual_q_implementation[
                    k
                ]
            ),
            "Current State Impl Q": float(
                current_state_q_implementation[
                    k
                ]
            ),
            "Q Error": float(
                current_actual_q_error[
                    k
                ]
            ),
            "Actual Product Weight": float(
                current_actual_reconstructed_weights[
                    k
                ]
            ),
            "Current State Weight": float(
                current_position_weights[
                    k
                ]
            ),
            "Weight Error": float(
                current_actual_weight_error[
                    k
                ]
            ),
        }
    )


current_actual_product_table = pd.DataFrame(
    current_actual_product_rows
)

current_actual_validation_table = pd.DataFrame(
    current_actual_validation_rows
)


print(
    "CURRENT ACTUAL-PRODUCT POSITIONS"
)

display(
    current_actual_product_table
)


print(
    "\nACTUAL-PRODUCT VS CURRENT-STATE RECONSTRUCTION"
)

display(
    current_actual_validation_table
)


max_current_actual_q_error = float(
    np.max(
        np.abs(
            current_actual_q_error
        )
    )
)

max_current_actual_weight_error = float(
    np.max(
        np.abs(
            current_actual_weight_error
        )
    )
)


print(
    "\nMaximum implementation-contract error: "
    f"{max_current_actual_q_error:.16e}"
)

print(
    "Maximum weight error: "
    f"{max_current_actual_weight_error:.16e}"
)


# ------------------------------------------------------------
# VALIDATE CURRENT-STATE / ACTUAL-PRODUCT CONSISTENCY
#
# Stage 7 builds current_state.csv weights from the exact
# source-margin commitment:
#
#   weight = ContractsHeld * source_margin / balance
#
# For micro implementation contracts (CL -> MCL, GC -> MGC,
# NQ -> MNQ), Stage 7 writes margin.csv after scaling the
# standard margin by 0.10 and applying ceiling().
#
# Therefore the current-state weight and the actual-product
# reconstruction can differ slightly even when the carried
# position is exactly correct.
#
# The ceiling changes implementation margin by strictly less
# than $1 per implementation contract. Thus the maximum
# legitimate weight discrepancy for each future is:
#
#   abs(q_implementation) * $1 / balance
#
# Direct contracts have no ceiling conversion and retain the
# strict floating-point tolerance.
# ------------------------------------------------------------

current_actual_weight_tolerance = np.full(
    len(future_names_product_test),
    1e-10,
    dtype=float,
)

for k, future_name in enumerate(
    future_names_product_test
):
    if future_name in {
        "Future_1",
        "Future_2",
        "Future_8",
    }:
        current_actual_weight_tolerance[k] = (
            abs(
                current_actual_q_implementation[k]
            )
            / discrete_reference_balance
            + 1e-10
        )


current_actual_weight_error_abs = np.abs(
    current_actual_weight_error
)

current_actual_weight_validation_pass = (
    current_actual_weight_error_abs
    <= current_actual_weight_tolerance
)


current_actual_validation_table[
    "Allowed Weight Error"
] = current_actual_weight_tolerance

current_actual_validation_table[
    "Validation Pass"
] = current_actual_weight_validation_pass


if not np.all(
    current_actual_weight_validation_pass
):
    failed_current_actual_validation = (
        current_actual_validation_table.loc[
            ~current_actual_weight_validation_pass
        ].copy()
    )

    display(
        failed_current_actual_validation
    )

    raise AssertionError(
        "Actual-product positions do not reconstruct "
        "current_state.csv within the Stage 7 "
        "implementation-margin rounding allowance."
    )


print(
    "\nALL CURRENT ACTUAL-PRODUCT POSITION TESTS PASSED"
)

# ------------------------------------------------------------
# BUILD EXPOSURE MAPS AND NEARBY CANDIDATES
# ------------------------------------------------------------

discrete_exposure_maps = {}

nearby_discrete_candidates = {}

candidate_summary_rows = []

for k, future_name in enumerate(
    future_names_product_test
):

    implementation_margin = float(
        current_margin_discrete[
            k
        ]
    )

    display_factor = float(
        standard_contract_display_factor[
            future_name
        ]
    )

    desired_weight = float(
        continuous_current_weights[
            k
        ]
    )

    desired_standard_exposure = (
        weight_to_standard_contract_target(
            weight=desired_weight,
            reference_balance=(
                discrete_reference_balance
            ),
            implementation_margin=(
                implementation_margin
            ),
            display_factor=(
                display_factor
            ),
        )
    )

    exposure_map = (
        build_underlying_discrete_exposure_map(
            future_name=(
                future_name
            ),
            implementation_margin=(
                implementation_margin
            ),
            product_specification=(
                tradable_contract_products
            ),
        )
    )

    discrete_exposure_maps[
        future_name
    ] = exposure_map

    candidates = (
        get_nearby_underlying_candidates(
            desired_standard_exposure=(
                desired_standard_exposure
            ),
            exposure_map=(
                exposure_map
            ),
            n_each_side=3,
        )
    )

    nearby_discrete_candidates[
        future_name
    ] = candidates

    for candidate_rank, candidate in enumerate(
        candidates
    ):

        product_position_text = ", ".join(
            f"{product_name}={position}"
            for product_name, position in zip(
                candidate[
                    "product_names"
                ],
                candidate[
                    "positions"
                ],
            )
            if position != 0
        )

        if product_position_text == "":

            product_position_text = "FLAT"

        candidate_weight = (
            candidate[
                "implementation_units"
            ]
            * implementation_margin
            / discrete_reference_balance
        )

        candidate_summary_rows.append(
            {
                "Future": future_name,
                "Underlying": (
                    underlying_symbols[
                        future_name
                    ]
                ),
                "Candidate Rank": (
                    candidate_rank
                ),
                "Continuous Weight": (
                    desired_weight
                ),
                "Desired Std Contracts": (
                    desired_standard_exposure
                ),
                "Candidate Std Contracts": (
                    candidate[
                        "standard_exposure"
                    ]
                ),
                "Candidate Weight": (
                    candidate_weight
                ),
                "Required Margin": (
                    candidate[
                        "required_margin"
                    ]
                ),
                "Actual Products": (
                    product_position_text
                ),
            }
        )


discrete_candidate_summary = pd.DataFrame(
    candidate_summary_rows
)

display(
    discrete_candidate_summary
)

# ------------------------------------------------------------
# SEARCH CONTROL
#
# Stage 1 normally produces about 5-7 candidates per underlying.
# The raw Cartesian product is still manageable here, but we
# immediately reject impossible portfolios and retain only the
# closest feasible candidates for Stage 3.
# ------------------------------------------------------------

max_joint_candidates_to_keep = max_joint_candidates_to_keep_toggle

execution_tol = 1e-10


# ------------------------------------------------------------
# CURRENT CARRIED POSITION
#
# The actual-product dictionary is now the authoritative
# discrete starting state.
#
# We retain the canonical implementation-contract vector
# reconstructed from it because the existing production
# execution engine operates in canonical implementation units.
#
# The previous validation cell already proved that this
# product-level state exactly reconstructs current_state.csv.
# ------------------------------------------------------------

current_carried_weights_discrete = np.asarray(
    current_position_weights,
    dtype=float,
)

if current_carried_weights_discrete.shape != (K,):
    raise ValueError(
        "current_position_weights must have shape (K,)."
    )

current_q_implementation = np.asarray(
    current_actual_q_implementation,
    dtype=float,
).copy()

if current_q_implementation.shape != (K,):
    raise ValueError(
        "current_actual_q_implementation must have shape (K,)."
    )

current_product_positions_discrete = {
    product_name: int(
        current_actual_product_positions_validated[
            product_name
        ]
    )
    for product_name in (
        expected_actual_products
    )
}

current_state_q_check = (
    current_carried_weights_discrete
    * discrete_reference_balance
    / current_margin_discrete
)

current_product_state_q_error = float(
    np.max(
        np.abs(
            current_q_implementation
            - current_state_q_check
        )
    )
)

# TEMPORARY live-run tolerance for the same small carried-state
# reconstruction discrepancy checked above.
current_product_state_q_tol = 0.01

if current_product_state_q_error > current_product_state_q_tol:
    raise ValueError(
        "Actual-product carried state does not reconstruct "
        "the current underlying optimizer state."
    )


# ------------------------------------------------------------
# BUILD THE PER-FUTURE CANDIDATE LISTS
# ------------------------------------------------------------

joint_candidate_lists = [
    nearby_discrete_candidates[
        future_name
    ]
    for future_name in (
        future_names_product_test
    )
]

raw_joint_count = int(
    np.prod(
        [
            len(candidate_list)
            for candidate_list in (
                joint_candidate_lists
            )
        ],
        dtype=np.int64,
    )
)

print(
    "Raw joint candidate count: "
    f"{raw_joint_count:,}"
)


# ------------------------------------------------------------
# ENUMERATE JOINT TARGETS
# ------------------------------------------------------------

stage2_start_time = time.perf_counter()

feasible_joint_candidates = []

rejection_counts = {
    "gross_weight": 0,
    "product_transition": 0,
    "execution": 0,
}

for candidate_tuple in product(
    *joint_candidate_lists
):

    # --------------------------------------------------------
    # Build the desired canonical implementation-contract
    # vector represented by this actual-product portfolio.
    # --------------------------------------------------------

    q_target = np.array(
        [
            discrete_candidate_to_implementation_contracts(
                candidate
            )
            for candidate in candidate_tuple
        ],
        dtype=float,
    )

    candidate_weights = (
        implementation_contracts_to_weights(
            q=q_target,
            margin_per_contract=(
                current_margin_discrete
            ),
            reference_balance=(
                discrete_reference_balance
            ),
        )
    )

    candidate_gross_weight = float(
        np.sum(
            np.abs(
                candidate_weights
            )
        )
    )

    # --------------------------------------------------------
    # Normal continuous gross-exposure rule.
    #
    # If the target itself exceeds 100% gross, it is only
    # potentially legal through grandfathering. The existing
    # execution engine below is the authoritative transition
    # check, so do not reject solely for gross > 1 here when
    # the carried portfolio itself is grandfathered.
    # --------------------------------------------------------

    current_gross_weight = float(
        np.sum(
            np.abs(
                current_carried_weights_discrete
            )
        )
    )

    if (
        candidate_gross_weight
        > 1.0 + execution_tol
        and current_gross_weight
        <= 1.0 + execution_tol
    ):
        rejection_counts[
            "gross_weight"
        ] += 1
        continue

    # --------------------------------------------------------
    # Build the exact actual-product target and validate the
    # transition from the actual products currently held.
    # --------------------------------------------------------

    actual_product_positions = {
        product_name: 0
        for product_name in (
            expected_actual_products
        )
    }

    for future_name, candidate in zip(
        future_names_product_test,
        candidate_tuple,
    ):

        for product_name, q_product in zip(
            candidate[
                "product_names"
            ],
            candidate[
                "positions"
            ],
        ):

            actual_product_positions[
                product_name
            ] = int(
                q_product
            )

    product_transition = (
        validate_actual_product_transition(
            current_positions=(
                current_product_positions_discrete
            ),
            target_positions=(
                actual_product_positions
            ),
            product_specification=(
                tradable_contract_products
            ),
            future_names=(
                future_names_product_test
            ),
            tol=execution_tol,
        )
    )

    if not product_transition[
        "feasible"
    ]:

        rejection_counts[
            "product_transition"
        ] += 1

        continue

    # --------------------------------------------------------
    # Apply the SAME grandfathered margin execution engine
    # already validated in the production optimizer.
    #
    # If the engine cannot reach q_target exactly, this target
    # is not an executable discrete portfolio at the current
    # timestamp.
    # --------------------------------------------------------

    q_executed = execute_target_contracts(
        balance=(
            discrete_reference_balance
        ),
        q_old=(
            current_q_implementation
        ),
        q_target=q_target,
        margin_per_contract=(
            current_margin_discrete
        ),
        tol=execution_tol,
    )

    execution_error = float(
        np.max(
            np.abs(
                q_executed
                - q_target
            )
        )
    )

    if execution_error > execution_tol:

        rejection_counts[
            "execution"
        ] += 1
        continue

    # --------------------------------------------------------
    # Exact target is executable.
    # --------------------------------------------------------

    required_margin = (
        required_margin_from_contracts(
            q_target,
            current_margin_discrete,
        )
    )

    weight_distance = (
        discrete_weight_distance(
            candidate_weights=(
                candidate_weights
            ),
            continuous_weights=(
                continuous_current_weights
            ),
        )
    )

    feasible_joint_candidates.append(
        {
            "q_target": (
                q_target.copy()
            ),
            "weights": (
                candidate_weights.copy()
            ),
            "gross_weight": (
                candidate_gross_weight
            ),
            "required_margin": float(
                required_margin
            ),
            "weight_distance": float(
                weight_distance
            ),
            "actual_product_positions": (
                actual_product_positions
            ),
            "actual_product_trades": (
                product_transition[
                    "trades"
                ].copy()
            ),
            "underlying_candidates": (
                candidate_tuple
            ),
        }
    )


# ------------------------------------------------------------
# SORT AND RETAIN THE CLOSEST FEASIBLE SET
#
# Again:
# this is NOT the final objective.
#
# Stage 3 will rank these by expected prize.
# ------------------------------------------------------------

stage2_enumeration_runtime = (
    time.perf_counter()
    - stage2_start_time
)

print()
print(
    "[STAGE 2 TIMING] "
    f"Enumerated {raw_joint_count:,} raw joint candidates "
    f"in {stage2_enumeration_runtime:.2f} sec"
)

print(
    "[STAGE 2 TIMING] "
    f"Feasible before retention: "
    f"{len(feasible_joint_candidates):,}"
)
        
feasible_joint_candidates = sorted(
    feasible_joint_candidates,
    key=lambda candidate: (
        candidate[
            "weight_distance"
        ],
        candidate[
            "required_margin"
        ],
    ),
)

total_feasible_joint_candidates = len(
    feasible_joint_candidates
)

stage3_joint_candidates = (
    feasible_joint_candidates[
        :max_joint_candidates_to_keep
    ]
)


# ------------------------------------------------------------
# DISPLAY THE CLOSEST FEASIBLE PORTFOLIOS
# ------------------------------------------------------------

joint_summary_rows = []

for rank, candidate in enumerate(
    stage3_joint_candidates[
        :25
    ]
):

    nonzero_products = [
        (
            f"{product_name}="
            f"{q_product:+d}"
        )
        for product_name, q_product in (
            candidate[
                "actual_product_positions"
            ].items()
        )
        if q_product != 0
    ]

    if len(nonzero_products) == 0:
        position_text = "FLAT"
    else:
        position_text = ", ".join(
            nonzero_products
        )

    joint_summary_rows.append(
        {
            "Search Rank": rank,
            "Weight Distance": (
                candidate[
                    "weight_distance"
                ]
            ),
            "Gross Weight": (
                candidate[
                    "gross_weight"
                ]
            ),
            "Required Margin": (
                candidate[
                    "required_margin"
                ]
            ),
            "Margin Remaining": (
                discrete_reference_balance
                - candidate[
                    "required_margin"
                ]
            ),
            "Actual Products": (
                position_text
            ),
        }
    )

joint_candidate_summary = pd.DataFrame(
    joint_summary_rows
)

print(
    "\nJOINT SEARCH SUMMARY"
)

print(
    "Current balance: "
    f"${discrete_reference_balance:,.2f}"
)

print(
    "Current carried gross weight: "
    f"{current_gross_weight:.6f}"
)

print(
    "Continuous target gross weight: "
    f"{np.sum(np.abs(continuous_current_weights)):.6f}"
)

print(
    "Candidates rejected by gross rule: "
    f"{rejection_counts['gross_weight']:,}"
)

print(
    "Candidates rejected by actual-product transition rule: "
    f"{rejection_counts['product_transition']:,}"
)

print(
    "Candidates rejected by execution/margin rule: "
    f"{rejection_counts['execution']:,}"
)

print(
    "Total exactly executable candidates: "
    f"{total_feasible_joint_candidates:,}"
)

print(
    "Candidates retained for Stage 3: "
    f"{len(stage3_joint_candidates):,}"
)

display(
    joint_candidate_summary
)

# ============================================================
# DISCRETE EXECUTION - STAGE 3D
# FULL SAME-WORLD EXPECTED-PRIZE SEARCH
# ============================================================
#
# Evaluate all Stage-2 retained executable candidates using:
#
#   - exact same Monte Carlo worlds
#   - exact same dollar-account mechanics
#   - exact same prize objective
#   - no gradients
#   - candidate batches for speed / memory control
#
# The winning portfolio is selected by EXPECTED PRIZE.
# Weight distance is used only as a tie-breaker.
# ============================================================


# ------------------------------------------------------------
# SEARCH CONTROL
# ------------------------------------------------------------

stage3d_batch_size = stage3d_batch_size_toggle

stage3d_n_candidates = len(
    stage3_joint_candidates
)

if stage3d_n_candidates == 0:
    raise RuntimeError(
        "No Stage-2 candidates are available."
    )


# ------------------------------------------------------------
# STORAGE
# ------------------------------------------------------------

stage3d_total_ev = np.full(
    stage3d_n_candidates,
    np.nan,
    dtype=float,
)

stage3d_final_ev = np.full(
    stage3d_n_candidates,
    np.nan,
    dtype=float,
)

stage3d_daily_ev = np.full(
    stage3d_n_candidates,
    np.nan,
    dtype=float,
)


# ------------------------------------------------------------
# FULL BATCHED SEARCH
# ------------------------------------------------------------

stage3d_start_time = time.perf_counter()

stage3d_batch_count = int(
    np.ceil(
        stage3d_n_candidates
        / stage3d_batch_size
    )
)

for batch_number, batch_start in enumerate(
    range(
        0,
        stage3d_n_candidates,
        stage3d_batch_size,
    ),
    start=1,
):

    batch_start_time = time.perf_counter()

    batch_end = min(
        batch_start
        + stage3d_batch_size,
        stage3d_n_candidates,
    )

    batch_candidates = (
        stage3_joint_candidates[
            batch_start:batch_end
        ]
    )

    batch_weights = np.stack(
        [
            candidate[
                "weights"
            ]
            for candidate in (
                batch_candidates
            )
        ],
        axis=0,
    )

    batch_result = (
        evaluate_discrete_candidate_batch_value_only(
            batch_weights
        )
    )

    batch_eval_runtime = (
        time.perf_counter()
        - batch_start_time
    )

    print(
        "[STAGE 3D TIMING] "
        f"batch {batch_number}/{stage3d_batch_count} | "
        f"candidates={batch_end - batch_start} | "
        f"eval_time={batch_eval_runtime:.3f}s | "
        f"sec/candidate="
        f"{batch_eval_runtime / (batch_end - batch_start):.4f}"
    )

    stage3d_total_ev[
        batch_start:batch_end
    ] = batch_result[
        "total_ev"
    ]

    stage3d_final_ev[
        batch_start:batch_end
    ] = batch_result[
        "final_ev"
    ]

    stage3d_daily_ev[
        batch_start:batch_end
    ] = batch_result[
        "daily_ev"
    ]

    if (
        batch_number == 1
        or batch_number % 10 == 0
        or batch_number == stage3d_batch_count
    ):

        elapsed = (
            time.perf_counter()
            - stage3d_start_time
        )

        print(
            "Completed batch "
            f"{batch_number:,}/"
            f"{stage3d_batch_count:,} "
            f"({batch_end:,}/"
            f"{stage3d_n_candidates:,} candidates), "
            f"elapsed {elapsed:.1f} sec"
        )


stage3d_runtime = (
    time.perf_counter()
    - stage3d_start_time
)


# ------------------------------------------------------------
# VALIDATE ALL RESULTS
# ------------------------------------------------------------

assert np.all(
    np.isfinite(
        stage3d_total_ev
    )
)

assert np.all(
    np.isfinite(
        stage3d_final_ev
    )
)

assert np.all(
    np.isfinite(
        stage3d_daily_ev
    )
)

stage3d_component_error = float(
    np.max(
        np.abs(
            stage3d_total_ev
            - (
                stage3d_final_ev
                + stage3d_daily_ev
            )
        )
    )
)

assert (
    stage3d_component_error
    <= 1e-10
)


# ------------------------------------------------------------
# BUILD COMPLETE RESULT TABLE
# ------------------------------------------------------------

stage3d_rows = []

for candidate_index, candidate in enumerate(
    stage3_joint_candidates
):

    nonzero_products = [
        (
            f"{product_name}="
            f"{q_product:+d}"
        )
        for product_name, q_product in (
            candidate[
                "actual_product_positions"
            ].items()
        )
        if q_product != 0
    ]

    if len(
        nonzero_products
    ) == 0:

        position_text = "FLAT"

    else:

        position_text = ", ".join(
            nonzero_products
        )

    stage3d_rows.append(
        {
            "Candidate": (
                candidate_index
            ),
            "Expected Prize": (
                stage3d_total_ev[
                    candidate_index
                ]
            ),
            "Final Prize EV": (
                stage3d_final_ev[
                    candidate_index
                ]
            ),
            "Daily Prize EV": (
                stage3d_daily_ev[
                    candidate_index
                ]
            ),
            "Weight Distance": (
                candidate[
                    "weight_distance"
                ]
            ),
            "Gross Weight": (
                candidate[
                    "gross_weight"
                ]
            ),
            "Required Margin": (
                candidate[
                    "required_margin"
                ]
            ),
            "Margin Remaining": (
                evaluation_reference_balance
                - candidate[
                    "required_margin"
                ]
            ),
            "Actual Products": (
                position_text
            ),
        }
    )


stage3d_results = (
    pd.DataFrame(
        stage3d_rows
    )
    .sort_values(
        by=[
            "Expected Prize",
            "Weight Distance",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# IDENTIFY WINNER
# ------------------------------------------------------------

stage3d_best_candidate_index = int(
    stage3d_results.iloc[
        0
    ][
        "Candidate"
    ]
)

stage3d_best_candidate = (
    stage3_joint_candidates[
        stage3d_best_candidate_index
    ]
)

stage3d_best_total_ev = float(
    stage3d_total_ev[
        stage3d_best_candidate_index
    ]
)

stage3d_best_final_ev = float(
    stage3d_final_ev[
        stage3d_best_candidate_index
    ]
)

stage3d_best_daily_ev = float(
    stage3d_daily_ev[
        stage3d_best_candidate_index
    ]
)


# ------------------------------------------------------------
# EXACT EXECUTION CHECK FOR WINNER
# ------------------------------------------------------------

stage3d_best_q_check = (
    execute_target_contracts(
        balance=(
            discrete_reference_balance
        ),
        q_old=(
            current_q_implementation
        ),
        q_target=(
            stage3d_best_candidate[
                "q_target"
            ]
        ),
        margin_per_contract=(
            current_margin_discrete
        ),
        tol=execution_tol,
    )
)

stage3d_best_execution_error = float(
    np.max(
        np.abs(
            stage3d_best_q_check
            - stage3d_best_candidate[
                "q_target"
            ]
        )
    )
)

assert (
    stage3d_best_execution_error
    <= execution_tol
)


# ------------------------------------------------------------
# AUTHORITATIVE PRODUCTION CHECK FOR THE WINNER
#
# One full production evaluation is cheap enough and gives us
# a final independent confirmation that the selected winner's
# EV is exactly reproduced by the original objective.
# ------------------------------------------------------------

stage3d_best_W_reference = (
    weights_remaining.copy()
)

stage3d_best_W_reference[
    0,
    :
] = stage3d_best_candidate[
    "weights"
]

(
    stage3d_best_reference_ev,
    _,
    stage3d_best_reference_diagnostics,
) = (
    total_expected_prize_and_batched_reverse_gradient_from_dollar_account(
        W=(
            stage3d_best_W_reference
        ),
        raw_return_sims=(
            evaluation_R_sims
        ),
        pathwise_notional=(
            evaluation_pathwise_notional
        ),
        margin=(
            evaluation_margin
        ),
        initial_contracts=(
            evaluation_initial_contracts
        ),
        B_sims=(
            evaluation_B_sims
        ),
        final_place_balance_sims=(
            evaluation_final_place_balance_sims
        ),
        day_labels=(
            evaluation_day_labels
        ),
        prizes=prizes,
        starting_balance=(
            stage3_starting_balance
        ),
        current_final_place_balances=(
            stage3_current_final_place_balances
        ),
        current_rank=(
            stage3_current_rank
        ),
        daily_prize=500.0,
        final_win_temperature=0.02,
        daily_competitor_temperature=(
            evaluation_final_daily_temperature
        ),
        daily_win_temperature=(
            evaluation_final_daily_temperature
        ),
        eligible_days=(
            stage3_eligible_days
        ),
        daily_threshold_days=(
            stage3_daily_threshold_days
        ),
        current_day=(
            stage3_current_day
        ),
        our_realized_current_day_pnl=(
            stage3_our_realized_current_day_pnl
        ),
        current_daily_rank=(
            stage3_current_daily_rank
        ),
        current_daily_place_pnl=(
            stage3_current_daily_place_pnl
        ),
        daily_prize_already_won=(
            stage3_daily_prize_already_won
        ),
        reference_balance=(
            evaluation_reference_balance
        ),
    )
)

stage3d_best_reference_ev = float(
    stage3d_best_reference_ev
)

stage3d_best_reference_error = abs(
    stage3d_best_total_ev
    - stage3d_best_reference_ev
)

assert (
    stage3d_best_reference_error
    <= 1e-10
)


# ------------------------------------------------------------
# CONTINUOUS BENCHMARK
# ------------------------------------------------------------

stage3d_continuous_ev = float(
    best_result[
        "expected_prize"
    ]
)

stage3d_continuous_final_ev = float(
    best_result[
        "final_expected_prize"
    ]
)

stage3d_continuous_daily_ev = float(
    best_result[
        "daily_expected_prize"
    ]
)

stage3d_discretization_cost = (
    stage3d_continuous_ev
    - stage3d_best_total_ev
)


# ------------------------------------------------------------
# WINNER POSITION TEXT
# ------------------------------------------------------------

stage3d_best_nonzero_products = [
    (
        f"{product_name}="
        f"{q_product:+d}"
    )
    for product_name, q_product in (
        stage3d_best_candidate[
            "actual_product_positions"
        ].items()
    )
    if q_product != 0
]

if len(
    stage3d_best_nonzero_products
) == 0:

    stage3d_best_position_text = "FLAT"

else:

    stage3d_best_position_text = ", ".join(
        stage3d_best_nonzero_products
    )


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 72
)

print(
    "STAGE 3D - FULL EXECUTABLE EXPECTED-PRIZE SEARCH"
)

print(
    "=" * 72
)

print(
    "Candidates evaluated: "
    f"{stage3d_n_candidates:,}"
)

print(
    "Batch size: "
    f"{stage3d_batch_size:,}"
)

print(
    "Runtime: "
    f"{stage3d_runtime:.2f} sec"
)

print(
    "\nTOP 25 EXECUTABLE PORTFOLIOS BY EXPECTED PRIZE"
)

display(
    stage3d_results.head(
        25
    )
)


print(
    "\nCONTINUOUS OPTIMUM"
)

print(
    "Total expected prize: "
    f"${stage3d_continuous_ev:.6f}"
)

print(
    "Final-leaderboard EV: "
    f"${stage3d_continuous_final_ev:.6f}"
)

print(
    "Daily-prize EV: "
    f"${stage3d_continuous_daily_ev:.6f}"
)


print(
    "\nBEST EXECUTABLE CURRENT-PERIOD PORTFOLIO"
)

print(
    "Actual products: "
    f"{stage3d_best_position_text}"
)

print(
    "Total expected prize: "
    f"${stage3d_best_total_ev:.6f}"
)

print(
    "Final-leaderboard EV: "
    f"${stage3d_best_final_ev:.6f}"
)

print(
    "Daily-prize EV: "
    f"${stage3d_best_daily_ev:.6f}"
)

print(
    "Gross weight: "
    f"{stage3d_best_candidate['gross_weight']:.6f}"
)

print(
    "Required margin: "
    f"${stage3d_best_candidate['required_margin']:,.2f}"
)

print(
    "Margin remaining: "
    f"${evaluation_reference_balance - stage3d_best_candidate['required_margin']:,.2f}"
)


print(
    "\nDISCRETIZATION COST"
)

print(
    "Continuous EV minus executable EV: "
    f"${stage3d_discretization_cost:.6f}"
)


print(
    "\nVALIDATION"
)

print(
    "Maximum total/component error: "
    f"{stage3d_component_error:.16e}"
)

print(
    "Winner exact-execution error: "
    f"{stage3d_best_execution_error:.16e}"
)

print(
    "Winner fast/reference EV error: "
    f"{stage3d_best_reference_error:.16e}"
)

print(
    "\nALL STAGE 3D FULL-SEARCH TESTS PASSED"
)

# ============================================================
# LIVE DISCRETE EXECUTION RECOMMENDATION
# ============================================================
#
# Production-facing output for the current live optimizer run.
#
# This cell does NOT re-optimize and does NOT resimulate.
#
# It consumes:
#
#   - the continuous optimum already produced by the optimizer
#   - the validated Stage 3D executable winner
#   - the authoritative current actual-product positions
#
# and reports the actual holdings and trades to place now.
# ============================================================


# ------------------------------------------------------------
# VALIDATE REQUIRED LIVE OBJECTS
# ------------------------------------------------------------

required_live_objects = [
    "stage3d_best_candidate",
    "stage3d_best_total_ev",
    "stage3d_best_final_ev",
    "stage3d_best_daily_ev",
    "stage3d_continuous_ev",
    "stage3d_continuous_final_ev",
    "stage3d_continuous_daily_ev",
    "stage3d_discretization_cost",
    "current_actual_product_positions_validated",
    "tradable_contract_products",
    "future_names_product_test",
    "underlying_symbols",
    "standard_contract_display_factor",
    "evaluation_reference_balance",
]

missing_live_objects = [
    name
    for name in required_live_objects
    if name not in globals()
]

if missing_live_objects:
    raise RuntimeError(
        "Live discrete recommendation cannot run because "
        "required objects are missing: "
        f"{missing_live_objects}"
    )


# ------------------------------------------------------------
# AUTHORITATIVE TARGET HOLDINGS AND TRADES
# ------------------------------------------------------------

live_target_actual_product_positions = {
    product_name: int(q_product)
    for product_name, q_product in (
        stage3d_best_candidate[
            "actual_product_positions"
        ].items()
    )
}

live_actual_product_trades = {
    product_name: int(q_product)
    for product_name, q_product in (
        stage3d_best_candidate[
            "actual_product_trades"
        ].items()
    )
}


# ------------------------------------------------------------
# REVALIDATE THE WINNING PRODUCT TRANSITION
#
# This is intentionally cheap. It does not rerun Stage 3D.
# ------------------------------------------------------------

live_transition_check = (
    validate_actual_product_transition(
        current_positions=(
            current_actual_product_positions_validated
        ),
        target_positions=(
            live_target_actual_product_positions
        ),
        product_specification=(
            tradable_contract_products
        ),
        future_names=(
            future_names_product_test
        ),
        tol=1e-12,
    )
)

if not live_transition_check["feasible"]:
    raise RuntimeError(
        "Selected Stage 3D winner failed final "
        "actual-product transition validation. "
        f"Reason={live_transition_check['reason']}"
    )

for product_name in (
    live_actual_product_trades
):

    expected_trade = int(
        live_transition_check[
            "trades"
        ][
            product_name
        ]
    )

    stored_trade = int(
        live_actual_product_trades[
            product_name
        ]
    )

    if stored_trade != expected_trade:
        raise RuntimeError(
            f"{product_name}: stored Stage 3D trade "
            f"{stored_trade:+d} does not match "
            f"revalidated trade {expected_trade:+d}."
        )


# ------------------------------------------------------------
# BUILD PRODUCT-LEVEL LIVE ORDER TABLE
# ------------------------------------------------------------

live_product_rows = []

for future_name in (
    future_names_product_test
):

    products = tradable_contract_products[
        future_name
    ]

    for product_name, product_spec in (
        products.items()
    ):

        current_position = int(
            current_actual_product_positions_validated[
                product_name
            ]
        )

        target_position = int(
            live_target_actual_product_positions[
                product_name
            ]
        )

        trade = int(
            live_actual_product_trades[
                product_name
            ]
        )

        if trade > 0:
            action = "BUY"
        elif trade < 0:
            action = "SELL"
        else:
            action = "HOLD"

        live_product_rows.append(
            {
                "Future": future_name,
                "Underlying": (
                    underlying_symbols[
                        future_name
                    ]
                ),
                "Product": product_name,
                "Current Position": (
                    current_position
                ),
                "Target Position": (
                    target_position
                ),
                "Trade": trade,
                "Action": action,
                "Order Quantity": abs(
                    trade
                ),
                "Standard Equivalent": (
                    target_position
                    * float(
                        product_spec[
                            "standard_equivalent"
                        ]
                    )
                ),
            }
        )


live_product_table = pd.DataFrame(
    live_product_rows
)


# ------------------------------------------------------------
# BUILD UNDERLYING-LEVEL TARGET SUMMARY
# ------------------------------------------------------------

live_underlying_rows = []

for k, future_name in enumerate(
    future_names_product_test
):

    products = tradable_contract_products[
        future_name
    ]

    target_standard_equivalent = 0.0
    current_standard_equivalent = 0.0

    for product_name, product_spec in (
        products.items()
    ):

        standard_equivalent = float(
            product_spec[
                "standard_equivalent"
            ]
        )

        target_standard_equivalent += (
            live_target_actual_product_positions[
                product_name
            ]
            * standard_equivalent
        )

        current_standard_equivalent += (
            current_actual_product_positions_validated[
                product_name
            ]
            * standard_equivalent
        )

    target_implementation_contracts = float(
        stage3d_best_candidate[
            "q_target"
        ][
            k
        ]
    )

    target_weight = float(
        stage3d_best_candidate[
            "weights"
        ][
            k
        ]
    )

    live_underlying_rows.append(
        {
            "Future": future_name,
            "Underlying": (
                underlying_symbols[
                    future_name
                ]
            ),
            "Current Standard Equivalent": (
                current_standard_equivalent
            ),
            "Target Standard Equivalent": (
                target_standard_equivalent
            ),
            "Target Implementation Contracts": (
                target_implementation_contracts
            ),
            "Target Weight": (
                target_weight
            ),
        }
    )


live_underlying_table = pd.DataFrame(
    live_underlying_rows
)

live_underlying_table[
    "Target Weight"
] = (
    live_underlying_table[
        "Target Weight"
    ]
    .map(
        lambda x: f"{100.0 * x:.4f}%"
    )
)


# ------------------------------------------------------------
# BUILD NONZERO ORDER TABLE
# ------------------------------------------------------------

live_order_table = (
    live_product_table.loc[
        live_product_table[
            "Trade"
        ] != 0,
        [
            "Underlying",
            "Product",
            "Current Position",
            "Target Position",
            "Action",
            "Order Quantity",
            "Trade",
        ],
    ]
    .copy()
)

# Execute exposure-reducing trades first so that margin is
# released before trades that add exposure.
live_order_table[
    "_reduces_exposure"
] = (
    live_order_table[
        "Current Position"
    ].abs()
    >
    live_order_table[
        "Target Position"
    ].abs()
)

live_order_table = (
    live_order_table
    .sort_values(
        by="_reduces_exposure",
        ascending=False,
        kind="stable",
    )
    .drop(
        columns="_reduces_exposure"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# MARGIN SUMMARY
# ------------------------------------------------------------

live_required_margin = float(
    stage3d_best_candidate[
        "required_margin"
    ]
)

live_margin_remaining = float(
    evaluation_reference_balance
    - live_required_margin
)

live_gross_weight = float(
    stage3d_best_candidate[
        "gross_weight"
    ]
)


# ------------------------------------------------------------
# FINAL PRODUCTION-FACING OUTPUT
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 72
)

print(
    "LIVE EXECUTABLE FUTURES RECOMMENDATION"
)

print(
    "=" * 72
)

print(
    f"\nCurrent t: {current_t}"
)

print(
    "Reference balance: "
    f"${evaluation_reference_balance:,.2f}"
)


print(
    "\nCONTINUOUS OPTIMUM"
)

print(
    "Total expected prize: "
    f"${stage3d_continuous_ev:.6f}"
)

print(
    "Final-leaderboard EV: "
    f"${stage3d_continuous_final_ev:.6f}"
)

print(
    "Daily-prize EV: "
    f"${stage3d_continuous_daily_ev:.6f}"
)


print(
    "\nSELECTED EXECUTABLE PORTFOLIO"
)

print(
    "Total expected prize: "
    f"${stage3d_best_total_ev:.6f}"
)

print(
    "Final-leaderboard EV: "
    f"${stage3d_best_final_ev:.6f}"
)

print(
    "Daily-prize EV: "
    f"${stage3d_best_daily_ev:.6f}"
)

print(
    "EV loss vs continuous: "
    f"${stage3d_discretization_cost:.6f}"
)

print(
    "Gross weight: "
    f"{live_gross_weight:.6f}"
)

print(
    "Required margin: "
    f"${live_required_margin:,.2f}"
)

print(
    "Margin remaining: "
    f"${live_margin_remaining:,.2f}"
)


print(
    "\nACTUAL TRADES TO PLACE NOW"
)

if len(
    live_order_table
) == 0:

    print(
        "NO TRADES - HOLD CURRENT POSITIONS"
    )

else:

    display(
        live_order_table
    )


print(
    "\nTARGET ACTUAL-PRODUCT HOLDINGS"
)

display(
    live_product_table[
        [
            "Underlying",
            "Product",
            "Current Position",
            "Target Position",
            "Trade",
            "Standard Equivalent",
        ]
    ]
)


print(
    "\nTARGET UNDERLYING EXPOSURES"
)

display(
    live_underlying_table
)


print(
    "\nLIVE DISCRETE RECOMMENDATION READY"
)